# 06b — Statistical Validation of PIE-SAE Features (Exploratory)

**Status:** Exploratory analysis sections A–E completed; 19/19 internal computational consistency checks passed. Independent methodological replication and full-dictionary selection-aware validation remain outstanding.

**Scientific role:** Extend the exploratory `06a_statistical_significance.ipynb` without treating its ten selected features or the 98-feature follow-up panel as independent confirmatory samples.

## Research objective

Evaluate the reproducibility and robustness of selected SAE feature–gene associations, functional enrichment, and cross-layer correspondence using explicitly documented retrospective statistical analyses. Quantify conditional uncertainty, define hypothesis families, and evaluate multiple null models to prioritize computational intervention targets.

**Boundaries:** Statistical associations and geometric correspondence are not evidence of computational or biological causality. Computational interventions are deferred to `07_causal_validation.ipynb`; independent biological validation requires additional evidence.

**Execution and data safety:** The notebook contains no explicit commands to overwrite original research data or checkpoints. It reads selected HDF5 activation arrays, loads a PIE checkpoint to recover condition metadata, and loads pretrained SAE checkpoints for inference. It does not retrain models or load the full 14.5 GB activation archive into RAM. External project helpers may manage their own caches; this was not independently audited here.

## Methodological provenance and PIE-SAE extensions

| Analysis component | Related upstream implementations | Scope of the PIE-SAE analysis |
|---|---|---|
| **A. Statistical protocol** | Feature catalogs and ontology enrichment in `bio-sae/src/03_analyze_features.py` and `04_annotate_features.py` | Reference-term reconstruction, saved-count verification, and explicit testing-family documentation |
| **B. Activation robustness and bootstrap** | SAE feature activation analysis and related bootstrap methods | Condition-level, cell-context-stratified sensitivity analysis on observed PIE conditions and selected features |
| **C. Functional enrichment** | Top-gene feature catalogs, GO/KEGG/Reactome/TRRUST references, and FDR analyses in `bio-sae` | Restricted 4,420-term family, global BH adjustment, and exploratory degree-aware permutations; full selection-aware correction remains unresolved |
| **D. Cross-layer validation** | Cross-layer feature matching and co-activation methods in `bio-sae` | Replication of saved matches; signed-coordinate geometric controls rerun matching, while archived co-activation and gene-overlap controls permute partners **after** matching |
| **E. Evidence synthesis** | Feature atlases and analysis summaries | Integration of exploratory findings and limitations to prioritize candidates for notebook 07 |

The upstream projects provide methodological context, but the statistical procedures in this notebook are not claimed to be exact reproductions of all upstream implementations. In particular, the bootstrap, fixed-panel maxT, and coarse degree-stratified permutation analyses are PIE-SAE extensions.

**Upstream repositories (attribution):**
- https://github.com/Biodyn-AI/bio-sae
- https://github.com/Biodyn-AI/bio-sae-circuits
- https://github.com/Biodyn-AI/neuro-sae

The SAE architectures and feature catalogs were established in earlier notebooks. The present analysis reuses saved artifacts rather than retraining the dictionaries.

## A. Statistical protocol and reproducibility audit

### A.1. Units, selection, and inferential scope

1. **Functional annotation:** a tested feature is a *fixed Top-20 gene list*, contrasted with ontology gene sets over the PIE output-gene universe. Tests depend on the explicit gene-label null and how the feature set was selected.
2. **Perturbation-dependent activation:** the unit for estimating variation over conditions is the **experimental condition**, keeping the gene vector together when resampling. Gene positions inside a condition are not independent replicates.
3. **Cell contexts:** HepG2, Jurkat, and RPE1 are three fixed observed contexts, not a random sample of all cell types; within-context resampling cannot establish population-wide generalization.
4. **Exploration vs confirmation:** the 32 validation conditions and the pilot ten features were already inspected in 06a. New analyses on them remain **exploratory/sensitivity analyses** unless an untouched evaluation set is established.
5. **Multiple testing:** define the eligible feature set, term family, filter thresholds, tested layers and adjusted-p-value family **before** evaluating outcomes.

### Initial checkpoint

Inventory source artifacts and reconstruct the ontology family. In 06a, 4,420 filtered GO BP/KEGG/Reactome sets were reconstructed, while an original annotation summary reported 4,422 terms. The difference is *unresolved*, and must not be silently relabeled an exact reproduction.


In [1]:
from pathlib import Path
import sys
import json
import platform

# Find the project root whether VS Code starts Jupyter in the
# repository root or in its notebooks/ directory.
working_dir = Path.cwd().resolve()
project_candidates = (working_dir, *working_dir.parents)

ROOT = next(
    (folder for folder in project_candidates
     if (folder / "notebooks" / "05_cross_layer_analysis.ipynb").is_file()),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "PIE-SAE project root not found. Open the notebook from "
        "the existing pie-sae repository in VS Code."
    )

print("=== ENVIRONMENT ===")
print("Project root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

ARTIFACTS = {
    "Notebook 06a": ROOT / "notebooks/06a_statistical_significance.ipynb",
    "SAE manifest": ROOT / "results/sae_training/independent_sae_manifest.json",
    "Extraction manifest": ROOT / "results/extraction_validation/extraction_manifest.json",
    "PIE gene reference": ROOT / "results/gene_reference/gene_index_to_symbol.json",
    "Training HDF5": ROOT / "data/activations/expanded_128_32/train_128.h5",
    "Validation HDF5": ROOT / "data/activations/expanded_128_32/val_32.h5",
    "Original annotation source": ROOT / "src/sae_biological_annotation.py",
    "Original cross-layer outputs": ROOT / "results/cross_layer_independent",
}

print("\n=== EXISTING ARTIFACTS ===")
for label, path in ARTIFACTS.items():
    print(f"{label:29s}: {'FOUND' if path.exists() else 'MISSING'}")

assert ARTIFACTS["PIE gene reference"].is_file(), "PIE gene vocabulary is required."
assert ARTIFACTS["Notebook 06a"].is_file(), "Completed 06a notebook is required."
print("\nRead-only environment audit complete.")

=== ENVIRONMENT ===
Project root: D:\bio_paper\pie-sae
Python: 3.12.13
Platform: Windows-11-10.0.26300-SP0

=== EXISTING ARTIFACTS ===
Notebook 06a                 : FOUND
SAE manifest                 : FOUND
Extraction manifest          : FOUND
PIE gene reference           : FOUND
Training HDF5                : FOUND
Validation HDF5              : FOUND
Original annotation source   : FOUND
Original cross-layer outputs : FOUND

Read-only environment audit complete.


### A.2. Inspect all decoder SAE dictionaries and annotation summaries

The intended first *full-dictionary* analysis concerns **four gene-aligned decoder SAE dictionaries**, rather than all 14 encoder/processor/decoder dictionaries. Other positions have different semantics and require different association definitions.

This audit only counts saved entries. **An annotated feature is not a causally validated feature**, and the number of original BH discoveries is not the number of new 06b findings.


In [2]:
from collections import Counter

FEATURE_ROOT = ROOT / "results/sae_feature_analysis_independent"
CATALOG_DIR = FEATURE_ROOT / "feature_catalogs"
ANNOT_DIR = FEATURE_ROOT / "biological_annotations"

print("=== SAVED DECODER FEATURE AND ANNOTATION INVENTORY ===")

for layer in range(4):
    print(f"\nDecoder block {layer}")

    for split in ("train", "validation"):
        path = CATALOG_DIR / f"decoder_{layer}_{split}_catalog.json"
        if not path.is_file():
            print(f"  {split:10s}: MISSING")
            continue

        catalog = json.loads(path.read_text(encoding="utf-8"))
        features = catalog.get("features", [])
        eligible = sum(
            len(f.get("top_genes", [])) >= 20
            and len({g.get("gene_name") for g in f.get("top_genes", [])[:20]}) == 20
            for f in features
        )
        print(f"  {split:10s}: {len(features)} features; "
              f"{eligible} have 20 distinct ranked genes")

    path = ANNOT_DIR / f"decoder_{layer}_annotations.json"
    if not path.is_file():
        print("  annotations : MISSING")
        continue

    annotation = json.loads(path.read_text(encoding="utf-8"))
    summary = annotation.get("summary", {})
    enrichments = annotation.get("significant_enrichments", [])
    counts = Counter(str(rec.get("source", "UNKNOWN")) for rec in enrichments)
    print("  annotations :", len(enrichments), "saved significant associations")
    print("  reported ontology terms:", summary.get("n_ontology_terms", "NOT REPORTED"))
    print("  source counts:", dict(sorted(counts.items())))

print("\nRead-only catalog inventory complete.")

=== SAVED DECODER FEATURE AND ANNOTATION INVENTORY ===

Decoder block 0
  train     : 3072 features; 3072 have 20 distinct ranked genes
  validation: 3072 features; 3072 have 20 distinct ranked genes
  annotations : 18976 saved significant associations
  reported ontology terms: 4422
  source counts: {'GO_BP': 6744, 'KEGG': 2938, 'Reactome': 9261, 'TRRUST_TF': 33}

Decoder block 1
  train     : 3072 features; 3072 have 20 distinct ranked genes
  validation: 3072 features; 3072 have 20 distinct ranked genes
  annotations : 18144 saved significant associations
  reported ontology terms: 4422
  source counts: {'GO_BP': 6287, 'KEGG': 2717, 'Reactome': 9106, 'TRRUST_TF': 34}

Decoder block 2
  train     : 3072 features; 3072 have 20 distinct ranked genes
  validation: 3072 features; 3070 have 20 distinct ranked genes
  annotations : 18321 saved significant associations
  reported ontology terms: 4422
  source counts: {'GO_BP': 6397, 'KEGG': 2768, 'Reactome': 9127, 'TRRUST_TF': 29}

Decoder 

### A.3. Reconstruct the restricted ontology family

Use the exact PIE output-gene vocabulary and the term-size rule **5–500 genes** used in the earlier pilot. Report each source separately; do not imply that the 4,420-term family includes TRRUST.

**Diagnostic to resolve:** the 4,422 terms reported by the original annotation JSON are not automatically explained by the 4,420 GO BP/KEGG/Reactome sets. The original code's term-count construction must be audited before claiming exact equivalence.


In [3]:
VOCAB_PATH = ARTIFACTS["PIE gene reference"]
vocab = json.loads(VOCAB_PATH.read_text(encoding="utf-8"))

if not isinstance(vocab, dict) or "genes" not in vocab:
    raise ValueError("Unexpected PIE gene vocabulary structure; inspect the source JSON.")

output_genes = vocab["genes"]
if len(output_genes) != len(set(output_genes)):
    raise ValueError("Duplicate symbols in the PIE output-gene vocabulary.")

universe = set(output_genes)
print("=== PIE OUTPUT-GENE UNIVERSE ===")
print("Number of unique genes:", len(universe))

REFERENCE_DIR = ROOT / "data/biological_references"
resources = {
    "GO_BP": "go_bp_gene_sets.json",
    "KEGG": "kegg_gene_sets.json",
    "Reactome": "reactome_gene_sets.json",
}

restricted_gene_sets = {}
print("\n=== TERM FILTER: 5 TO 500 PIE GENES ===")
for source, filename in resources.items():
    path = REFERENCE_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing source: {path}")

    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise TypeError(f"Unexpected ontology format for {filename}")

    accepted = 0
    for term, members in raw.items():
        if not isinstance(members, list):
            raise TypeError(f"Term has unexpected member type: {source} / {term}")
        selected = universe.intersection(members)
        if 5 <= len(selected) <= 500:
            restricted_gene_sets[(source, str(term))] = selected
            accepted += 1

    print(f"{source:10s}: {len(raw):5d} raw -> {accepted:5d} retained")

reconstructed_total = len(restricted_gene_sets)
print("\nReconstructed restricted family:", reconstructed_total)

original_summary_path = ANNOT_DIR / "decoder_0_annotations.json"
original_annotation = json.loads(
    original_summary_path.read_text(encoding="utf-8")
)
reported_total = original_annotation.get("summary", {}).get("n_ontology_terms")

print("Original decoder.0 reported terms:", reported_total)
if isinstance(reported_total, int) and reported_total != reconstructed_total:
    print("UNRESOLVED DISCREPANCY:", reported_total - reconstructed_total)
    print("Do not claim an exact reconstruction of the original test family.")

original_sig_terms = {
    (str(r.get("source")), str(r.get("term")))
    for r in original_annotation.get("significant_enrichments", [])
}
not_reconstructed = original_sig_terms - set(restricted_gene_sets)
print("\nSignificant original terms outside this restricted family:",
      len(not_reconstructed))
print("Missing significant terms by source:",
      dict(Counter(source for source, _ in not_reconstructed)))
for source, term in sorted(not_reconstructed)[:5]:
    print("  ", source, "/", term[:100])

print("\nRead-only ontology audit complete.")

=== PIE OUTPUT-GENE UNIVERSE ===
Number of unique genes: 6642

=== TERM FILTER: 5 TO 500 PIE GENES ===
GO_BP     :  5343 raw ->  2838 retained
KEGG      :   320 raw ->   286 retained
Reactome  :  1818 raw ->  1296 retained

Reconstructed restricted family: 4420
Original decoder.0 reported terms: 4422
UNRESOLVED DISCREPANCY: 2
Do not claim an exact reconstruction of the original test family.

Significant original terms outside this restricted family: 1
Missing significant terms by source: {'TRRUST_TF': 1}
   TRRUST_TF / curated_transcription_factors

Read-only ontology audit complete.


### A.4. Reproducibility scope and data safeguards

The initial inventory verifies the presence of the saved train/validation catalogs, decoder annotation files, PIE output-gene vocabulary, and local model and activation artifacts. The annotation term-count discrepancy is recorded rather than resolved by changing the denominator.

The subsequent analyses use explicitly named reference sets and distinguish training-derived gene lists from validation-condition resampling. Gene positions within a perturbation condition are treated as correlated observations; selected feature panels are not independent confirmatory samples.

**Data handling:** Original HDF5 files, PIE and SAE checkpoints, saved annotation files, and notebook 05 artifacts are not overwritten by any explicit operation in this notebook. Computation occurs in memory; existing scientific outputs remain reference material.

**Inferential scope:** The notebook reports statistical and geometric sensitivity analyses. Full-dictionary selection-adjusted, degree-aware inference, independent retraining, and computational/biological causal validation are not completed here.

### A.5. Exact Annotation-Term Construction Audit

The restricted analysis reconstructed 4,420 functional
gene sets from GO Biological Process, KEGG and Reactome.

The original decoder annotation summary reports
4,422 tested terms.

The original annotation implementation also includes
a curated TRRUST transcription-factor gene set.

To identify the remaining discrepancy, the original
term-construction functions are reused with the
same PIE gene universe and term-size thresholds.

The resulting term identifiers and memberships are
compared against the independently reconstructed
restricted ontology family.

This is a read-only reproducibility check.
No SAE inference, training, or HDF5 loading is required.

In [4]:
import sys
import importlib
from collections import Counter

# --------------------------------------------------
# 1. Load the original annotation implementation
# --------------------------------------------------

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

annotation_module = importlib.import_module(
    "src.sae_biological_annotation"
)

# --------------------------------------------------
# 2. Reproduce original ontology construction
# --------------------------------------------------

original_terms, _ = annotation_module._prepare_terms(
    REFERENCE_DIR,
    universe,
    5,
    500,
)

print("=== ORIGINAL TERM CONSTRUCTION ===")

print(
    "GO/KEGG/Reactome terms:",
    len(original_terms)
)

print(
    "Counts by source:",
    dict(Counter(
        t["source"] for t in original_terms
    ))
)

# --------------------------------------------------
# 3. Reproduce TRRUST term construction
# --------------------------------------------------

_, _, trrust_tfs, _ = (
    annotation_module._load_networks(
        REFERENCE_DIR,
        universe,
    )
)

include_trrust = len(trrust_tfs) >= 5

print("\n=== TRRUST ===")
print("Curated TF genes:", len(trrust_tfs))
print("TRRUST term included:", include_trrust)

full_terms = list(original_terms)

if include_trrust:
    full_terms.append({
        "source": "TRRUST_TF",
        "term": "curated_transcription_factors",
        "genes": trrust_tfs,
    })

print("\n=== FULL ORIGINAL FAMILY ===")

print("Reconstructed original term count:", len(full_terms))
print("Saved original term count:", reported_total)

# --------------------------------------------------
# 4. Compare term identities and memberships
# --------------------------------------------------

original_keys = [
    (str(t["source"]), str(t["term"]))
    for t in original_terms
]

restricted_keys = set(restricted_gene_sets)

extra_keys = set(original_keys) - restricted_keys
missing_keys = restricted_keys - set(original_keys)

duplicates = {
    key: count
    for key, count in Counter(original_keys).items()
    if count > 1
}

print("\n=== ONTOLOGY FAMILY COMPARISON ===")

print("Extra original ontology keys:", len(extra_keys))
print("Missing original ontology keys:", len(missing_keys))
print("Duplicate original keys:", len(duplicates))

for key in sorted(extra_keys)[:10]:
    print("Extra:", key)

for key in sorted(missing_keys)[:10]:
    print("Missing:", key)

for key, count in list(duplicates.items())[:10]:
    print("Duplicate:", key, "count:", count)

original_members = {
    (str(t["source"]), str(t["term"])): set(t["genes"])
    for t in original_terms
}

membership_mismatches = [
    key
    for key in original_members.keys() & restricted_keys
    if original_members[key] != restricted_gene_sets[key]
]

print(
    "Shared terms with different gene memberships:",
    len(membership_mismatches)
)

for key in sorted(membership_mismatches)[:10]:
    print("Membership mismatch:", key)

print("\nRead-only original-term audit completed.")

=== ORIGINAL TERM CONSTRUCTION ===
GO/KEGG/Reactome terms: 4420
Counts by source: {'GO_BP': 2838, 'KEGG': 286, 'Reactome': 1296}

=== TRRUST ===
Curated TF genes: 346
TRRUST term included: True

=== FULL ORIGINAL FAMILY ===
Reconstructed original term count: 4421
Saved original term count: 4422

=== ONTOLOGY FAMILY COMPARISON ===
Extra original ontology keys: 0
Missing original ontology keys: 0
Duplicate original keys: 0
Shared terms with different gene memberships: 0

Read-only original-term audit completed.


### A.6. Reported Hypothesis-Family Size Audit

The original term-construction function reproduces exactly
4,420 GO BP, KEGG and Reactome terms, including their
gene memberships.

Adding the curated TRRUST transcription-factor term
produces 4,421 terms, while the saved decoder annotation
summaries report 4,422.

The remaining discrepancy may reflect an additional
special test or a difference in how the hypothesis-family
size was recorded.

This audit inspects the saved annotation metadata and
the original source code responsible for term counting
and multiple-testing correction.

No original results are modified.

In [5]:
import json
import re

print("=== HYPOTHESIS-FAMILY COUNT AUDIT ===")

annotation_path = (
    ROOT
    / "results/sae_feature_analysis_independent"
    / "biological_annotations"
    / "decoder_0_annotations.json"
)

payload = json.loads(
    annotation_path.read_text(encoding="utf-8")
)

print("\nAnnotation JSON top-level keys:")
print(list(payload.keys()))

# Inspect scalar metadata without printing feature catalogs.

matches = []

def inspect_metadata(obj, path="", depth=0):
    if depth > 5:
        return

    if isinstance(obj, dict):
        for key, value in obj.items():
            current = f"{path}.{key}" if path else key

            if isinstance(value, (int, float, str, bool)):
                name = key.lower()

                if (
                    "term" in name
                    or "hypothes" in name
                    or "test" in name
                    or value == 4422
                ):
                    matches.append((current, value))

            elif isinstance(value, dict):
                inspect_metadata(value, current, depth + 1)

            elif isinstance(value, list) and len(value) <= 10:
                inspect_metadata(value, current, depth + 1)

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            inspect_metadata(value, f"{path}[{i}]", depth + 1)

inspect_metadata(payload)

print("\n=== SAVED COUNT METADATA ===")

for key, value in matches[:30]:
    print(f"{key}: {value}")

# Inspect source-code locations where the
# hypothesis-family size may be constructed.

source_paths = [
    ROOT / "src" / "sae_biological_annotation.py",
    ROOT / "src" / "sae_annotation_null.py",
]

pattern = re.compile(
    r"n_terms|num_terms|total_terms|tested_terms|"
    r"hypothes|total_tests|num_tests|"
    r"len\(terms\)|len\(all_terms\)|"
    r"_prepare_terms|_load_networks|"
    r"TRRUST_TF|trrust_tfs|multipletests",
    re.IGNORECASE,
)

for source_path in source_paths:

    print(f"\n=== SOURCE: {source_path.name} ===")

    if not source_path.exists():
        print("Source file not found.")
        continue

    lines = source_path.read_text(
        encoding="utf-8"
    ).splitlines()

    hit_indices = [
        i for i, line in enumerate(lines)
        if pattern.search(line)
    ]

    print("Matching source lines:", len(hit_indices))

    for i in hit_indices[:35]:
        start = max(0, i - 1)
        end = min(len(lines), i + 2)

        print(f"\nContext around line {i + 1}:")

        for j in range(start, end):
            print(f"{j + 1:4d}: {lines[j]}")

print("\nRead-only hypothesis-family audit completed.")

=== HYPOTHESIS-FAMILY COUNT AUDIT ===

Annotation JSON top-level keys:
['method', 'inputs', 'summary', 'significant_enrichments', 'feature_annotations', 'network_statistics', 'generated_utc']

=== SAVED COUNT METADATA ===
method.term_filter: retain 5..500 member genes after intersecting with PIE vocabulary
method.min_overlap_for_explicit_test: 2
method.omitted_gene_set_tests: assigned p=1 (conservative; exact Fisher p for one-gene overlap is not evaluated)
method.statistical_test: one-sided Fisher exact; vectorized equivalent hypergeometric survival function
method.multiple_testing: Benjamini-Hochberg across ALL gene-set/TF-term tests x 3072 features, per decoder layer
summary.n_ontology_terms: 4422
summary.n_hypotheses_for_BH: 13584384
summary.n_explicit_gene_set_tests_with_overlap_ge_2: 155479
summary.n_significant_feature_term_associations: 18976
feature_annotations.0[0].term: Neddylation R-HSA-8951664
feature_annotations.0[0].term_size_in_PIE: 149
feature_annotations.0[1].term: Pur

### A.7. Hypothesis-Family Count Provenance

The original annotation summaries report 4,422
reference terms per decoder dictionary.

The original ontology construction reproduces
4,420 GO BP, KEGG and Reactome terms. Adding
the curated TRRUST transcription-factor set
accounts for one additional term.

The remaining difference is investigated by
inspecting the exact source expressions used
to construct the hypothesis-family size.

This audit distinguishes the actual reference
terms from the number used in multiple-testing
correction. No data or model artifacts are modified.

In [6]:
import ast
import re

print("=== ORIGINAL TERM-COUNT PROVENANCE ===")

source_files = [
    ROOT / "src" / "sae_biological_annotation.py",
    ROOT / "src" / "sae_annotation_null.py",
]

target_names = {
    "n_terms",
    "n_reference_terms",
    "n_ontology_terms",
    "n_hypotheses_for_BH",
}

for path in source_files:

    print(f"\n=== {path.name} ===")

    if not path.exists():
        print("File not found.")
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)

    # Locate assignments to variables that control
    # the hypothesis-family size.

    assignments = []

    for node in ast.walk(tree):

        if isinstance(node, ast.Assign):
            names = {
                child.id
                for target in node.targets
                for child in ast.walk(target)
                if isinstance(child, ast.Name)
            }

        elif isinstance(node, ast.AnnAssign):
            names = {
                child.id
                for child in ast.walk(node.target)
                if isinstance(child, ast.Name)
            }

        elif isinstance(node, ast.AugAssign):
            names = {
                child.id
                for child in ast.walk(node.target)
                if isinstance(child, ast.Name)
            }

        else:
            continue

        if names & target_names:
            assignments.append(node)

    print("Relevant assignments:", len(assignments))

    for node in assignments:

        start = max(0, node.lineno - 4)
        end = min(len(lines), node.end_lineno + 3)

        print(f"\nLines {start + 1}-{end}:")

        for i in range(start, end):
            print(f"{i + 1:4d}: {lines[i]}")

    # Show references to the saved term count.
    pattern = re.compile(
        r"n_ontology_terms|n_hypotheses_for_BH|"
        r"n_reference_terms|n_terms\s*="
    )

    print("\nSaved-count references:")

    for i, line in enumerate(lines):
        if pattern.search(line):
            print(f"{i + 1:4d}: {line.strip()}")

print("\nRead-only count-provenance audit completed.")

=== ORIGINAL TERM-COUNT PROVENANCE ===

=== sae_biological_annotation.py ===
Relevant assignments: 1

Lines 166-172:
 166:     string_pairs, trrust_edges, trrust_tfs, network_refs = _load_networks(ref_dir, vocabulary)
 167:     if len(trrust_tfs) >= min_term_size:
 168:         terms.append({"source": "TRRUST_TF", "term": "curated_transcription_factors", "genes": trrust_tfs})
 169:     n_terms = len(terms)
 170:     if n_terms == 0:
 171:         raise ValueError("No valid annotation terms after PIE vocabulary filtering")
 172: 

Saved-count references:
 169: n_terms = len(terms)
 170: if n_terms == 0:
 281: "n_ontology_terms": n_terms,
 282: "n_hypotheses_for_BH": m,

=== sae_annotation_null.py ===
Relevant assignments: 2

Lines 89-95:
  89:     rows = overlap.row[mask]
  90:     term_cols = overlap.col[mask]
  91:     counts = overlap.data[mask]
  92:     n_features, n_terms = features_by_genes.shape[0], genes_by_terms.shape[1]
  93:     m = n_features * n_terms
  94:     feature_siz

### A.8. Statistical Protocol Audit — Conclusions

The original term-construction implementation defines
`n_terms` as the length of the term list after the optional
TRRUST transcription-factor set has been appended.

Using the current reference files and PIE vocabulary,
the original functions reconstruct 4,420 GO BP, KEGG
and Reactome terms, plus one TRRUST term.

The resulting total is 4,421, whereas the saved decoder
annotation summaries report 4,422 terms.

The 4,420 reconstructed ontology terms match the original
ontology-construction function in both identifiers and
gene memberships.

The remaining discrepancy cannot be resolved from the
available results. It may reflect historical differences
in reference data, implementation, or saved metadata.

The original BH-adjusted results are therefore retained
as historical outputs, not claimed to have been
independently reproduced.

Subsequent statistical experiments will explicitly define
their hypothesis families and use the current reconstructed
reference sets.

No source data, checkpoints, or original results were
modified during this audit.

## B. Activation Robustness and Bootstrap

### B.1. Reconstruction of Validation Activations

Before estimating statistical uncertainty, the activation
matrix used in the exploratory study is reconstructed.

The analysis focuses initially on SAE feature 0 in
decoder.0 and its ten highest-ranked training-associated
genes across 32 validation conditions.

Only the required hidden-state vectors are read from
the existing HDF5 dataset. The pretrained SAE checkpoint
is loaded for CPU inference without additional training.

The reconstructed common-condition variation statistic
is compared against the result reported in notebook 06a.

This step establishes a reproducible baseline for the
subsequent condition-level bootstrap analysis.

In [7]:
import json
import sys
import h5py
import numpy as np
import torch

FEATURE_ID = 0
LAYER = "decoder.0"
VALIDATION_PATH = ARTIFACTS["Validation HDF5"]

# --------------------------------------------------
# 1. Recover the training-selected gene panel
# --------------------------------------------------

train_catalog_path = (
    CATALOG_DIR / "decoder_0_train_catalog.json"
)

train_catalog = json.loads(
    train_catalog_path.read_text(encoding="utf-8")
)

feature_record = next(
    f for f in train_catalog["features"]
    if int(f["feature_idx"]) == FEATURE_ID
)

top_gene_records = feature_record["top_genes"][:10]

top_gene_ids = [
    int(r["gene_id"]) for r in top_gene_records
]

top_gene_names = [
    str(r["gene_name"]) for r in top_gene_records
]

assert len(top_gene_ids) == len(set(top_gene_ids)) == 10


# --------------------------------------------------
# 2. Read only the required validation vectors
# --------------------------------------------------

with h5py.File(VALIDATION_PATH, "r") as h5:

    assert h5.attrs["split"] == "validation"

    val_condition_ids = np.asarray(
        h5["condition_indices"][:],
        dtype=np.int64
    )

    val_gene_ids = np.asarray(h5["gene_ids"][:])

    decoder = h5["decoder"]["0"]

    assert len(val_condition_ids) == 32
    assert decoder.shape[0] == 32

    vectors = np.empty(
        (32, 10, 768),
        dtype=np.float32
    )

    for c in range(32):

        gene_positions = {
            int(g): p
            for p, g in enumerate(val_gene_ids[c])
        }

        if not all(
            g in gene_positions
            for g in top_gene_ids
        ):
            raise ValueError(
                f"Selected gene missing from condition {c}"
            )

        positions = np.array([
            gene_positions[g]
            for g in top_gene_ids
        ])

        order = np.argsort(positions)

        sorted_vectors = decoder[
            c,
            positions[order].tolist(),
            :
        ]

        vectors[c] = sorted_vectors[
            np.argsort(order)
        ]


# --------------------------------------------------
# 3. Load the existing SAE checkpoint
# --------------------------------------------------

sae_manifest = json.loads(
    ARTIFACTS["SAE manifest"].read_text(
        encoding="utf-8"
    )
)

registry = {
    entry["layer"]: entry
    for entry in sae_manifest["checkpoints"]
}

checkpoint_path = (
    ROOT / registry[LAYER]["checkpoint"]
)

sae_source = ROOT / "external/bio-sae/src"

if str(sae_source) not in sys.path:
    sys.path.insert(0, str(sae_source))

from sae_model import TopKSAE

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

sae = TopKSAE(
    d_model=768,
    n_features=3072,
    k=32
)

sae.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

sae.eval()

activation_mean = np.load(
    checkpoint_path.parent / "activation_mean.npy"
).astype(np.float32)


# --------------------------------------------------
# 4. Reconstruct SAE feature activations
# --------------------------------------------------

with torch.inference_mode():

    centered = torch.from_numpy(
        (vectors - activation_mean).reshape(-1, 768)
    )

    encoded, _ = sae.encode(centered)

    B_06b = (
        encoded[:, FEATURE_ID]
        .cpu()
        .numpy()
        .reshape(32, 10)
        .astype(np.float64)
    )


# --------------------------------------------------
# 5. Reproduce the 06a statistic
# --------------------------------------------------

def common_fraction(matrix):

    centered = (
        matrix
        - matrix.mean(axis=0, keepdims=True)
    )

    total = float(np.sum(centered ** 2))

    if total <= 0:
        return np.nan

    shared = (
        matrix.shape[1]
        * float(np.sum(centered.mean(axis=1) ** 2))
    )

    return shared / total


observed_common = common_fraction(B_06b)

assert B_06b.shape == (32, 10)
assert np.isfinite(B_06b).all()

assert np.isclose(
    observed_common,
    0.55476,
    atol=5e-4
), f"06a baseline mismatch: {observed_common:.6f}"


# --------------------------------------------------
# 6. Report
# --------------------------------------------------

print("=== B.1: RECONSTRUCTED SAE ACTIVATIONS ===")

print("Layer and feature:", LAYER, FEATURE_ID)
print("Selected genes:", top_gene_names)

print("Validation conditions:", len(val_condition_ids))
print("Activation matrix shape:", B_06b.shape)

print(
    "Observed common fraction:",
    f"{100 * observed_common:.3f}%"
)

print("Comparison with 06a: PASS")
print("Read-only reconstruction completed.")

=== B.1: RECONSTRUCTED SAE ACTIVATIONS ===
Layer and feature: decoder.0 0
Selected genes: ['PPCS', 'WSB1', 'WSB2', 'ASB1', 'CCS', 'ASB3', 'PPCDC', 'PSPH', 'PRPSAP1', 'ASB8']
Validation conditions: 32
Activation matrix shape: (32, 10)
Observed common fraction: 55.476%
Comparison with 06a: PASS
Read-only reconstruction completed.


### B.2. Context-Stratified Bootstrap of Activation Statistics

The previous analysis reconstructed the 32 × 10 activation
matrix for SAE feature 0 and reproduced the 06a
common-condition variation fraction.

This experiment estimates the sensitivity of two statistics
to the sampled validation conditions:

1. The original common-condition variation fraction.
2. The common-condition fraction after removing
   gene-specific cell-context means.

Experimental conditions are resampled with replacement
within HepG2, Jurkat, and RPE1. Each resampling unit
contains the full activation vector across all ten genes.

The analysis uses 2,000 bootstrap replicates and reports
percentile intervals.

These intervals are conditional on the selected conditions,
fixed gene panel, and assumed exchangeability within
cell contexts. They do not represent uncertainty across
all possible perturbations or cell types.

In [8]:
import os
import sys
import json
import numpy as np
from collections import Counter

# --------------------------------------------------
# 1. Recover original PIE condition metadata
# --------------------------------------------------

os.environ["PIE_DATA_ROOT"] = str(ROOT / "data")
os.environ["PIE_CACHE_DIR"] = str(ROOT / "cache")
os.environ["PIE_RUNS_ROOT"] = str(ROOT / "runs")
os.environ["HF_HUB_OFFLINE"] = "1"

PIE_SRC = ROOT / "external/pie/src"

if str(PIE_SRC) not in sys.path:
    sys.path.insert(0, str(PIE_SRC))

from pie.predict import load_checkpoint
from pie.data.datamodule import PieDataModule

pie_checkpoint_path = (
    ROOT
    / "models/replogle_wdataset/k562/best_auprc.ckpt"
)

loaded_pie = load_checkpoint(pie_checkpoint_path)

dm_boot = PieDataModule(
    cfg=loaded_pie.config.data,
    run_dir=None,
    stats=loaded_pie.stats,
    seed=loaded_pie.config.seed,
    evidence_source=loaded_pie.config.data,
)

dm_boot.setup_stats(rank=0)
dm_boot.setup(stage="fit")

extraction_manifest = json.loads(
    ARTIFACTS["Extraction manifest"].read_text(
        encoding="utf-8"
    )
)

recorded_val = (
    extraction_manifest["extraction"]["datasets"]["validation"]
)

assert len(dm_boot.val_dataset) == int(
    recorded_val["dataset_size"]
)

assert np.array_equal(
    val_condition_ids,
    np.asarray(
        recorded_val["condition_indices"],
        dtype=np.int64
    )
)

contexts_boot = np.array([
    str(dm_boot.val_dataset.rows[int(idx)].context)
    for idx in val_condition_ids
])

assert len(contexts_boot) == len(B_06b) == 32

groups_boot = {
    context: np.flatnonzero(contexts_boot == context)
    for context in np.unique(contexts_boot)
}

print("=== VALIDATION CELL CONTEXTS ===")
print(dict(Counter(contexts_boot)))

# --------------------------------------------------
# 2. Define both statistics
# --------------------------------------------------

def original_common_fraction(matrix):

    C = matrix - matrix.mean(
        axis=0, keepdims=True
    )

    denominator = np.sum(C ** 2)

    if denominator <= 0:
        return np.nan

    numerator = (
        matrix.shape[1]
        * np.sum(C.mean(axis=1) ** 2)
    )

    return float(numerator / denominator)


def within_context_common_fraction(matrix, groups):

    C = matrix - matrix.mean(
        axis=0, keepdims=True
    )

    W = C.copy()

    for indices in groups.values():
        W[indices] -= W[indices].mean(
            axis=0, keepdims=True
        )

    denominator = np.sum(W ** 2)

    if denominator <= 0:
        return np.nan

    numerator = (
        matrix.shape[1]
        * np.sum(W.mean(axis=1) ** 2)
    )

    return float(numerator / denominator)


# --------------------------------------------------
# 3. Calculate observed statistics
# --------------------------------------------------

observed_original = original_common_fraction(B_06b)

observed_within = within_context_common_fraction(
    B_06b,
    groups_boot
)

print("\n=== OBSERVED STATISTICS ===")

print(
    "Original common fraction:",
    f"{100 * observed_original:.3f}%"
)

print(
    "Within-context common fraction:",
    f"{100 * observed_within:.3f}%"
)

assert np.isclose(
    observed_original, 0.55476, atol=5e-4
)

assert np.isclose(
    observed_within, 0.68854, atol=5e-4
)

# --------------------------------------------------
# 4. Context-stratified paired bootstrap
# --------------------------------------------------

rng = np.random.default_rng(20261113)

N_BOOT = 2000

bootstrap_original = np.empty(N_BOOT)
bootstrap_within = np.empty(N_BOOT)

for b in range(N_BOOT):

    sampled_indices = np.arange(
        len(B_06b), dtype=np.int64
    )

    for indices in groups_boot.values():

        sampled_indices[indices] = rng.choice(
            indices,
            size=len(indices),
            replace=True
        )

    sampled_matrix = B_06b[sampled_indices]

    bootstrap_original[b] = (
        original_common_fraction(sampled_matrix)
    )

    bootstrap_within[b] = (
        within_context_common_fraction(
            sampled_matrix,
            groups_boot
        )
    )

assert np.isfinite(bootstrap_original).all()
assert np.isfinite(bootstrap_within).all()

# --------------------------------------------------
# 5. Report bootstrap intervals
# --------------------------------------------------

def report_bootstrap(name, observed, samples):

    lower, upper = np.percentile(
        samples, [2.5, 97.5]
    )

    print(f"\n=== {name} ===")
    print(f"Observed: {100 * observed:.3f}%")
    print(f"Bootstrap mean: {100 * samples.mean():.3f}%")
    print(f"Bootstrap SD: {100 * samples.std(ddof=1):.3f}%")

    print(
        "95% percentile interval:",
        f"[{100 * lower:.3f}%, {100 * upper:.3f}%]"
    )


print("\n=== STRATIFIED BOOTSTRAP ===")
print("Replicates:", N_BOOT)

report_bootstrap(
    "Original common variation",
    observed_original,
    bootstrap_original
)

report_bootstrap(
    "Within-context common variation",
    observed_within,
    bootstrap_within
)

print("\nBootstrap analysis completed.")

d:\bio_paper\pie-sae\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== VALIDATION CELL CONTEXTS ===
{np.str_('hepg2'): 10, np.str_('jurkat'): 13, np.str_('rpe1'): 9}

=== OBSERVED STATISTICS ===
Original common fraction: 55.476%
Within-context common fraction: 68.854%

=== STRATIFIED BOOTSTRAP ===
Replicates: 2000

=== Original common variation ===
Observed: 55.476%
Bootstrap mean: 54.420%
Bootstrap SD: 5.999%
95% percentile interval: [41.133%, 64.680%]

=== Within-context common variation ===
Observed: 68.854%
Bootstrap mean: 67.862%
Bootstrap SD: 5.647%
95% percentile interval: [54.998%, 76.907%]

Bootstrap analysis completed.


### B.2.1. Bootstrap Results and Interpretation

The original common-condition variation fraction was
55.476%, with a context-stratified bootstrap percentile
interval of [41.133%, 64.680%].

After removing gene-specific cell-context means,
the within-context common-condition fraction was
68.854%, with a bootstrap percentile interval of
[54.998%, 76.907%].

Both analyses used 2,000 bootstrap replicates.

The bootstrap means were 54.420% and 67.862%,
respectively, indicating modest downward shifts
relative to the observed estimates.

These results show that the estimated common-condition
variation is sensitive to the composition of the
32 validation conditions.

The two percentages use different variance
denominators and should not be interpreted
as a direct before-and-after improvement.

The percentile intervals are conditional on the
fixed gene panel, the three observed cell contexts,
and the assumption that conditions are exchangeable
within each context.

They do not establish SAE specificity, biological
causality, or generalization to unobserved cell types.

### B.3. Cross-Split Stability and Catalog Completeness

This analysis evaluates the stability of feature–gene
associations across training and validation conditions
for all four decoder SAE dictionaries.

First, the actual number of ranked genes available
for each feature is recorded separately for both splits.

Some features have fewer than 20 available gene
associations. These cases are reported explicitly
rather than treated as complete Top-20 lists.

The primary stability comparison is restricted to
features with 20 distinct gene identifiers in both
training and validation catalogs.

For eligible features, stability is summarized using
the number of shared genes and Jaccard similarity.

The incomplete catalogs are analyzed separately.
Their exclusion may introduce selection bias because
catalog completeness can depend on feature activity.

Consequently, the primary results describe stability
conditional on complete Top-20 coverage, not stability
of the entire SAE dictionary.

This experiment uses existing feature catalogs and
does not repeat SAE inference or training.

In [9]:
import json
import numpy as np
import pandas as pd

print("=== B.3. CROSS-SPLIT GENE ASSOCIATION STABILITY ===")

catalog_dir = (
    ROOT
    / "results"
    / "sae_feature_analysis_independent"
    / "feature_catalogs"
)

pilot_features = [
    0, 2678, 2586, 189, 672,
    1508, 3027, 2120, 240, 3021
]

records = []

# --------------------------------------------------
# 1. Load and compare feature catalogs
# --------------------------------------------------

for layer_num in range(4):

    train_path = (
        catalog_dir
        / f"decoder_{layer_num}_train_catalog.json"
    )

    val_path = (
        catalog_dir
        / f"decoder_{layer_num}_validation_catalog.json"
    )

    if not val_path.exists():
        val_path = (
            catalog_dir
            / f"decoder_{layer_num}_val_catalog.json"
        )

    if not train_path.exists() or not val_path.exists():
        raise FileNotFoundError(
            f"Missing catalogs for decoder.{layer_num}"
        )

    train_data = json.loads(
        train_path.read_text(encoding="utf-8")
    )

    val_data = json.loads(
        val_path.read_text(encoding="utf-8")
    )

    train_by_id = {
        int(f["feature_idx"]): f
        for f in train_data["features"]
    }

    val_by_id = {
        int(f["feature_idx"]): f
        for f in val_data["features"]
    }

    assert len(train_by_id) == 3072
    assert len(val_by_id) == 3072
    assert set(train_by_id) == set(val_by_id)

    layer_count = 0

    for feature_id in sorted(train_by_id):

        train_ids = [
            int(g["gene_id"])
            for g in train_by_id[feature_id]["top_genes"][:20]
        ]

        val_ids = [
            int(g["gene_id"])
            for g in val_by_id[feature_id]["top_genes"][:20]
        ]

        train_set = set(train_ids)
        val_set = set(val_ids)

        if len(train_set) != len(train_ids):
            raise ValueError(
                f"Duplicate train IDs: "
                f"decoder.{layer_num}, {feature_id}"
            )

        if len(val_set) != len(val_ids):
            raise ValueError(
                f"Duplicate validation IDs: "
                f"decoder.{layer_num}, {feature_id}"
            )

        shared = len(train_set & val_set)
        union = len(train_set | val_set)

        complete = (
            len(train_set) == 20
            and len(val_set) == 20
        )

        # IMPORTANT:
        # This append belongs inside the feature loop,
        # but outside both duplicate-ID checks.

        records.append({
            "layer": f"decoder.{layer_num}",
            "feature": feature_id,
            "train_genes": len(train_set),
            "val_genes": len(val_set),
            "complete_pair": complete,
            "shared_genes": shared,
            "jaccard": shared / union if union else np.nan,
        })

        layer_count += 1

    print(
        f"decoder.{layer_num}: "
        f"{layer_count} feature pairs processed"
    )

    assert layer_count == 3072

# --------------------------------------------------
# 2. Validate total record coverage
# --------------------------------------------------

stability_df = pd.DataFrame(records)

assert len(stability_df) == 12288

assert (
    stability_df.groupby("layer").size() == 3072
).all()

print("\nTotal feature pairs:", len(stability_df))

# --------------------------------------------------
# 3. Catalog completeness audit
# --------------------------------------------------

coverage = stability_df.groupby("layer").agg(
    features=("feature", "count"),
    complete_pairs=("complete_pair", "sum"),
    train_short=("train_genes", lambda x: (x < 20).sum()),
    val_short=("val_genes", lambda x: (x < 20).sum()),
    train_empty=("train_genes", lambda x: (x == 0).sum()),
    val_empty=("val_genes", lambda x: (x == 0).sum()),
)

coverage["complete_fraction"] = (
    coverage["complete_pairs"] / coverage["features"]
)

print("\n=== CATALOG COMPLETENESS ===")
print(coverage.round(4).to_string())

# --------------------------------------------------
# 4. Stability for complete Top-20 pairs
# --------------------------------------------------

complete_df = stability_df[
    stability_df["complete_pair"]
].copy()

print("\n=== COMPLETE TOP-20 PAIRS ===")

if not complete_df.empty:

    summary = complete_df.groupby("layer").agg(
        features=("feature", "count"),
        mean_shared=("shared_genes", "mean"),
        median_shared=("shared_genes", "median"),
        mean_jaccard=("jaccard", "mean"),
        fraction_ge_5=(
            "shared_genes",
            lambda x: (x >= 5).mean()
        ),
        fraction_ge_10=(
            "shared_genes",
            lambda x: (x >= 10).mean()
        ),
    )

    print(summary.round(4).to_string())

else:
    print("No complete feature pairs.")

# --------------------------------------------------
# 5. Original ten-feature pilot
# --------------------------------------------------

pilot_df = stability_df[
    (stability_df["layer"] == "decoder.0")
    & stability_df["feature"].isin(pilot_features)
]

print("\n=== TEN-FEATURE PILOT ===")

print(
    pilot_df.sort_values("feature").to_string(index=False)
)

# --------------------------------------------------
# 6. Incomplete feature catalogs
# --------------------------------------------------

incomplete_df = stability_df[
    ~stability_df["complete_pair"]
]

print("\n=== INCOMPLETE CATALOGS ===")

print("Incomplete pairs:", len(incomplete_df))

if not incomplete_df.empty:
    print(
        incomplete_df[
            ["layer", "feature", "train_genes",
             "val_genes", "shared_genes"]
        ].head(15).to_string(index=False)
    )

print("\n=== PREVIOUS DIAGNOSTIC CASE ===")

print(
    stability_df.loc[
        (stability_df["layer"] == "decoder.2")
        & (stability_df["feature"] == 1333)
    ].to_string(index=False)
)

print("\nRead-only cross-split audit completed.")

=== B.3. CROSS-SPLIT GENE ASSOCIATION STABILITY ===
decoder.0: 3072 feature pairs processed
decoder.1: 3072 feature pairs processed
decoder.2: 3072 feature pairs processed
decoder.3: 3072 feature pairs processed

Total feature pairs: 12288

=== CATALOG COMPLETENESS ===
           features  complete_pairs  train_short  val_short  train_empty  val_empty  complete_fraction
layer                                                                                                 
decoder.0      3072            3072            0          0            0          0             1.0000
decoder.1      3072            3072            0          0            0          0             1.0000
decoder.2      3072            3070            0          2            0          0             0.9993
decoder.3      3072            3071            0          1            0          0             0.9997

=== COMPLETE TOP-20 PAIRS ===
           features  mean_shared  median_shared  mean_jaccard  fraction_ge_5  fra

### B.3.1. Catalog Coverage Verification

The cross-split analysis identified three feature pairs
with incomplete validation Top-20 lists.

This audit verifies their identities directly against
the saved catalogs and reconciles the result with the
initial catalog inventory.

The previously selected ten-feature pilot is also
reported separately.

No activation data are loaded and no source files
are modified.

In [10]:
import json
import pandas as pd

print("=== CATALOG COVERAGE VERIFICATION ===")

for layer_num in range(4):

    path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_validation_catalog.json"
    )

    catalog = json.loads(
        path.read_text(encoding="utf-8")
    )

    counts = {
        int(f["feature_idx"]): len(f["top_genes"])
        for f in catalog["features"]
    }

    incomplete = {
        fid: count
        for fid, count in counts.items()
        if count < 20
    }

    measured = stability_df.loc[
        stability_df["layer"] == f"decoder.{layer_num}",
        ["feature", "val_genes"]
    ].set_index("feature")["val_genes"].to_dict()

    mismatches = [
        fid for fid, count in counts.items()
        if measured.get(fid) != count
    ]

    print(f"\ndecoder.{layer_num}")
    print("Catalog features:", len(counts))
    print("Features with 20 genes:",
          sum(count == 20 for count in counts.values()))
    print("Incomplete features:", incomplete)
    print("Disagreements with B.3:", len(mismatches))

    assert not mismatches

print("\n=== ORIGINAL TEN-FEATURE PILOT ===")

pilot_ids = [
    0, 2678, 2586, 189, 672,
    1508, 3027, 2120, 240, 3021
]

pilot_check = stability_df.loc[
    stability_df["layer"].eq("decoder.0")
    & stability_df["feature"].isin(pilot_ids)
].copy()

assert len(pilot_check) == 10

print(
    pilot_check.sort_values("feature").to_string(
        index=False
    )
)

print("\nCatalog verification completed.")

=== CATALOG COVERAGE VERIFICATION ===

decoder.0
Catalog features: 3072
Features with 20 genes: 3072
Incomplete features: {}
Disagreements with B.3: 0

decoder.1
Catalog features: 3072
Features with 20 genes: 3072
Incomplete features: {}
Disagreements with B.3: 0

decoder.2
Catalog features: 3072
Features with 20 genes: 3070
Incomplete features: {1333: 4, 2817: 18}
Disagreements with B.3: 0

decoder.3
Catalog features: 3072
Features with 20 genes: 3071
Incomplete features: {1365: 8}
Disagreements with B.3: 0

=== ORIGINAL TEN-FEATURE PILOT ===
    layer  feature  train_genes  val_genes  complete_pair  shared_genes  jaccard
decoder.0        0           20         20           True            20 1.000000
decoder.0      189           20         20           True            19 0.904762
decoder.0      240           20         20           True            19 0.904762
decoder.0      672           20         20           True            19 0.904762
decoder.0     1508           20         20   

### B.3.2. Cross-Split Stability Results

The train and validation Top-20 catalogs were compared
for all 12,288 decoder SAE features.

A total of 12,285 feature pairs had complete Top-20
gene lists in both splits.

Three validation catalogs were incomplete:

- decoder.2, feature 1333: 4 genes.
- decoder.2, feature 2817: 18 genes.
- decoder.3, feature 1365: 8 genes.

Direct inspection of the saved JSON catalogs confirmed
these counts, with no discrepancies relative to the
cross-split comparison.

Among complete feature pairs, the average numbers
of shared Top-20 genes were:

| Layer | Mean shared genes | Mean Jaccard |
|---|---:|---:|
| decoder.0 | 19.6816 | 0.9758 |
| decoder.1 | 19.3844 | 0.9590 |
| decoder.2 | 19.0873 | 0.9411 |
| decoder.3 | 18.7607 | 0.9227 |

These results demonstrate strong descriptive stability
of the saved gene-association catalogs across splits.

However, stable gene rankings may reflect persistent
gene-identity effects rather than perturbation-sensitive
biological features.

A feature-matching permutation control is therefore
needed to establish whether corresponding SAE features
agree more strongly than randomly paired features from
the same dictionary.

This control does not replace condition-level bootstrap
or establish biological causality.

### B.4. Feature-Matching Permutation Control

The high cross-split overlap of gene-association
catalogs may arise partly from recurrent genes
shared by multiple SAE features.

This experiment tests whether the correctly matched
train-validation feature pairs exhibit greater
Top-20 overlap than randomly reassigned feature pairs.

For each decoder dictionary:

1. Retain features with complete Top-20 gene lists
   in both training and validation catalogs.
2. Calculate the mean overlap for correctly
   matched feature identifiers.
3. Randomly permute validation feature identifiers,
   preserving the complete collection of gene sets.
4. Recalculate the mean overlap for 1,000 permutations.
5. Estimate the empirical probability of obtaining
   an overlap at least as large as observed.

The permutation preserves gene-set frequencies and
the number of selected features within each dictionary.

It does not control for feature activity, differences
between cell contexts, or perturbation sensitivity.

The test evaluates feature-pair specificity of saved
gene rankings, not biological causality.

In [11]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

print("=== B.4. FEATURE-MATCHING PERMUTATION CONTROL ===")

rng = np.random.default_rng(20261113)
N_PERM = 1000

results_b4 = []

for layer_num in range(4):

    train_path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_train_catalog.json"
    )

    val_path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_validation_catalog.json"
    )

    if not val_path.exists():
        val_path = (
            CATALOG_DIR
            / f"decoder_{layer_num}_val_catalog.json"
        )

    train_data = json.loads(
        train_path.read_text(encoding="utf-8")
    )

    val_data = json.loads(
        val_path.read_text(encoding="utf-8")
    )

    train_features = {
        int(f["feature_idx"]): f
        for f in train_data["features"]
    }

    val_features = {
        int(f["feature_idx"]): f
        for f in val_data["features"]
    }

    assert set(train_features) == set(val_features)

    # ----------------------------------------------
    # 1. Retain complete Top-20 feature pairs
    # ----------------------------------------------

    feature_ids = stability_df.loc[
        (stability_df["layer"] == f"decoder.{layer_num}")
        & stability_df["complete_pair"],
        "feature"
    ].sort_values().tolist()

    train_sets = [
        {
            int(g["gene_id"])
            for g in train_features[fid]["top_genes"][:20]
        }
        for fid in feature_ids
    ]

    val_sets = [
        {
            int(g["gene_id"])
            for g in val_features[fid]["top_genes"][:20]
        }
        for fid in feature_ids
    ]

    n_features = len(feature_ids)

    assert all(len(s) == 20 for s in train_sets)
    assert all(len(s) == 20 for s in val_sets)

    # ----------------------------------------------
    # 2. Construct sparse gene-membership matrices
    # ----------------------------------------------

    all_genes = sorted(
        set().union(*train_sets, *val_sets)
    )

    gene_lookup = {
        gene: idx
        for idx, gene in enumerate(all_genes)
    }

    def membership_matrix(gene_sets):

        row_indices = []
        col_indices = []

        for row, genes in enumerate(gene_sets):
            for gene in genes:
                row_indices.append(row)
                col_indices.append(gene_lookup[gene])

        return csr_matrix(
            (
                np.ones(
                    len(row_indices),
                    dtype=np.int16
                ),
                (row_indices, col_indices)
            ),
            shape=(len(gene_sets), len(all_genes)),
            dtype=np.int16
        )

    T = membership_matrix(train_sets)
    V = membership_matrix(val_sets)

    # ----------------------------------------------
    # 3. Correct feature pairing
    # ----------------------------------------------

    observed_overlap = np.asarray(
        T.multiply(V).sum(axis=1)
    ).ravel()

    observed_mean = float(
        observed_overlap.mean()
    )

    # ----------------------------------------------
    # 4. Random feature reassignment
    # ----------------------------------------------

    null_means = np.empty(N_PERM)

    for b in range(N_PERM):

        permutation = rng.permutation(n_features)

        null_means[b] = (
            T.multiply(V[permutation]).sum()
            / n_features
        )

    empirical_p = (
        1 + np.count_nonzero(
            null_means >= observed_mean
        )
    ) / (N_PERM + 1)

    results_b4.append({
        "layer": f"decoder.{layer_num}",
        "features": n_features,
        "observed_mean": observed_mean,
        "null_mean": float(null_means.mean()),
        "null_95th": float(
            np.percentile(null_means, 95)
        ),
        "empirical_p": empirical_p,
    })

    print(f"\n=== decoder.{layer_num} ===")
    print("Eligible features:", n_features)

    print(
        "Correct-pair mean overlap:",
        f"{observed_mean:.4f}"
    )

    print(
        "Random-pair mean overlap:",
        f"{null_means.mean():.4f}"
    )

    print(
        "Random-pair 95th percentile:",
        f"{np.percentile(null_means, 95):.4f}"
    )

    print(
        "Empirical p-value:",
        f"{empirical_p:.6f}"
    )

# --------------------------------------------------
# 5. Consolidated results
# --------------------------------------------------

b4_summary = pd.DataFrame(results_b4)

print("\n=== FEATURE-MATCHING CONTROL SUMMARY ===")

print(
    b4_summary.round(6).to_string(index=False)
)

print("\nRead-only feature-matching control completed.")

=== B.4. FEATURE-MATCHING PERMUTATION CONTROL ===

=== decoder.0 ===
Eligible features: 3072
Correct-pair mean overlap: 19.6816
Random-pair mean overlap: 0.0677
Random-pair 95th percentile: 0.0817
Empirical p-value: 0.000999

=== decoder.1 ===
Eligible features: 3072
Correct-pair mean overlap: 19.3844
Random-pair mean overlap: 0.0673
Random-pair 95th percentile: 0.0820
Empirical p-value: 0.000999

=== decoder.2 ===
Eligible features: 3070
Correct-pair mean overlap: 19.0873
Random-pair mean overlap: 0.0672
Random-pair 95th percentile: 0.0824
Empirical p-value: 0.000999

=== decoder.3 ===
Eligible features: 3071
Correct-pair mean overlap: 18.7607
Random-pair mean overlap: 0.0678
Random-pair 95th percentile: 0.0827
Empirical p-value: 0.000999

=== FEATURE-MATCHING CONTROL SUMMARY ===
    layer  features  observed_mean  null_mean  null_95th  empirical_p
decoder.0      3072      19.681641   0.067722   0.081722     0.000999
decoder.1      3072      19.384440   0.067276   0.082031     0.00099

### B.4.1. Feature-Matching Permutation Results

Correctly matched SAE features exhibited substantially
greater Top-20 gene overlap across training and validation
conditions than randomly paired features.

| Layer | Correct pairing | Random pairing | Empirical p |
|---|---:|---:|---:|
| decoder.0 | 19.6816 | 0.0677 | 0.000999 |
| decoder.1 | 19.3844 | 0.0673 | 0.000999 |
| decoder.2 | 19.0873 | 0.0672 | 0.000999 |
| decoder.3 | 18.7607 | 0.0678 | 0.000999 |

Each null distribution was estimated using 1,000
permutations of validation feature identifiers.

The results indicate that cross-split gene-set agreement
is strongly specific to corresponding feature identifiers
under the selected matching null model.

However, the same trained SAE dictionaries were applied
to both splits. The result does not establish feature
replication across independently trained SAE models.

High agreement may also reflect persistent gene-identity
effects rather than sensitivity to genetic perturbations.

No biological or computational causal conclusions follow
from this matching experiment alone.

### B.5. Cross-Split Gene-Ranking Stability

The preceding analyses established strong overlap
between training and validation Top-20 gene sets.

However, set overlap does not measure the consistency
of gene ordering within the ranked lists.

This experiment evaluates three complementary measures:

1. Top-5 gene-set overlap.
2. Spearman rank correlation among genes shared
   by the training and validation Top-20 lists.
3. The fraction of features with identical
   complete Top-20 ordering.

The analysis includes all decoder features with
complete Top-20 lists in both splits.

Rank correlations are descriptive and conditional
on the shared gene subset. No confirmatory p-values
are inferred from these correlations.

The subsequent bootstrap analysis will examine
the sensitivity of gene rankings to resampling
experimental conditions.

In [12]:
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=== B.5. CROSS-SPLIT GENE-RANKING STABILITY ===")

ranking_records = []

for layer_num in range(4):

    train_path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_train_catalog.json"
    )

    val_path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_validation_catalog.json"
    )

    if not val_path.exists():
        val_path = (
            CATALOG_DIR
            / f"decoder_{layer_num}_val_catalog.json"
        )

    train_data = json.loads(
        train_path.read_text(encoding="utf-8")
    )

    val_data = json.loads(
        val_path.read_text(encoding="utf-8")
    )

    train_by_id = {
        int(f["feature_idx"]): f
        for f in train_data["features"]
    }

    val_by_id = {
        int(f["feature_idx"]): f
        for f in val_data["features"]
    }

    assert set(train_by_id) == set(val_by_id)

    for fid in sorted(train_by_id):

        train_ids = [
            int(g["gene_id"])
            for g in train_by_id[fid]["top_genes"][:20]
        ]

        val_ids = [
            int(g["gene_id"])
            for g in val_by_id[fid]["top_genes"][:20]
        ]

        if len(train_ids) != 20 or len(val_ids) != 20:
            continue

        if len(set(train_ids)) != 20:
            raise ValueError(
                f"Duplicate train IDs: decoder.{layer_num}/{fid}"
            )

        if len(set(val_ids)) != 20:
            raise ValueError(
                f"Duplicate validation IDs: decoder.{layer_num}/{fid}"
            )

        train_ranks = {
            gene: rank
            for rank, gene in enumerate(train_ids, start=1)
        }

        val_ranks = {
            gene: rank
            for rank, gene in enumerate(val_ids, start=1)
        }

        shared = set(train_ranks) & set(val_ranks)

        top5_shared = len(
            set(train_ids[:5]) & set(val_ids[:5])
        )

        if len(shared) >= 3:

            genes = sorted(shared)

            train_rank_values = [
                train_ranks[g] for g in genes
            ]

            val_rank_values = [
                val_ranks[g] for g in genes
            ]

            rho = float(
                spearmanr(
                    train_rank_values,
                    val_rank_values
                ).statistic
            )

        else:
            rho = np.nan

        ranking_records.append({
            "layer": f"decoder.{layer_num}",
            "feature": fid,
            "shared_top20": len(shared),
            "shared_top5": top5_shared,
            "spearman_shared": rho,
            "identical_order": train_ids == val_ids,
        })

ranking_df = pd.DataFrame(ranking_records)

assert len(ranking_df) == int(
    stability_df["complete_pair"].sum()
)

print("\n=== RANKING STABILITY BY LAYER ===")

ranking_summary = ranking_df.groupby("layer").agg(
    features=("feature", "count"),
    mean_top5_overlap=("shared_top5", "mean"),
    median_top5_overlap=("shared_top5", "median"),
    mean_spearman=("spearman_shared", "mean"),
    median_spearman=("spearman_shared", "median"),
    identical_order_fraction=("identical_order", "mean"),
)

print(ranking_summary.round(4).to_string())

print("\n=== ORIGINAL TEN-FEATURE PILOT ===")

pilot_ids = [
    0, 2678, 2586, 189, 672,
    1508, 3027, 2120, 240, 3021
]

pilot_rankings = ranking_df.loc[
    ranking_df["layer"].eq("decoder.0")
    & ranking_df["feature"].isin(pilot_ids)
].sort_values("feature")

assert len(pilot_rankings) == 10

print(pilot_rankings.to_string(index=False))

print("\n=== LOW RANK-STABILITY EXAMPLES ===")

low_rank = ranking_df.sort_values(
    "spearman_shared",
    ascending=True
).head(10)

print(low_rank.to_string(index=False))

print("\nRead-only ranking stability analysis completed.")

=== B.5. CROSS-SPLIT GENE-RANKING STABILITY ===

=== RANKING STABILITY BY LAYER ===
           features  mean_top5_overlap  median_top5_overlap  mean_spearman  median_spearman  identical_order_fraction
layer                                                                                                                
decoder.0      3072             4.9170                  5.0         0.9946           0.9982                    0.1644
decoder.1      3072             4.8298                  5.0         0.9916           0.9970                    0.0980
decoder.2      3070             4.7635                  5.0         0.9901           0.9955                    0.0658
decoder.3      3071             4.6838                  5.0         0.9866           0.9955                    0.0511

=== ORIGINAL TEN-FEATURE PILOT ===
    layer  feature  shared_top20  shared_top5  spearman_shared  identical_order
decoder.0        0            20            5         1.000000             True
decoder.0   

### B.5.1. Gene-Ranking Stability Results

The analysis evaluated gene-ranking consistency across
training and validation conditions for 12,285 decoder
SAE features with complete Top-20 catalogs.

| Layer | Mean Top-5 overlap | Mean Spearman rho | Identical order |
|---|---:|---:|---:|
| decoder.0 | 4.9170 | 0.9946 | 16.44% |
| decoder.1 | 4.8298 | 0.9916 | 9.80% |
| decoder.2 | 4.7635 | 0.9901 | 6.58% |
| decoder.3 | 4.6838 | 0.9866 | 5.11% |

Most features exhibit strong cross-split agreement
in their highest-ranked genes and relative gene order.

The Spearman correlations are computed conditionally
on genes shared between the two Top-20 lists.
Consequently, these averages should not be interpreted
as unconditional stability across all features.

### Notable exception: decoder.0 feature 2120

Feature 2120 has only one gene in common between its
training and validation Top-20 lists and no shared
genes in the Top-5 lists.

Its Spearman correlation is undefined because the
shared set contains fewer than three genes.

This is notable because feature 2120 previously passed
both enrichment permutation controls in notebook 06a.

The result illustrates that statistically unusual
functional enrichment does not necessarily imply
stable feature–gene associations across conditions.

Feature 2120 should therefore be treated as a
lower-confidence candidate until additional
replication and activation-stability tests are completed.

### Interpretation

The high overall cross-split agreement is consistent
with stable gene-associated activation patterns.

However, the same trained SAE dictionary is used for
both splits, and persistent gene-identity effects may
explain part of this stability.

Condition-level resampling is needed to assess the
uncertainty of the underlying association estimates.

### B.6. Gene-Association Score Provenance

The previous experiments established high overall
cross-split stability of gene-association rankings,
while identifying several unstable features.

Before performing condition-level bootstrap, the
original feature-catalog construction procedure must
be examined.

This audit identifies:

1. The statistic used to rank genes for each SAE feature.
2. The aggregation procedure across experimental conditions.
3. The criteria used to include genes in the Top-20 catalog.
4. The handling of inactive and rarely activated features.

The bootstrap analysis will reproduce the original
association statistic rather than resampling already
selected Top-20 lists.

This is a read-only source-code and catalog inspection.

In [13]:
import json
import re
from pathlib import Path

print("=== B.6. GENE-ASSOCIATION SCORE PROVENANCE ===")

# --------------------------------------------------
# 1. Inspect saved feature-catalog structure
# --------------------------------------------------

catalog_path = (
    CATALOG_DIR
    / "decoder_0_train_catalog.json"
)

catalog = json.loads(
    catalog_path.read_text(encoding="utf-8")
)

print("\n=== CATALOG STRUCTURE ===")

print("Top-level keys:", list(catalog.keys()))

for fid in [0, 189, 2120]:

    feature = next(
        f for f in catalog["features"]
        if int(f["feature_idx"]) == fid
    )

    print(f"\nFeature {fid}")

    print("Feature keys:", list(feature.keys()))

    print("Top-gene records:", len(feature["top_genes"]))

    print("First three gene records:")

    for record in feature["top_genes"][:3]:
        print(json.dumps(record, ensure_ascii=False))


# --------------------------------------------------
# 2. Locate catalog-construction source code
# --------------------------------------------------

search_roots = [
    ROOT / "src",
    ROOT / "scripts",
]

patterns = [
    re.compile(r"top_genes", re.IGNORECASE),
    re.compile(r"feature_catalog", re.IGNORECASE),
    re.compile(r"gene_score", re.IGNORECASE),
    re.compile(r"gene_rank", re.IGNORECASE),
]

candidates = []

for search_root in search_roots:

    if not search_root.exists():
        continue

    for path in search_root.rglob("*.py"):

        source = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        lines = source.splitlines()

        hits = [
            i for i, line in enumerate(lines)
            if any(
                pattern.search(line)
                for pattern in patterns
            )
        ]

        if hits:
            candidates.append(
                (len(hits), path, lines, hits)
            )

candidates.sort(
    key=lambda item: item[0],
    reverse=True
)

print("\n=== CATALOG SOURCE CANDIDATES ===")

for count, path, _, _ in candidates[:12]:

    print(
        f"{path.relative_to(ROOT)}: "
        f"{count} matching lines"
    )


# --------------------------------------------------
# 3. Inspect relevant implementation excerpts
# --------------------------------------------------

print("\n=== SOURCE-CODE EXCERPTS ===")

for count, path, lines, hits in candidates[:4]:

    print(f"\nFILE: {path.relative_to(ROOT)}")

    # Limit output size for notebook readability.
    for index in hits[:5]:

        start = max(0, index - 3)
        end = min(len(lines), index + 5)

        print(f"\nLines {start + 1}-{end}:")

        for j in range(start, end):
            print(f"{j + 1:4d}: {lines[j]}")


print("\nRead-only ranking-provenance audit completed.")

=== B.6. GENE-ASSOCIATION SCORE PROVENANCE ===

=== CATALOG STRUCTURE ===
Top-level keys: ['provenance', 'summary', 'features']

Feature 0
Feature keys: ['feature_idx', 'is_dead', 'activation_freq', 'mean_activation', 'mean_activation_when_firing', 'fire_count', 'n_unique_genes', 'top_genes']
Top-gene records: 20
First three gene records:
{"gene_id": 204, "gene_name": "PPCS", "mean_activation": 22.554791495203972, "fire_count": 128, "supporting_conditions": 128}
{"gene_id": 5221, "gene_name": "WSB1", "mean_activation": 21.569206535816193, "fire_count": 128, "supporting_conditions": 128}
{"gene_id": 4141, "gene_name": "WSB2", "mean_activation": 19.33488093316555, "fire_count": 128, "supporting_conditions": 128}

Feature 189
Feature keys: ['feature_idx', 'is_dead', 'activation_freq', 'mean_activation', 'mean_activation_when_firing', 'fire_count', 'n_unique_genes', 'top_genes']
Top-gene records: 20
First three gene records:
{"gene_id": 459, "gene_name": "ASH1L", "mean_activation": 19.8859

### B.6.1. Verification of the Original Gene-Ranking Statistic

The feature catalogs provide per-gene mean activation,
firing count, and supporting-condition statistics.

Before condition-level bootstrap, the original
aggregation and ranking procedure must be identified.

In particular, the analysis verifies whether gene
scores are averaged over all conditions or only
over conditions in which the feature is active.

The original gene inclusion and Top-20 selection
rules must also be preserved.

This prevents the bootstrap experiment from
silently changing the original association statistic.

In [14]:
from pathlib import Path

print("=== GENE-RANKING IMPLEMENTATION AUDIT ===")

search_dirs = [
    ROOT / "src",
    ROOT / "scripts",
]

candidates = []

for directory in search_dirs:
    if not directory.exists():
        continue

    for path in directory.rglob("*.py"):
        source = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        if (
            "supporting_conditions" in source
            and "top_genes" in source
        ):
            candidates.append(path)

print("Candidate source files:")

for path in candidates:
    print(path.relative_to(ROOT))

# Show small, targeted excerpts only.
# Avoid printing entire source files.

for path in candidates[:3]:

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    print(
        f"\n=== {path.relative_to(ROOT)} ==="
    )

    anchors = [
        "supporting_conditions",
        "top_genes",
        "mean_activation",
        "argsort",
    ]

    selected = set()

    for anchor in anchors:

        hits = [
            i for i, line in enumerate(lines)
            if anchor in line
        ]

        for i in hits[:2]:
            selected.update(
                range(
                    max(0, i - 5),
                    min(len(lines), i + 7)
                )
            )

    ordered = sorted(selected)

    # Limit notebook output.
    for i in ordered[:90]:
        print(f"{i + 1:4d}: {lines[i]}")

print("\nRead-only ranking audit completed.")

=== GENE-RANKING IMPLEMENTATION AUDIT ===
Candidate source files:
src\sae_feature_catalog.py

=== src\sae_feature_catalog.py ===
  91:         if len(gene_idx):
  92:             conditional_mean = gene_sum[feature_idx, gene_idx] / gene_count[feature_idx, gene_idx]
  93:             # Stable deterministic ordering: descending conditional mean, then gene index.
  94:             order = np.lexsort((gene_idx, -conditional_mean))[:top_n]
  95:             ranked = gene_idx[order]
  96:             top_genes = [
  97:                 {
  98:                     "gene_id": int(gid),
  99:                     "gene_name": gene_symbols[int(gid)],
 100:                     "mean_activation": float(gene_sum[feature_idx, gid] / gene_count[feature_idx, gid]),
 101:                     "fire_count": int(gene_count[feature_idx, gid]),
 102:                     "supporting_conditions": int(gene_count[feature_idx, gid]),
 103:                 }
 104:                 for gid in ranked
 105:           

### B.6.2. Verification of Gene-Score Accumulation

The original feature catalog ranks genes by their
conditional mean SAE activation, calculated as the
ratio of accumulated activation to the corresponding
gene-specific count.

Before implementing condition-level bootstrap, the
accumulation rules for both quantities must be verified.

The audit also examines how active features, gene
identifiers, and experimental conditions contribute
to the catalog statistics.

The original scoring, eligibility, and tie-breaking
rules will be preserved in the bootstrap implementation.

In [15]:
import re

source_path = ROOT / "src" / "sae_feature_catalog.py"

lines = source_path.read_text(
    encoding="utf-8"
).splitlines()

patterns = [
    r"gene_sum",
    r"gene_count",
    r"gene_idx",
    r"gene_condition",
    r"supporting_conditions",
    r"activation.*>.*0",
    r"np\.lexsort",
]

matching_lines = [
    i
    for i, line in enumerate(lines)
    if any(
        re.search(pattern, line)
        for pattern in patterns
    )
]

# Show context around accumulation and ranking.
selected = set()

for i in matching_lines:
    selected.update(
        range(
            max(0, i - 2),
            min(len(lines), i + 3)
        )
    )

print("=== GENE SCORE ACCUMULATION AUDIT ===")
print("Source:", source_path.relative_to(ROOT))
print("Matching source lines:", len(matching_lines))

previous = -2

for i in sorted(selected):
    if i > previous + 1:
        print("    ...")

    print(f"{i + 1:4d}: {lines[i]}")
    previous = i

print("\nRead-only accumulation audit completed.")

=== GENE SCORE ACCUMULATION AUDIT ===
Source: src\sae_feature_catalog.py
Matching source lines: 23
    ...
  43:     topk_features: np.ndarray,
  44:     topk_values: np.ndarray,
  45:     gene_sum: np.ndarray,
  46:     gene_count: np.ndarray,
  47:     feature_sum: np.ndarray,
  48:     feature_count: np.ndarray,
    ...
  62:         return 0
  63:     if (features.min() < 0 or features.max() >= len(feature_count) or
  64:             genes.min() < 0 or genes.max() >= gene_count.shape[1]):
  65:         raise ValueError("Unexpected gene or feature index")
  66: 
    ...
  69:     # Each decoder gene occurs once per condition. Thus, for a gene-feature pair,
  70:     # a firing count is also the number of experimental conditions supporting it.
  71:     np.add.at(gene_sum, (features, genes), values)
  72:     np.add.at(gene_count, (features, genes), 1)
  73:     return len(features)
  74: 
  75: 
  76: def _create_catalog(
  77:     gene_sum: np.ndarray,
  78:     gene_count: np.ndar

### B.7. Reconstruction of Condition-Level Gene Associations

The original gene-ranking implementation accumulates
SAE activation sums and firing counts for each
feature–gene pair.

Genes are ranked by the conditional mean activation
over recorded firing events, with gene identifier
used as a deterministic tie-breaker.

To support condition-level bootstrap, the validation
activations of three selected decoder.0 SAE features
are reconstructed across all 6,642 output genes
and 32 validation conditions.

The selected features are 0, 189 and 2120.

The original Top-20 rankings are reconstructed
before any resampling. Agreement with the saved
validation catalogs serves as an implementation check.

This experiment uses the existing pretrained SAE.
Only decoder.0 validation representations are read
in batches. No training or file modification occurs.

In [16]:
import json
import h5py
import numpy as np
import torch

print("=== B.7. GENE-ASSOCIATION RECONSTRUCTION ===")

B7_FEATURES = [0, 189, 2120]
B7_BATCH_SIZE = 512

device_b7 = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

sae_b7 = sae.to(device_b7)
sae_b7.eval()

mean_b7 = torch.as_tensor(
    np.asarray(activation_mean, dtype=np.float32),
    device=device_b7
).reshape(1, -1)

print("Inference device:", device_b7)
print("Selected features:", B7_FEATURES)
print("Batch size:", B7_BATCH_SIZE)

# --------------------------------------------------
# 1. Load saved validation feature catalogs
# --------------------------------------------------

catalog_path_b7 = (
    CATALOG_DIR / "decoder_0_validation_catalog.json"
)

if not catalog_path_b7.exists():
    catalog_path_b7 = (
        CATALOG_DIR / "decoder_0_val_catalog.json"
    )

val_catalog_b7 = json.loads(
    catalog_path_b7.read_text(encoding="utf-8")
)

val_features_b7 = {
    int(f["feature_idx"]): f
    for f in val_catalog_b7["features"]
}

# --------------------------------------------------
# 2. Stream decoder.0 representations
# --------------------------------------------------

with h5py.File(ARTIFACTS["Validation HDF5"], "r") as h5:

    decoder_ds = h5["decoder"]["0"]
    gene_ds = h5["gene_ids"]

    n_conditions, n_positions, d_model = decoder_ds.shape

    assert n_conditions == 32
    assert n_positions == 6642
    assert d_model == 768
    assert len(B7_FEATURES) == 3

    recorded_conditions = np.asarray(
        h5["condition_indices"][:]
    )

    assert np.array_equal(
        recorded_conditions,
        val_condition_ids
    )

    # Each entry stores the selected SAE activation
    # for one condition, gene, and feature.
    B7_activations = np.zeros(
        (n_conditions, n_positions, len(B7_FEATURES)),
        dtype=np.float32
    )

    with torch.inference_mode():

        for c in range(n_conditions):

            gene_ids = np.asarray(
                gene_ds[c, :],
                dtype=np.int64
            )

            assert len(np.unique(gene_ids)) == n_positions
            assert gene_ids.min() >= 0
            assert gene_ids.max() < n_positions

            for start in range(
                0, n_positions, B7_BATCH_SIZE
            ):

                stop = min(
                    start + B7_BATCH_SIZE,
                    n_positions
                )

                hidden = np.asarray(
                    decoder_ds[c, start:stop, :],
                    dtype=np.float32
                )

                hidden_tensor = torch.from_numpy(
                    hidden
                ).to(device_b7)

                encoded, _ = sae_b7.encode(
                    hidden_tensor - mean_b7
                )

                selected = (
                    encoded[:, B7_FEATURES]
                    .detach()
                    .cpu()
                    .numpy()
                )

                if np.min(selected) < -1e-6:
                    raise ValueError(
                        "Unexpected negative SAE activations"
                    )

                B7_activations[
                    c,
                    gene_ids[start:stop],
                    :
                ] = np.maximum(selected, 0.0)

            if (c + 1) % 8 == 0:
                print(
                    f"Processed {c + 1}/{n_conditions} "
                    "validation conditions"
                )

# --------------------------------------------------
# 3. Reconstruct conditional-mean gene rankings
# --------------------------------------------------

B7_gene_sums = B7_activations.sum(
    axis=0,
    dtype=np.float64
)

B7_fire_counts = (
    B7_activations > 0
).sum(axis=0)

B7_reconstructed_top20 = {}

print("\n=== VALIDATION CATALOG REPRODUCTION ===")

all_exact = True

for j, fid in enumerate(B7_FEATURES):

    counts = B7_fire_counts[:, j]
    sums = B7_gene_sums[:, j]

    eligible = np.flatnonzero(counts > 0)

    conditional_means = (
        sums[eligible] / counts[eligible]
    )

    ordering = np.lexsort(
        (eligible, -conditional_means)
    )

    reconstructed = eligible[ordering[:20]].tolist()

    saved = [
        int(g["gene_id"])
        for g in val_features_b7[fid]["top_genes"][:20]
    ]

    B7_reconstructed_top20[fid] = reconstructed

    overlap = len(set(reconstructed) & set(saved))
    exact = reconstructed == saved

    all_exact = all_exact and exact

    print(f"\nFeature {fid}")
    print("Genes with recorded firings:", len(eligible))
    print("Saved Top-20 length:", len(saved))
    print("Reconstructed Top-20 length:", len(reconstructed))
    print("Top-20 overlap:", overlap)
    print("Exact ranking match:", exact)
    print("First five saved IDs:", saved[:5])
    print("First five reconstructed IDs:", reconstructed[:5])

print("\n=== B.7 VERIFICATION ===")
print("All three rankings reproduced:", all_exact)
print("Activation array shape:", B7_activations.shape)

if not all_exact:
    raise AssertionError(
        "Catalog reconstruction mismatch: "
        "verify firing-event selection and score aggregation "
        "before performing bootstrap."
    )

print("\nRead-only reconstruction completed.")

=== B.7. GENE-ASSOCIATION RECONSTRUCTION ===
Inference device: cuda:0
Selected features: [0, 189, 2120]
Batch size: 512
Processed 8/32 validation conditions
Processed 16/32 validation conditions
Processed 24/32 validation conditions
Processed 32/32 validation conditions

=== VALIDATION CATALOG REPRODUCTION ===

Feature 0
Genes with recorded firings: 171
Saved Top-20 length: 20
Reconstructed Top-20 length: 20
Top-20 overlap: 20
Exact ranking match: True
First five saved IDs: [204, 5221, 4141, 1114, 3684]
First five reconstructed IDs: [204, 5221, 4141, 1114, 3684]

Feature 189
Genes with recorded firings: 151
Saved Top-20 length: 20
Reconstructed Top-20 length: 20
Top-20 overlap: 20
Exact ranking match: True
First five saved IDs: [459, 2768, 2591, 4956, 1283]
First five reconstructed IDs: [459, 2768, 2591, 4956, 1283]

Feature 2120
Genes with recorded firings: 5995
Saved Top-20 length: 20
Reconstructed Top-20 length: 20
Top-20 overlap: 20
Exact ranking match: True
First five saved IDs: [

### B.8. Context-Stratified Bootstrap of Gene Rankings

This experiment evaluates the sensitivity of
SAE feature–gene associations to resampling
validation conditions.

The analysis uses the three decoder.0 features
whose original validation rankings were exactly
reconstructed in B.7: 0, 189, and 2120.

For each of 2,000 bootstrap replicates:

1. Resample experimental conditions with replacement
   within the three observed cell contexts.
2. Recalculate per-gene SAE activation sums and
   firing counts using the original scoring rule.
3. Rank all eligible genes by conditional mean
   activation, preserving gene-ID tie-breaking.
4. Compare the resulting Top-20 with the
   original validation Top-20.

Reported statistics include:

- Top-20 gene retention.
- Jaccard similarity.
- Top-5 overlap.
- Frequency of complete Top-20 availability.
- Per-gene bootstrap inclusion frequency.

All statistics are conditional on the fixed
trained SAE, observed validation conditions,
and the original feature-catalog construction.

Resampling the same conditions does not provide
independent biological replication and cannot
establish perturbation specificity or causality.

In [17]:
import numpy as np
import pandas as pd

print("=== B.8. CONDITION-LEVEL GENE-RANKING BOOTSTRAP ===")

B8_N_BOOT = 2000
B8_TOP_K = 20
B8_SEED = 20261113

rng_b8 = np.random.default_rng(B8_SEED)

C, G, F = B7_activations.shape

assert C == 32
assert G == 6642
assert F == len(B7_FEATURES) == 3

# Verify that cell-context groups partition conditions.
group_indices = list(groups_boot.values())

all_group_indices = np.concatenate(group_indices)

assert np.array_equal(
    np.sort(all_group_indices),
    np.arange(C)
)

# --------------------------------------------------
# 1. Prepare activation and firing matrices
# --------------------------------------------------

# Resampling weights are applied to complete
# condition vectors. No HDF5 access is required.

flat_activations = (
    B7_activations.reshape(C, -1)
    .astype(np.float64)
)

flat_firings = (
    (B7_activations > 0)
    .reshape(C, -1)
    .astype(np.float64)
)

reference_top20 = {
    fid: np.asarray(
        B7_reconstructed_top20[fid],
        dtype=np.int32
    )
    for fid in B7_FEATURES
}

assert all(
    len(reference_top20[fid]) == B8_TOP_K
    for fid in B7_FEATURES
)

# --------------------------------------------------
# 2. Initialize bootstrap result arrays
# --------------------------------------------------

shared_top20 = np.zeros(
    (B8_N_BOOT, F),
    dtype=np.int16
)

shared_top5 = np.zeros(
    (B8_N_BOOT, F),
    dtype=np.int16
)

jaccard_b8 = np.zeros(
    (B8_N_BOOT, F),
    dtype=np.float64
)

eligible_count = np.zeros(
    (B8_N_BOOT, F),
    dtype=np.int32
)

selected_count = np.zeros(
    (B8_N_BOOT, F),
    dtype=np.int16
)

inclusion_count = np.zeros(
    (G, F),
    dtype=np.int32
)

# --------------------------------------------------
# 3. Perform context-stratified bootstrap
# --------------------------------------------------

for b in range(B8_N_BOOT):

    weights = np.zeros(C, dtype=np.float64)

    for indices in group_indices:

        
        sampled = rng_b8.choice(
            indices,
            size=len(indices),
            replace=True
        )

        weights += np.bincount(
            sampled,
            minlength=C
        )

    assert int(weights.sum()) == C

    # Weighted aggregation is equivalent to
    # physically duplicating sampled conditions.

    gene_sums = (
        weights @ flat_activations
    ).reshape(G, F)

    gene_counts = (
        weights @ flat_firings
    ).reshape(G, F)

    for j, fid in enumerate(B7_FEATURES):

        eligible = np.flatnonzero(
            gene_counts[:, j] > 0
        )

        eligible_count[b, j] = len(eligible)

        conditional_means = (
            gene_sums[eligible, j]
            / gene_counts[eligible, j]
        )

        order = np.lexsort(
            (eligible, -conditional_means)
        )

        selected = eligible[
            order[:B8_TOP_K]
        ]

        selected_count[b, j] = len(selected)

        inclusion_count[selected, j] += 1

        original = reference_top20[fid]

        selected_set = set(selected.tolist())
        original_set = set(original.tolist())

        shared = len(
            selected_set & original_set
        )

        union = len(
            selected_set | original_set
        )

        shared_top20[b, j] = shared

        jaccard_b8[b, j] = (
            shared / union
            if union > 0
            else np.nan
        )

        shared_top5[b, j] = len(
            set(selected[:5].tolist())
            & set(original[:5].tolist())
        )

# --------------------------------------------------
# 4. Summarize bootstrap distributions
# --------------------------------------------------

summary_rows_b8 = []

for j, fid in enumerate(B7_FEATURES):

    overlap = shared_top20[:, j]
    jaccard = jaccard_b8[:, j]

    overlap_interval = np.percentile(
        overlap,
        [2.5, 97.5]
    )

    summary_rows_b8.append({
        "feature": fid,
        "mean_shared_top20": overlap.mean(),
        "median_shared_top20": np.median(overlap),
        "overlap_q025": overlap_interval[0],
        "overlap_q975": overlap_interval[1],
        "mean_jaccard": jaccard.mean(),
        "mean_shared_top5": shared_top5[:, j].mean(),
        "complete_top20_fraction": np.mean(
            selected_count[:, j] == 20
        ),
        "mean_eligible_genes": eligible_count[:, j].mean(),
    })

summary_b8 = pd.DataFrame(summary_rows_b8)

print("\n=== BOOTSTRAP SUMMARY ===")
print("Bootstrap repetitions:", B8_N_BOOT)

print(
    summary_b8.round(4).to_string(index=False)
)

# --------------------------------------------------
# 5. Original Top-20 inclusion frequencies
# --------------------------------------------------

print("\n=== ORIGINAL TOP-20 GENE STABILITY ===")

for j, fid in enumerate(B7_FEATURES):

    original = reference_top20[fid]

    frequencies = (
        inclusion_count[original, j]
        / B8_N_BOOT
    )

    ranking = np.argsort(frequencies)

    print(f"\nFeature {fid}")

    print(
        "Mean original Top-20 retention:",
        f"{frequencies.mean():.4f}"
    )

    print(
        "Original genes retained in >=95% "
        "of bootstrap samples:",
        int(np.sum(frequencies >= 0.95)),
        "/20"
    )

    print("Five least stable original genes:")

    for idx in ranking[:5]:

        print(
            f"  gene_id={original[idx]}: "
            f"inclusion={frequencies[idx]:.4f}"
        )

    # Identify genes entering from outside
    # the original validation Top-20.

    new_gene_frequencies = (
        inclusion_count[:, j].astype(np.float64)
        / B8_N_BOOT
    )

    new_gene_frequencies[original] = 0.0

    new_candidates = np.argsort(
        -new_gene_frequencies
    )[:5]

    print("Most frequent new entrants:")

    for gene_id in new_candidates:

        if new_gene_frequencies[gene_id] > 0:

            print(
                f"  gene_id={gene_id}: "
                f"inclusion="
                f"{new_gene_frequencies[gene_id]:.4f}"
            )

# --------------------------------------------------
# 6. Final integrity checks
# --------------------------------------------------

assert np.isfinite(jaccard_b8).all()

assert np.all(
    (shared_top20 >= 0)
    & (shared_top20 <= B8_TOP_K)
)

assert np.all(
    selected_count <= B8_TOP_K
)

assert np.all(
    inclusion_count.sum(axis=0)
    == selected_count.sum(axis=0)
)

print("\nBootstrap gene-ranking analysis completed.")
print("Original data and checkpoints were not modified.")

=== B.8. CONDITION-LEVEL GENE-RANKING BOOTSTRAP ===

=== BOOTSTRAP SUMMARY ===
Bootstrap repetitions: 2000
 feature  mean_shared_top20  median_shared_top20  overlap_q025  overlap_q975  mean_jaccard  mean_shared_top5  complete_top20_fraction  mean_eligible_genes
       0            19.8680                 20.0          19.0          20.0        0.9874            5.0000                      1.0              135.231
     189            19.5440                 20.0          19.0          20.0        0.9566            5.0000                      1.0              132.684
    2120             5.3375                  5.0           1.0          13.0        0.1658            1.0735                      1.0             5495.881

=== ORIGINAL TOP-20 GENE STABILITY ===

Feature 0
Mean original Top-20 retention: 0.9934
Original genes retained in >=95% of bootstrap samples: 19 /20
Five least stable original genes:
  gene_id=2615: inclusion=0.8680
  gene_id=204: inclusion=1.0000
  gene_id=4141: inclus

### B.8.1. Bootstrap Gene-Ranking Results

A context-stratified bootstrap with 2,000 replicates
was performed for three decoder.0 SAE features.

All gene-association scores were recomputed using
the original conditional-mean activation statistic.

| Feature | Mean Top-20 overlap | Mean Jaccard | Overlap 2.5–97.5% |
|---|---:|---:|---|
| 0 | 19.868 | 0.9874 | 19–20 |
| 189 | 19.544 | 0.9566 | 19–20 |
| 2120 | 5.338 | 0.1658 | 1–13 |

Features 0 and 189 exhibited high ranking stability
under resampling of validation conditions.

Feature 2120 exhibited substantially lower stability,
with an average of approximately five original
Top-20 genes retained per bootstrap replicate.

The mean numbers of genes with recorded firing
events in the bootstrap samples were approximately
135, 133, and 5,496 for features 0, 189, and 2120.

This substantial difference motivates an additional
investigation of firing support and ranking reliability.

The overlap quantiles describe the bootstrap
distribution of a descriptive stability statistic.
They are not confidence intervals for the number
of biologically functional genes.

These results are conditional on the fixed SAE,
observed validation conditions, and the chosen
cell-context-stratified resampling procedure.

### B.9. Firing Support and Gene-Ranking Reliability

The original feature catalog ranks genes by their
mean SAE activation conditional on firing.

This statistic does not directly penalize genes
supported by only a small number of conditions.

The present experiment examines whether the
instability of feature 2120 is associated with
low firing support among highly ranked genes.

For features 0, 189, and 2120, we evaluate:

1. The distribution of firing counts among the
   original validation Top-20 genes.
2. The stability of Top-20 rankings under increasing
   minimum firing-support requirements.
3. The agreement between conditional-mean and
   unconditional-mean activation rankings.

Support thresholds are sensitivity analyses.
They do not replace the original catalog method.

The experiment uses reconstructed activation
summaries from B.7 and does not repeat SAE inference.

In [18]:
import numpy as np
import pandas as pd

print("=== B.9. FIRING-SUPPORT AND RANKING AUDIT ===")

C, G, F = B7_activations.shape

assert C == 32
assert G == 6642
assert F == len(B7_FEATURES) == 3

assert B7_gene_sums.shape == (G, F)
assert B7_fire_counts.shape == (G, F)

support_thresholds = [1, 2, 4, 8, 16, 24, 32]

support_records = []
threshold_records = []
alternative_records = []

# --------------------------------------------------
# 1. Inspect firing support of original Top-20 genes
# --------------------------------------------------

for j, fid in enumerate(B7_FEATURES):

    reference = np.asarray(
        B7_reconstructed_top20[fid],
        dtype=np.int64
    )

    assert len(reference) == 20

    counts = B7_fire_counts[:, j]
    sums = B7_gene_sums[:, j]

    original_support = counts[reference]

    support_records.append({
        "feature": fid,
        "active_genes": int((counts > 0).sum()),
        "top20_min_support": int(original_support.min()),
        "top20_median_support": float(
            np.median(original_support)
        ),
        "top20_max_support": int(original_support.max()),
        "top20_support_le_2": int(
            (original_support <= 2).sum()
        ),
        "top20_support_ge_8": int(
            (original_support >= 8).sum()
        ),
        "top20_support_ge_16": int(
            (original_support >= 16).sum()
        ),
    })

    # ----------------------------------------------
    # 2. Minimum-support sensitivity analysis
    # ----------------------------------------------

    for min_support in support_thresholds:

        eligible = np.flatnonzero(
            counts >= min_support
        )

        scores = (
            sums[eligible] / counts[eligible]
        )

        order = np.lexsort(
            (eligible, -scores)
        )

        selected = eligible[order[:20]]

        shared = len(
            set(reference.tolist())
            & set(selected.tolist())
        )

        union = len(
            set(reference.tolist())
            | set(selected.tolist())
        )

        threshold_records.append({
            "feature": fid,
            "min_support": min_support,
            "eligible_genes": len(eligible),
            "selected_genes": len(selected),
            "shared_original_top20": shared,
            "jaccard": (
                shared / union if union else np.nan
            ),
        })

    # ----------------------------------------------
    # 3. Alternative unconditional-mean ranking
    # ----------------------------------------------

    # This is a sensitivity control, not the
    # original gene-association statistic.

    eligible = np.flatnonzero(counts > 0)

    unconditional_scores = sums[eligible] / C

    order = np.lexsort(
        (eligible, -unconditional_scores)
    )

    unconditional_top20 = eligible[order[:20]]

    shared = len(
        set(reference.tolist())
        & set(unconditional_top20.tolist())
    )

    alternative_records.append({
        "feature": fid,
        "shared_with_unconditional_top20": shared,
        "jaccard": shared / (40 - shared),
    })

# --------------------------------------------------
# 4. Report
# --------------------------------------------------

support_df_b9 = pd.DataFrame(support_records)
threshold_df_b9 = pd.DataFrame(threshold_records)
alternative_df_b9 = pd.DataFrame(alternative_records)

print("\n=== ORIGINAL TOP-20 FIRING SUPPORT ===")

print(
    support_df_b9.to_string(index=False)
)

print("\n=== MINIMUM-SUPPORT SENSITIVITY ===")

print(
    threshold_df_b9.round(4).to_string(index=False)
)

print("\n=== ALTERNATIVE RANKING COMPARISON ===")

print(
    alternative_df_b9.round(4).to_string(index=False)
)

# --------------------------------------------------
# 5. Integrity checks
# --------------------------------------------------

assert (
    threshold_df_b9.loc[
        threshold_df_b9["min_support"] == 1,
        "shared_original_top20"
    ] == 20
).all()

assert (
    threshold_df_b9["selected_genes"] <= 20
).all()

assert (
    support_df_b9["top20_max_support"] <= C
).all()

print("\nRead-only firing-support audit completed.")

=== B.9. FIRING-SUPPORT AND RANKING AUDIT ===

=== ORIGINAL TOP-20 FIRING SUPPORT ===
 feature  active_genes  top20_min_support  top20_median_support  top20_max_support  top20_support_le_2  top20_support_ge_8  top20_support_ge_16
       0           171                 32                  32.0                 32                   0                  20                   20
     189           151                 32                  32.0                 32                   0                  20                   20
    2120          5995                  1                   1.0                  6                  14                   0                    0

=== MINIMUM-SUPPORT SENSITIVITY ===
 feature  min_support  eligible_genes  selected_genes  shared_original_top20  jaccard
       0            1             171              20                     20   1.0000
       0            2              76              20                     20   1.0000
       0            4              44      

### B.9.1. Firing-Support Sensitivity Results

The firing-support audit revealed substantial
differences between the three selected SAE features.

Features 0 and 189 had complete firing support
for every gene in their validation Top-20 lists:
all twenty genes were active in all 32 conditions.

Feature 2120 showed a markedly different pattern.

Its Top-20 gene-support counts ranged from 1 to 6,
with a median of 1. Fourteen of its twenty
top-ranked genes were supported by no more than
two validation conditions.

The original conditional-mean ranking can assign
high scores to rare, high-magnitude activations.

This provides a plausible statistical explanation
for the low bootstrap stability and cross-split
agreement previously observed for feature 2120.

The ranking based on unconditional mean activation
also differed substantially from its original
conditional-mean ranking.

These findings motivate explicit firing-support
criteria and ranking-sensitivity checks in the
subsequent full-dictionary analysis.

The original catalog scores are retained unchanged.
Alternative rankings are treated as sensitivity
analyses rather than replacement ground truth.

### B.10. Conclusions of Activation Robustness Analysis

The activation robustness analyses established
three main findings.

First, the common-condition variation statistics
of SAE feature 0 were reproduced and characterized
using context-stratified bootstrap resampling.

Second, cross-split gene-association catalogs showed
high overall consistency across four decoder SAE
dictionaries. Correct feature pairing substantially
outperformed random feature reassignment.

Third, gene-ranking reliability varied considerably
between individual SAE features.

Features 0 and 189 exhibited high bootstrap stability
and complete firing support among their validation
Top-20 genes.

Feature 2120 exhibited low ranking stability,
low firing support, and strong sensitivity to
the choice of gene-association statistic.

High cross-split stability does not, by itself,
establish perturbation sensitivity, SAE specificity
relative to suitable representation baselines,
or biological causality.

The next stage extends functional enrichment
validation to a predefined family of decoder
SAE features and incorporates the identified
sources of statistical uncertainty.

## C. Full-Dictionary Functional Enrichment Validation

### C.1. Hypothesis Family and Gene-Support Audit

The primary analysis considers all 12,288 features
from the four decoder SAE dictionaries.

Each feature is represented by its training-derived
Top-20 gene associations.

The functional reference family contains the
4,420 reconstructed GO Biological Process, KEGG,
and Reactome terms.

The primary analysis will evaluate one-sided
hypergeometric enrichment and apply multiple-testing
correction across the explicitly defined family
of feature–term hypotheses.

Gene-ranking reliability will be examined separately
using firing-support statistics.

The original conditional-mean rankings will not be
silently replaced by support-filtered rankings.

This is a retrospective exploratory analysis,
not an independently preregistered confirmatory study.

### Statistical scope

- Gene universe: 6,642 PIE output genes.
- Features: 4 × 3,072 decoder SAE features.
- Gene sets per feature: training Top-20.
- Reference terms: 4,420 GO BP / KEGG / Reactome sets.
- Primary family: 54,312,960 feature–term hypotheses.
- TRRUST: excluded from this restricted analysis.

The original annotation results remain available
for comparison, but are not treated as an exact
reproduction of the reconstructed hypothesis family.

In [19]:
import json
import numpy as np
import pandas as pd
from collections import Counter

print("=== C.1. FULL-DICTIONARY ANALYSIS SPECIFICATION ===")

N_GENES = 6642
N_FEATURES_PER_LAYER = 3072
N_TERMS = 4420
N_TRAIN_CONDITIONS = 128

assert len(universe) == N_GENES
assert len(restricted_gene_sets) == N_TERMS

term_source_counts = Counter(
    key[0] for key in restricted_gene_sets
)

print("\n=== REFERENCE FAMILY ===")
print("Gene universe:", N_GENES)
print("Functional terms:", N_TERMS)
print("Sources:", dict(term_source_counts))

print("\n=== FEATURE FAMILY ===")
print("Decoder dictionaries:", 4)
print("Features per dictionary:", N_FEATURES_PER_LAYER)
print("Total features:", 4 * N_FEATURES_PER_LAYER)

print(
    "Total feature-term hypotheses:",
    4 * N_FEATURES_PER_LAYER * N_TERMS
)

# --------------------------------------------------
# 1. Inspect training Top-20 support
# --------------------------------------------------

support_records_c1 = []

for layer_num in range(4):

    path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_train_catalog.json"
    )

    catalog = json.loads(
        path.read_text(encoding="utf-8")
    )

    assert len(catalog["features"]) == 3072

    for feature in catalog["features"]:

        fid = int(feature["feature_idx"])

        top = feature["top_genes"][:20]

        if len(top) != 20:
            raise ValueError(
                f"Incomplete training Top-20: "
                f"decoder.{layer_num}/{fid}"
            )

        gene_ids = [
            int(g["gene_id"]) for g in top
        ]

        assert len(set(gene_ids)) == 20

        support = np.array([
            int(g["supporting_conditions"])
            for g in top
        ])

        if np.any(support < 1):
            raise ValueError(
                f"Invalid support for feature {fid}"
            )

        if np.any(support > N_TRAIN_CONDITIONS):
            raise ValueError(
                f"Support exceeds training conditions: {fid}"
            )

        support_records_c1.append({
            "layer": f"decoder.{layer_num}",
            "feature": fid,
            "min_support": int(support.min()),
            "median_support": float(np.median(support)),
            "median_support_fraction": float(
                np.median(support) / N_TRAIN_CONDITIONS
            ),
            "genes_with_support_le_2": int(
                (support <= 2).sum()
            ),
            "genes_with_support_lt_25pct": int(
                (support < 0.25 * N_TRAIN_CONDITIONS).sum()
            ),
            "all_20_support_ge_25pct": bool(
                np.all(support >= 0.25 * N_TRAIN_CONDITIONS)
            ),
        })

support_df_c1 = pd.DataFrame(support_records_c1)

assert len(support_df_c1) == 12288

# --------------------------------------------------
# 2. Summarize support by decoder dictionary
# --------------------------------------------------

support_summary_c1 = support_df_c1.groupby("layer").agg(
    features=("feature", "count"),
    median_feature_support=(
        "median_support_fraction", "median"
    ),
    features_with_rare_top_genes=(
        "genes_with_support_le_2",
        lambda x: int((x > 0).sum())
    ),
    features_all_top20_ge_25pct=(
        "all_20_support_ge_25pct", "sum"
    ),
)

print("\n=== TRAINING GENE-SUPPORT AUDIT ===")
print(support_summary_c1.round(4).to_string())

# --------------------------------------------------
# 3. Examine original pilot candidates
# --------------------------------------------------

pilot_ids = [0, 189, 2120]

pilot_support_c1 = support_df_c1.loc[
    support_df_c1["layer"].eq("decoder.0")
    & support_df_c1["feature"].isin(pilot_ids)
].sort_values("feature")

assert len(pilot_support_c1) == 3

print("\n=== SELECTED FEATURES: TRAINING SUPPORT ===")

print(
    pilot_support_c1.to_string(index=False)
)

print("\nRead-only support audit completed.")

=== C.1. FULL-DICTIONARY ANALYSIS SPECIFICATION ===

=== REFERENCE FAMILY ===
Gene universe: 6642
Functional terms: 4420
Sources: {'GO_BP': 2838, 'KEGG': 286, 'Reactome': 1296}

=== FEATURE FAMILY ===
Decoder dictionaries: 4
Features per dictionary: 3072
Total features: 12288
Total feature-term hypotheses: 54312960

=== TRAINING GENE-SUPPORT AUDIT ===
           features  median_feature_support  features_with_rare_top_genes  features_all_top20_ge_25pct
layer                                                                                                 
decoder.0      3072                     1.0                            24                         3029
decoder.1      3072                     1.0                            46                         2984
decoder.2      3072                     1.0                           117                         2933
decoder.3      3072                     1.0                           183                         2867

=== SELECTED FEATURES: TRAI

### C.2. Sparse Feature–Term Overlap Matrix

The full-dictionary analysis is implemented using
sparse binary gene-membership matrices.

The feature matrix contains 12,288 rows, corresponding
to the 3,072 SAE features in each of the four decoder
dictionaries. Each row represents the 20
training-associated genes of one feature.

The reference matrix contains 4,420 functional
terms constructed from the current GO BP, KEGG,
and Reactome reference sets.

Both matrices use the same 6,642-gene PIE universe.

Sparse matrix multiplication computes the number
of shared genes for every feature–term pair.

The resulting overlaps are checked against
previously examined enrichment examples and the
original saved annotation summaries.

This stage computes overlap counts only.
Statistical testing and multiple-testing correction
are performed separately in the following stage.

All existing catalogs and annotation files remain
unchanged.

In [20]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

print("=== C.2. SPARSE FEATURE-TERM OVERLAP MATRIX ===")

# --------------------------------------------------
# 1. Construct a shared gene index
# --------------------------------------------------

C2_genes = sorted(str(g) for g in universe)

assert len(C2_genes) == 6642
assert len(set(C2_genes)) == 6642

C2_gene_index = {
    gene: idx
    for idx, gene in enumerate(C2_genes)
}

# --------------------------------------------------
# 2. Construct the ontology-term matrix
# --------------------------------------------------

C2_terms = sorted(
    restricted_gene_sets.keys(),
    key=lambda x: (str(x[0]), str(x[1]))
)

assert len(C2_terms) == 4420

term_rows = []
term_cols = []

for t, key in enumerate(C2_terms):

    genes = set(restricted_gene_sets[key])

    assert 5 <= len(genes) <= 500

    missing = genes - set(C2_genes)

    if missing:
        raise ValueError(
            f"Ontology term contains unknown genes: "
            f"{key}, examples={sorted(missing)[:5]}"
        )

    for gene in genes:
        term_rows.append(C2_gene_index[gene])
        term_cols.append(t)

C2_term_matrix = csr_matrix(
    (
        np.ones(len(term_rows), dtype=np.int16),
        (term_rows, term_cols)
    ),
    shape=(6642, 4420),
    dtype=np.int16
)

C2_term_sizes = np.asarray(
    C2_term_matrix.sum(axis=0)
).ravel()

assert C2_term_sizes.min() >= 5
assert C2_term_sizes.max() <= 500

print("Term matrix:", C2_term_matrix.shape)
print("Term memberships:", C2_term_matrix.nnz)

# --------------------------------------------------
# 3. Construct the decoder feature matrix
# --------------------------------------------------

feature_rows = []
feature_cols = []
feature_metadata = []

for layer_num in range(4):

    path = (
        CATALOG_DIR
        / f"decoder_{layer_num}_train_catalog.json"
    )

    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    by_id = {
        int(f["feature_idx"]): f
        for f in payload["features"]
    }

    assert set(by_id) == set(range(3072))

    for fid in range(3072):

        feature = by_id[fid]

        top_genes = feature["top_genes"][:20]

        names = [
            str(g["gene_name"])
            for g in top_genes
        ]

        if len(names) != 20 or len(set(names)) != 20:
            raise ValueError(
                f"Invalid Top-20 list: decoder.{layer_num}/{fid}"
            )

        missing = set(names) - C2_gene_index.keys()

        if missing:
            raise ValueError(
                f"Unknown gene symbols: decoder.{layer_num}/"
                f"{fid}: {sorted(missing)[:5]}"
            )

        row = layer_num * 3072 + fid

        for gene in names:
            feature_rows.append(row)
            feature_cols.append(C2_gene_index[gene])

        feature_metadata.append({
            "layer": f"decoder.{layer_num}",
            "feature": fid,
            "matrix_row": row,
        })

C2_feature_matrix = csr_matrix(
    (
        np.ones(len(feature_rows), dtype=np.int16),
        (feature_rows, feature_cols)
    ),
    shape=(12288, 6642),
    dtype=np.int16
)

C2_feature_metadata = pd.DataFrame(feature_metadata)

assert np.all(
    np.asarray(
        C2_feature_matrix.sum(axis=1)
    ).ravel() == 20
)

print("\nFeature matrix:", C2_feature_matrix.shape)
print("Feature memberships:", C2_feature_matrix.nnz)

# --------------------------------------------------
# 4. Compute all feature-term overlaps
# --------------------------------------------------

C2_overlap = (
    C2_feature_matrix @ C2_term_matrix
).tocsr()

assert C2_overlap.shape == (12288, 4420)
assert np.all(C2_overlap.data >= 1)
assert np.all(C2_overlap.data <= 20)

print("\nOverlap matrix:", C2_overlap.shape)

print(
    "Total feature-term hypotheses:",
    C2_overlap.shape[0] * C2_overlap.shape[1]
)

print(
    "Pairs with overlap >= 1:",
    C2_overlap.nnz
)

print(
    "Pairs with overlap >= 2:",
    int(np.count_nonzero(C2_overlap.data >= 2))
)

# --------------------------------------------------
# 5. Layer-level overlap audit
# --------------------------------------------------

layer_rows = []

for layer_num in range(4):

    start = layer_num * 3072
    stop = start + 3072

    layer_overlap = C2_overlap[start:stop]

    explicit_tests = int(
        np.count_nonzero(layer_overlap.data >= 2)
    )

    annotation_path = (
        ROOT
        / "results/sae_feature_analysis_independent"
        / "biological_annotations"
        / f"decoder_{layer_num}_annotations.json"
    )

    annotation = json.loads(
        annotation_path.read_text(encoding="utf-8")
    )

    historical_tests = int(
        annotation["summary"][
            "n_explicit_gene_set_tests_with_overlap_ge_2"
        ]
    )

    layer_rows.append({
        "layer": f"decoder.{layer_num}",
        "overlap_ge_1": layer_overlap.nnz,
        "overlap_ge_2": explicit_tests,
        "original_reported_ge_2": historical_tests,
        "difference": explicit_tests - historical_tests,
    })

C2_overlap_summary = pd.DataFrame(layer_rows)

print("\n=== LAYER-LEVEL OVERLAP AUDIT ===")

print(
    C2_overlap_summary.to_string(index=False)
)

# --------------------------------------------------
# 6. Verify previously examined examples
# --------------------------------------------------

print("\n=== KNOWN FEATURE-TERM EXAMPLES ===")

examples = [
    (0, "GO:0072522"),
    (189, "R-HSA-3214841"),
]

for fid, term_identifier in examples:

    matches = [
        idx
        for idx, key in enumerate(C2_terms)
        if term_identifier in str(key[1])
    ]

    if len(matches) == 1:

        t = matches[0]

        overlap = int(C2_overlap[fid, t])

        print(
            f"decoder.0 feature {fid}, "
            f"{term_identifier}: "
            f"overlap={overlap}"
        )

    else:
        print(
            f"{term_identifier}: "
            f"{len(matches)} matching term identifiers"
        )

print("\nRead-only sparse overlap construction completed.")

=== C.2. SPARSE FEATURE-TERM OVERLAP MATRIX ===
Term matrix: (6642, 4420)
Term memberships: 122232

Feature matrix: (12288, 6642)
Feature memberships: 245760

Overlap matrix: (12288, 4420)
Total feature-term hypotheses: 54312960
Pairs with overlap >= 1: 3316033
Pairs with overlap >= 2: 609033

=== LAYER-LEVEL OVERLAP AUDIT ===
    layer  overlap_ge_1  overlap_ge_2  original_reported_ge_2  difference
decoder.0        827814        154530                  155479        -949
decoder.1        828012        150889                  151806        -917
decoder.2        825910        151854                  152821        -967
decoder.3        834297        151760                  152706        -946

=== KNOWN FEATURE-TERM EXAMPLES ===
decoder.0 feature 0, GO:0072522: overlap=4
decoder.0 feature 189, R-HSA-3214841: overlap=6

Read-only sparse overlap construction completed.


### C.2.1. TRRUST Contribution to Historical Test Counts

The restricted ontology analysis includes 4,420
GO BP, KEGG and Reactome terms.

The original annotation implementation additionally
includes a curated TRRUST transcription-factor gene set.

This audit calculates the overlap of that gene set
with every decoder SAE feature and compares the
number of overlaps of at least two genes with the
difference between reconstructed and historical
explicit-test counts.

The calculation distinguishes two questions:

1. Whether the TRRUST term explains the difference
   in explicit overlap-based tests.
2. Whether the originally reported total of 4,422
   reference terms can be reproduced.

Resolving the first question does not necessarily
resolve the second.

All analyses are read-only.

In [21]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

print("=== C.2.1. TRRUST OVERLAP RECONCILIATION ===")

# --------------------------------------------------
# 1. Recover the original TRRUST TF gene set
# --------------------------------------------------

_, _, trrust_tfs_c2, _ = annotation_module._load_networks(
    REFERENCE_DIR,
    universe
)

tf_genes = {str(g) for g in trrust_tfs_c2}

unknown = tf_genes - set(C2_gene_index)

print("TRRUST TF genes:", len(tf_genes))
print("Genes outside PIE universe:", len(unknown))

if unknown:
    raise ValueError(
        f"Unexpected TRRUST gene identifiers: "
        f"{sorted(unknown)[:10]}"
    )

# --------------------------------------------------
# 2. Construct a single sparse TRRUST column
# --------------------------------------------------

tf_indices = sorted(
    C2_gene_index[g] for g in tf_genes
)

tf_column = csr_matrix(
    (
        np.ones(len(tf_indices), dtype=np.int16),
        (
            np.asarray(tf_indices, dtype=np.int32),
            np.zeros(len(tf_indices), dtype=np.int32),
        ),
    ),
    shape=(6642, 1),
    dtype=np.int16,
)

trrust_overlap = (
    C2_feature_matrix @ tf_column
).toarray().ravel()

assert len(trrust_overlap) == 12288
assert trrust_overlap.min() >= 0
assert trrust_overlap.max() <= 20

# --------------------------------------------------
# 3. Compare with historical explicit-test counts
# --------------------------------------------------

comparison_rows = []

for layer_num in range(4):

    label = f"decoder.{layer_num}"

    start = layer_num * 3072
    stop = start + 3072

    tf_values = trrust_overlap[start:stop]

    tf_ge2 = int(np.count_nonzero(tf_values >= 2))

    row = C2_overlap_summary.loc[
        C2_overlap_summary["layer"] == label
    ].iloc[0]

    restricted_count = int(row["overlap_ge_2"])

    historical_count = int(
        row["original_reported_ge_2"]
    )

    historical_difference = (
        historical_count - restricted_count
    )

    residual = historical_difference - tf_ge2

    comparison_rows.append({
        "layer": label,
        "restricted_ge2": restricted_count,
        "trrust_ge2": tf_ge2,
        "reconstructed_ge2": restricted_count + tf_ge2,
        "historical_ge2": historical_count,
        "residual": residual,
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\n=== EXPLICIT-TEST COUNT COMPARISON ===")

print(comparison_df.to_string(index=False))

all_matched = bool(
    (comparison_df["residual"] == 0).all()
)

print("\n=== INTERPRETATION ===")

print(
    "TRRUST fully explains explicit-test differences:",
    all_matched
)

print(
    "Current reconstructed reference terms:",
    len(C2_terms) + 1
)

print(
    "Historically reported reference terms:",
    4422
)

print(
    "Unresolved reference-term count difference:",
    4422 - (len(C2_terms) + 1)
)

print("\nRead-only TRRUST reconciliation completed.")

=== C.2.1. TRRUST OVERLAP RECONCILIATION ===
TRRUST TF genes: 346
Genes outside PIE universe: 0

=== EXPLICIT-TEST COUNT COMPARISON ===
    layer  restricted_ge2  trrust_ge2  reconstructed_ge2  historical_ge2  residual
decoder.0          154530         851             155381          155479        98
decoder.1          150889         854             151743          151806        63
decoder.2          151854         867             152721          152821       100
decoder.3          151760         867             152627          152706        79

=== INTERPRETATION ===
TRRUST fully explains explicit-test differences: False
Current reconstructed reference terms: 4421
Historically reported reference terms: 4422
Unresolved reference-term count difference: 1

Read-only TRRUST reconciliation completed.


### C.2.2. Historical Overlap-Count Reconciliation

The reconstructed ontology family contains 4,420
GO BP, KEGG and Reactome terms.

The current original annotation implementation
additionally constructs one curated TRRUST
transcription-factor term, giving 4,421 terms.

The historical annotation summaries report 4,422
reference terms per decoder dictionary.

The TRRUST term accounts for most, but not all,
of the difference in explicit tests with gene
overlap of at least two:

| Layer | Explained by TRRUST | Remaining difference |
|---|---:|---:|
| decoder.0 | 851 | 98 |
| decoder.1 | 854 | 63 |
| decoder.2 | 867 | 100 |
| decoder.3 | 867 | 79 |

A total of 337 historical explicit tests remain
unexplained after accounting for TRRUST.

The discrepancy may reflect historical differences
in the reference-term family, but its exact cause
has not been established.

All subsequent enrichment calculations therefore
use the explicitly reconstructed 4,420-term
GO BP / KEGG / Reactome family.

The original annotation outputs remain available
for historical comparison, without claiming exact
reproduction of their multiple-testing family.

### C.3. Full-Dictionary Enrichment and BH Correction

One-sided hypergeometric enrichment is evaluated for
all decoder SAE feature–term pairs.

The analysis uses:

- 12,288 decoder SAE features.
- 4,420 reconstructed functional terms.
- 6,642 PIE output genes.
- 20 training-associated genes per feature.

Pairs with gene overlap below two receive p = 1,
following the convention of the original annotation
implementation.

For computational efficiency, explicit p-values are
computed only for pairs with overlap at least two.

Benjamini–Hochberg adjustment nevertheless accounts
for the full family of 54,312,960 hypotheses.

The primary analysis uses a single global BH family
across all four decoder dictionaries.

This differs from the original per-layer adjustment
and is not presented as an exact reproduction of
the historical annotation results.

BH-adjusted results remain subject to assumptions
about dependence between tests. Substantial dependence
among biological terms and SAE features is expected.

The analysis is retrospective and exploratory.
Functional enrichment does not establish
computational or biological causality.

No original annotation files or SAE checkpoints
are modified.

In [22]:
import numpy as np
import pandas as pd
from scipy.stats import hypergeom

print("=== C.3. FULL-DICTIONARY ENRICHMENT ===")

N = len(C2_genes)
M_FEATURES, M_TERMS = C2_overlap.shape
M_TESTS = M_FEATURES * M_TERMS

assert N == 6642
assert M_FEATURES == 12288
assert M_TERMS == 4420
assert M_TESTS == 54312960

# --------------------------------------------------
# 1. Identify explicitly evaluated hypotheses
# --------------------------------------------------

overlap_csr = C2_overlap.tocsr()

explicit_positions = np.flatnonzero(
    overlap_csr.data >= 2
)

overlap_values = overlap_csr.data[
    explicit_positions
].astype(np.int32)

term_indices = overlap_csr.indices[
    explicit_positions
]

feature_rows = (
    np.searchsorted(
        overlap_csr.indptr,
        explicit_positions,
        side="right"
    ) - 1
).astype(np.int32)

N_EXPLICIT = len(explicit_positions)

assert N_EXPLICIT == 609033

print("Total hypotheses:", M_TESTS)
print("Explicit overlap >= 2:", N_EXPLICIT)
print("Implicit p = 1:", M_TESTS - N_EXPLICIT)

# --------------------------------------------------
# 2. Precompute exact hypergeometric probabilities
# --------------------------------------------------

# All features contain exactly 20 selected genes.
# Term sizes are integers between 5 and 500.

MAX_OVERLAP = 20
MAX_TERM_SIZE = 500

p_lookup = np.ones(
    (MAX_OVERLAP + 1, MAX_TERM_SIZE + 1),
    dtype=np.float64
)

term_size_grid = np.arange(
    MAX_TERM_SIZE + 1
)

for overlap in range(2, MAX_OVERLAP + 1):

    p_lookup[overlap] = hypergeom.sf(
        overlap - 1,
        N,
        term_size_grid,
        20
    )

explicit_p = p_lookup[
    overlap_values,
    C2_term_sizes[term_indices]
]

assert np.isfinite(explicit_p).all()
assert np.all((explicit_p >= 0) & (explicit_p <= 1))

print("\nExact enrichment p-values computed.")

# --------------------------------------------------
# 3. Global Benjamini-Hochberg correction
# --------------------------------------------------

# The remaining hypotheses have p = 1.
# They contribute to the total family size M_TESTS.
# We do not need to materialize them individually.

sort_order = np.argsort(
    explicit_p,
    kind="mergesort"
)

sorted_p = explicit_p[sort_order]

ranks = np.arange(
    1,
    N_EXPLICIT + 1,
    dtype=np.float64
)

raw_adjusted = (
    sorted_p * M_TESTS / ranks
)

adjusted_sorted = np.minimum.accumulate(
    raw_adjusted[::-1]
)[::-1]

adjusted_sorted = np.clip(
    adjusted_sorted,
    0.0,
    1.0
)

q_global = np.empty_like(explicit_p)

q_global[sort_order] = adjusted_sorted

assert np.isfinite(q_global).all()
assert np.all(q_global >= explicit_p - 1e-12)
assert np.all(q_global <= 1)

# --------------------------------------------------
# 4. Identify significant associations
# --------------------------------------------------

FDR_THRESHOLD = 0.05

significant_mask = (
    q_global <= FDR_THRESHOLD
)

significant_rows = feature_rows[
    significant_mask
]

significant_terms = term_indices[
    significant_mask
]

significant_overlap = overlap_values[
    significant_mask
]

significant_p = explicit_p[
    significant_mask
]

significant_q = q_global[
    significant_mask
]

C3_significant = pd.DataFrame({
    "layer": significant_rows // 3072,
    "feature": significant_rows % 3072,
    "term_index": significant_terms,
    "overlap": significant_overlap,
    "raw_p": significant_p,
    "q_global": significant_q,
})

print("\n=== GLOBAL BH RESULTS ===")

print(
    "Significant feature-term associations:",
    len(C3_significant)
)

print(
    "Features with significant associations:",
    len(np.unique(significant_rows))
)

# --------------------------------------------------
# 5. Layer-level summaries
# --------------------------------------------------

summary_rows = []

for layer_num in range(4):

    layer_result = C3_significant[
        C3_significant["layer"] == layer_num
    ]

    summary_rows.append({
        "layer": f"decoder.{layer_num}",
        "features": 3072,
        "hypotheses": 3072 * M_TERMS,
        "significant_associations": len(layer_result),
        "features_with_significant_terms": (
            layer_result["feature"].nunique()
        ),
    })

C3_summary = pd.DataFrame(summary_rows)

print("\n=== LAYER-LEVEL SUMMARY ===")
print(C3_summary.to_string(index=False))

# --------------------------------------------------
# 6. Functional-source summaries
# --------------------------------------------------

sources = np.asarray([
    str(key[0]) for key in C2_terms
])

significant_sources = sources[
    significant_terms
]

print("\n=== SIGNIFICANT ASSOCIATIONS BY SOURCE ===")

for source in ["GO_BP", "KEGG", "Reactome"]:
    print(
        f"{source}:",
        int(np.count_nonzero(
            significant_sources == source
        ))
    )

# --------------------------------------------------
# 7. Inspect selected exploratory features
# --------------------------------------------------

print("\n=== SELECTED FEATURE RESULTS ===")

for fid in [0, 189, 2120]:

    subset = C3_significant.loc[
        (C3_significant["layer"] == 0)
        & (C3_significant["feature"] == fid)
    ].sort_values("q_global")

    print(f"\ndecoder.0 feature {fid}")
    print("Significant terms:", len(subset))

    for row in subset.head(3).itertuples():

        source, term = C2_terms[row.term_index]

        print(
            f"  {source}: {term}\n"
            f"    overlap={row.overlap}, "
            f"p={row.raw_p:.4e}, "
            f"q={row.q_global:.4e}"
        )

# --------------------------------------------------
# 8. Integrity checks
# --------------------------------------------------

assert (
    C3_summary["significant_associations"].sum()
    == len(C3_significant)
)

assert len(C3_significant) <= N_EXPLICIT

print("\nFull-dictionary enrichment completed.")
print("Original data and annotations were not modified.")

=== C.3. FULL-DICTIONARY ENRICHMENT ===
Total hypotheses: 54312960
Explicit overlap >= 2: 609033
Implicit p = 1: 53703927

Exact enrichment p-values computed.

=== GLOBAL BH RESULTS ===
Significant feature-term associations: 73646
Features with significant associations: 6491

=== LAYER-LEVEL SUMMARY ===
    layer  features  hypotheses  significant_associations  features_with_significant_terms
decoder.0      3072    13578240                     18928                             1683
decoder.1      3072    13578240                     18144                             1609
decoder.2      3072    13578240                     18289                             1598
decoder.3      3072    13578240                     18285                             1601

=== SIGNIFICANT ASSOCIATIONS BY SOURCE ===
GO_BP: 25753
KEGG: 11220
Reactome: 36673

=== SELECTED FEATURE RESULTS ===

decoder.0 feature 0
Significant terms: 2
  GO_BP: Purine-Containing Compound Biosynthetic Process (GO:0072522)
    overl

### C.3.1. Global BH Enrichment Results

The full-dictionary enrichment analysis evaluated
54,312,960 feature–term hypotheses across four
decoder SAE dictionaries and 4,420 functional terms.

Only 609,033 pairs with an overlap of at least
two genes required explicit hypergeometric testing.
The remaining hypotheses were assigned p = 1
under the original annotation convention.

Global Benjamini–Hochberg adjustment at q <= 0.05
identified 73,646 significant feature–term
associations involving 6,491 distinct SAE features.

| Layer | Significant associations | Annotated features |
|---|---:|---:|
| decoder.0 | 18,928 | 1,683 |
| decoder.1 | 18,144 | 1,609 |
| decoder.2 | 18,289 | 1,598 |
| decoder.3 | 18,285 | 1,601 |

Significant associations by reference source:

- GO BP: 25,753
- KEGG: 11,220
- Reactome: 36,673

These results use a single global BH family
across all four decoder dictionaries.

They are not directly equivalent to the original
per-layer annotation results.

### Interpretation

The analysis identifies extensive functional
enrichment among training-associated SAE gene sets
under the specified hypergeometric null model.

However, the validity and interpretation of these
associations depend on the gene-selection mechanism,
annotation structure, and multiple-testing assumptions.

Statistical significance does not imply SAE-specific
biological function or computational causality.

The next analyses evaluate robustness to gene
annotation degree and firing-support limitations.

### C.3.2. Multiple-Testing and Feature-Support Audit

The global BH rejection count is independently
checked using the step-up threshold rule.

The prevalence of significant functional
associations is then compared between features
with and without complete Top-20 gene support
at the predefined 25% training-condition threshold.

This comparison is descriptive and exploratory.

Support-based stratification is not treated as
an independent statistical correction.

Finally, selected decoder.0 feature associations
are inspected together with their enrichment
effect sizes and adjusted p-values.

In [23]:
import numpy as np
import pandas as pd

print("=== C.3.2. BH AND SUPPORT AUDIT ===")

# --------------------------------------------------
# 1. Independently verify the BH rejection count
# --------------------------------------------------

alpha = 0.05

critical_values = (
    alpha
    * np.arange(1, N_EXPLICIT + 1)
    / M_TESTS
)

passing_ranks = np.flatnonzero(
    sorted_p <= critical_values
)

bh_rejections = (
    int(passing_ranks[-1]) + 1
    if len(passing_ranks)
    else 0
)

assert bh_rejections == len(C3_significant)

print("\n=== BH VERIFICATION ===")
print("Step-up rejection count:", bh_rejections)
print("Adjusted-q rejection count:", len(C3_significant))
print("BH consistency: PASS")

# --------------------------------------------------
# 2. Map significance to every decoder feature
# --------------------------------------------------

significant_features = set(
    zip(
        C3_significant["layer"].astype(int),
        C3_significant["feature"].astype(int),
    )
)

feature_audit = support_df_c1.copy()

feature_audit["has_significant_term"] = [
    (
        int(layer.split(".")[1]),
        int(feature_id)
    ) in significant_features
    for layer, feature_id in zip(
        feature_audit["layer"],
        feature_audit["feature"]
    )
]

# --------------------------------------------------
# 3. Compare gene-support categories
# --------------------------------------------------

support_comparison = (
    feature_audit
    .groupby(
        ["layer", "all_20_support_ge_25pct"]
    )
    .agg(
        features=("feature", "count"),
        significant_features=(
            "has_significant_term", "sum"
        ),
        significant_fraction=(
            "has_significant_term", "mean"
        ),
    )
    .reset_index()
)

print("\n=== SIGNIFICANCE BY GENE SUPPORT ===")

print(
    support_comparison.round(4).to_string(index=False)
)

# --------------------------------------------------
# 4. Inspect selected feature-term associations
# --------------------------------------------------

print("\n=== SELECTED DECODER.0 FEATURES ===")

for fid in [0, 189, 2120]:

    subset = C3_significant.loc[
        C3_significant["layer"].eq(0)
        & C3_significant["feature"].eq(fid)
    ].sort_values(
        ["q_global", "raw_p"]
    )

    print(f"\nFeature {fid}")
    print("Significant associations:", len(subset))

    if subset.empty:
        continue

    for row in subset.head(3).itertuples():

        term_id = int(row.term_index)
        source, term = C2_terms[term_id]

        term_size = int(C2_term_sizes[term_id])

        expected_overlap = (
            20 * term_size / N
        )

        fold_enrichment = (
            row.overlap / expected_overlap
        )

        print(f"  {source}: {term}")
        print(
            f"  overlap={row.overlap}, "
            f"term_size={term_size}"
        )
        print(
            f"  fold_enrichment={fold_enrichment:.2f}, "
            f"p={row.raw_p:.3e}, "
            f"q={row.q_global:.3e}"
        )

print("\nRead-only BH and support audit completed.")

=== C.3.2. BH AND SUPPORT AUDIT ===

=== BH VERIFICATION ===
Step-up rejection count: 73646
Adjusted-q rejection count: 73646
BH consistency: PASS

=== SIGNIFICANCE BY GENE SUPPORT ===
    layer  all_20_support_ge_25pct  features  significant_features  significant_fraction
decoder.0                    False        43                    12                0.2791
decoder.0                     True      3029                  1671                0.5517
decoder.1                    False        88                    26                0.2955
decoder.1                     True      2984                  1583                0.5305
decoder.2                    False       139                    35                0.2518
decoder.2                     True      2933                  1563                0.5329
decoder.3                    False       205                    51                0.2488
decoder.3                     True      2867                  1550                0.5406

=== SELECTED 

### C.4. Degree-Aware Enrichment: Exploratory Evaluation Panel

The global BH analysis identified 73,646 significant
feature–term associations, involving 6,491 decoder
SAE features.

Significant associations were more frequent among
features whose Top-20 genes had stronger firing support.

This experiment evaluates whether functional
enrichment remains unusual under null models that
account for the annotation-degree distribution
of the selected genes.

Two gene-label reassignment procedures are considered:

1. Global reassignment across the PIE gene universe.
2. Reassignment within annotation-degree strata.

An exploratory evaluation panel is constructed from
all four decoder dictionaries.

Within each decoder dictionary, six features are
sampled from each combination of:

- Strong versus weaker Top-20 firing support.
- Presence versus absence of global BH significance.

This yields 96 stratified evaluation features.

Previously examined features 0, 189, and 2120
from decoder.0 are retained as named diagnostic cases.

The panel is selected using previously observed
support and significance outcomes. Consequently,
it is not a representative random sample of all
SAE features, and the evaluation is exploratory.

The same 4,420 functional reference sets and the
original training-derived Top-20 gene lists are
retained throughout.

Gene rankings, original annotations, and SAE
checkpoints are not modified.

In [24]:
import numpy as np
import pandas as pd

print("=== C.4. DEGREE-AWARE EVALUATION DESIGN ===")

C4_SEED = 20261113
C4_PER_GROUP = 6

rng_c4 = np.random.default_rng(C4_SEED)

# --------------------------------------------------
# 1. Classify all decoder SAE features
# --------------------------------------------------

significant_pairs_c4 = set(
    zip(
        C3_significant["layer"].astype(int),
        C3_significant["feature"].astype(int),
    )
)

C4_population = support_df_c1.copy()

C4_population["layer_num"] = (
    C4_population["layer"]
    .str.split(".")
    .str[-1]
    .astype(int)
)

C4_population["bh_significant"] = [
    (int(layer), int(fid)) in significant_pairs_c4
    for layer, fid in zip(
        C4_population["layer_num"],
        C4_population["feature"]
    )
]

C4_population["strong_support"] = (
    C4_population["all_20_support_ge_25pct"]
    .astype(bool)
)

assert len(C4_population) == 12288

# --------------------------------------------------
# 2. Construct the exploratory feature panel
# --------------------------------------------------

selected_rows_c4 = []

for layer_num in range(4):

    layer_df = C4_population[
        C4_population["layer_num"] == layer_num
    ]

    for strong_support in [False, True]:
        for bh_significant in [False, True]:

            candidates = layer_df[
                layer_df["strong_support"].eq(strong_support)
                & layer_df["bh_significant"].eq(bh_significant)
            ]

            if len(candidates) < C4_PER_GROUP:
                raise ValueError(
                    f"Insufficient candidates: decoder.{layer_num}, "
                    f"support={strong_support}, "
                    f"BH={bh_significant}"
                )

            sampled = rng_c4.choice(
                candidates["feature"].to_numpy(),
                size=C4_PER_GROUP,
                replace=False
            )

            for fid in sampled:
                selected_rows_c4.append(
                    (layer_num, int(fid))
                )

# Include explicitly named diagnostic features.
for fid in [0, 189, 2120]:
    selected_rows_c4.append((0, fid))

selected_rows_c4 = sorted(set(selected_rows_c4))

C4_selected_rows = np.array(
    [
        layer * 3072 + fid
        for layer, fid in selected_rows_c4
    ],
    dtype=np.int32
)

C4_panel = C4_population.iloc[
    C4_selected_rows
].copy().reset_index(drop=True)

assert len(C4_selected_rows) >= 96
assert len(C4_selected_rows) <= 99
assert len(set(C4_selected_rows)) == len(C4_selected_rows)

print("\n=== SELECTED FEATURE PANEL ===")

print("Selected features:", len(C4_panel))
print("Decoder dictionaries:", C4_panel["layer"].nunique())

print(
    pd.crosstab(
        [
            C4_panel["layer"],
            C4_panel["strong_support"],
        ],
        C4_panel["bh_significant"]
    ).to_string()
)

# --------------------------------------------------
# 3. Calculate gene annotation degrees
# --------------------------------------------------

C4_gene_degrees = np.asarray(
    C2_term_matrix.sum(axis=1)
).ravel().astype(np.int32)

assert len(C4_gene_degrees) == 6642
assert C4_gene_degrees.min() >= 0

positive_degrees = C4_gene_degrees[
    C4_gene_degrees > 0
]

C4_cutoffs = np.unique(
    np.quantile(
        positive_degrees,
        [0.2, 0.4, 0.6, 0.8]
    )
)

C4_gene_strata = np.zeros(
    6642,
    dtype=np.int16
)

positive_mask = C4_gene_degrees > 0

C4_gene_strata[positive_mask] = (
    1 + np.searchsorted(
        C4_cutoffs,
        C4_gene_degrees[positive_mask],
        side="right"
    )
)

C4_stratum_indices = {
    int(label): np.flatnonzero(
        C4_gene_strata == label
    )
    for label in np.unique(C4_gene_strata)
}

print("\n=== ANNOTATION-DEGREE STRATA ===")

print("Degree cutoffs:", C4_cutoffs.tolist())

for label, indices in C4_stratum_indices.items():

    values = C4_gene_degrees[indices]

    print(
        f"Stratum {label}: "
        f"genes={len(indices)}, "
        f"degree_range=[{values.min()}, {values.max()}]"
    )

# --------------------------------------------------
# 4. Recover selected Top-20 gene sets
# --------------------------------------------------

C4_selected_gene_positions = []

for row in C4_selected_rows:

    start = C2_feature_matrix.indptr[row]
    end = C2_feature_matrix.indptr[row + 1]

    indices = C2_feature_matrix.indices[
        start:end
    ].copy()

    assert len(indices) == 20
    assert len(np.unique(indices)) == 20

    C4_selected_gene_positions.append(indices)

C4_selected_gene_positions = np.stack(
    C4_selected_gene_positions
)

assert C4_selected_gene_positions.shape == (
    len(C4_panel), 20
)

# --------------------------------------------------
# 5. Verify named diagnostic features
# --------------------------------------------------

print("\n=== NAMED DIAGNOSTIC FEATURES ===")

diagnostics = C4_panel.loc[
    C4_panel["layer_num"].eq(0)
    & C4_panel["feature"].isin([0, 189, 2120]),
    [
        "feature",
        "strong_support",
        "bh_significant",
        "median_support_fraction",
    ]
]

assert len(diagnostics) == 3

print(diagnostics.to_string(index=False))

print("\nRead-only evaluation panel completed.")

=== C.4. DEGREE-AWARE EVALUATION DESIGN ===

=== SELECTED FEATURE PANEL ===
Selected features: 98
Decoder dictionaries: 4
bh_significant            False  True 
layer     strong_support              
decoder.0 False               6      6
          True                6      8
decoder.1 False               6      6
          True                6      6
decoder.2 False               6      6
          True                6      6
decoder.3 False               6      6
          True                6      6

=== ANNOTATION-DEGREE STRATA ===
Degree cutoffs: [5.0, 10.0, 16.0, 29.0]
Stratum 0: genes=908, degree_range=[0, 0]
Stratum 1: genes=1037, degree_range=[1, 4]
Stratum 2: genes=1228, degree_range=[5, 9]
Stratum 3: genes=1110, degree_range=[10, 15]
Stratum 4: genes=1175, degree_range=[16, 28]
Stratum 5: genes=1184, degree_range=[29, 375]

=== NAMED DIAGNOSTIC FEATURES ===
 feature  strong_support  bh_significant  median_support_fraction
       0            True            True         

### C.4.1. Global and Degree-Stratified Permutation Tests

The exploratory panel contains 98 SAE features selected
across four decoder dictionaries, firing-support groups,
and global BH significance categories.

For each feature, the observed statistic is the strongest
one-sided hypergeometric enrichment across the 4,420
functional terms:

T = -log10(minimum enrichment p-value).

Terms with fewer than two overlapping genes receive p = 1,
consistent with the original annotation convention.

Two null models are evaluated:

1. Global permutation of PIE gene labels.
2. Permutation within six annotation-degree strata.

Within each permutation replicate, the same gene-label
mapping is applied to all selected features. This preserves
overlap relationships among feature gene sets.

Both feature-level empirical p-values and single-step
maxT p-values across the fixed 98-feature panel are reported.

The panel was selected partly using prior BH results.
Therefore, the empirical p-values are exploratory
sensitivity diagnostics, not selection-adjusted
confirmatory p-values for the full SAE population.

The degree-stratified null preserves coarse annotation
degree classes, not exact gene degrees or biological
covariates.

No original data, annotations, or checkpoints are modified.

In [25]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from scipy.stats import hypergeom

print("=== C.4.1. DEGREE-AWARE PERMUTATION TESTS ===")

C41_SEED = 20261113
C41_N_PERM = 1000

rng_c41 = np.random.default_rng(C41_SEED)

# --------------------------------------------------
# 1. Verify the evaluation panel
# --------------------------------------------------

n_genes = C2_feature_matrix.shape[1]
n_features = len(C4_selected_rows)
n_terms = len(C2_terms)

assert n_genes == 6642
assert n_features == len(C4_panel) == 98
assert n_terms == 4420

assert C4_selected_gene_positions.shape == (
    n_features, 20
)

assert np.all(
    (C4_selected_gene_positions >= 0)
    & (C4_selected_gene_positions < n_genes)
)

stratum_members = np.concatenate(
    list(C4_stratum_indices.values())
)

assert np.array_equal(
    np.sort(stratum_members),
    np.arange(n_genes)
)

# --------------------------------------------------
# 2. Precompute exact enrichment probabilities
# --------------------------------------------------

p_table_c41 = np.ones(
    (21, 501),
    dtype=np.float64
)

term_sizes_grid = np.arange(5, 501)

for overlap in range(2, 21):

    p_table_c41[overlap, 5:501] = hypergeom.sf(
        overlap - 1,
        n_genes,
        term_sizes_grid,
        20
    )

assert np.all(
    (C2_term_sizes >= 5)
    & (C2_term_sizes <= 500)
)

# --------------------------------------------------
# 3. Define a batch enrichment statistic
# --------------------------------------------------

panel_row_indices = np.repeat(
    np.arange(n_features, dtype=np.int32),
    20
)

def panel_enrichment_statistics(gene_positions):

    gene_positions = np.asarray(
        gene_positions,
        dtype=np.int32
    )

    assert gene_positions.shape == (n_features, 20)

    feature_matrix = csr_matrix(
        (
            np.ones(
                n_features * 20,
                dtype=np.int16
            ),
            (
                panel_row_indices,
                gene_positions.ravel()
            )
        ),
        shape=(n_features, n_genes),
        dtype=np.int16
    )

    overlaps = (
        feature_matrix @ C2_term_matrix
    ).tocsr()

    explicit_positions = np.flatnonzero(
        overlaps.data >= 2
    )

    statistics = np.zeros(
        n_features,
        dtype=np.float64
    )

    if len(explicit_positions) == 0:
        return statistics

    row_indices = (
        np.searchsorted(
            overlaps.indptr,
            explicit_positions,
            side="right"
        ) - 1
    )

    term_indices = overlaps.indices[
        explicit_positions
    ]

    overlap_counts = overlaps.data[
        explicit_positions
    ].astype(np.int32)

    p_values = p_table_c41[
        overlap_counts,
        C2_term_sizes[term_indices]
    ]

    if not np.all(
        np.isfinite(p_values) & (p_values > 0)
    ):
        raise ValueError(
            "Invalid enrichment probability"
        )

    scores = -np.log10(p_values)

    np.maximum.at(
        statistics,
        row_indices,
        scores
    )

    return statistics

# --------------------------------------------------
# 4. Reproduce observed enrichment
# --------------------------------------------------

observed_c41 = panel_enrichment_statistics(
    C4_selected_gene_positions
)

assert np.isfinite(observed_c41).all()

# Cross-check against the full overlap matrix.
original_overlap = C2_overlap[
    C4_selected_rows
].tocsr()

assert original_overlap.shape == (
    n_features, n_terms
)

def direct_row_statistic(row):

    start = original_overlap.indptr[row]
    end = original_overlap.indptr[row + 1]

    mask = (
        original_overlap.data[start:end] >= 2
    )

    overlaps = (
        original_overlap.data[start:end][mask]
    )

    terms = (
        original_overlap.indices[start:end][mask]
    )

    if len(overlaps) == 0:
        return 0.0

    p = p_table_c41[
        overlaps,
        C2_term_sizes[terms]
    ]

    return float(-np.log10(p.min()))

for i in range(n_features):

    assert np.isclose(
        observed_c41[i],
        direct_row_statistic(i),
        atol=1e-10
    )

print("Panel features:", n_features)
print("Functional terms:", n_terms)
print("Observed-statistic verification: PASS")

# --------------------------------------------------
# 5. Global and degree-stratified permutations
# --------------------------------------------------

null_global_c41 = np.empty(
    (C41_N_PERM, n_features),
    dtype=np.float64
)

null_strat_c41 = np.empty_like(
    null_global_c41
)

for b in range(C41_N_PERM):

    # Global bijection of gene labels.
    global_mapping = rng_c41.permutation(
        n_genes
    )

    # Bijection within annotation-degree strata.
    strat_mapping = np.arange(
        n_genes,
        dtype=np.int32
    )

    for indices in C4_stratum_indices.values():

        strat_mapping[indices] = (
            rng_c41.permutation(indices)
        )

    global_genes = global_mapping[
        C4_selected_gene_positions
    ]

    strat_genes = strat_mapping[
        C4_selected_gene_positions
    ]

    null_global_c41[b] = (
        panel_enrichment_statistics(global_genes)
    )

    null_strat_c41[b] = (
        panel_enrichment_statistics(strat_genes)
    )

print("Permutations per null model:", C41_N_PERM)

# --------------------------------------------------
# 6. Empirical p-values and fixed-panel maxT
# --------------------------------------------------

def empirical_feature_p(null_matrix, observed):

    return (
        1 + np.count_nonzero(
            null_matrix >= observed[None, :],
            axis=0
        )
    ) / (null_matrix.shape[0] + 1)


def empirical_maxT_p(null_matrix, observed):

    null_maximum = null_matrix.max(axis=1)

    return (
        1 + np.count_nonzero(
            null_maximum[:, None] >= observed[None, :],
            axis=0
        )
    ) / (null_matrix.shape[0] + 1)


p_global_c41 = empirical_feature_p(
    null_global_c41,
    observed_c41
)

p_strat_c41 = empirical_feature_p(
    null_strat_c41,
    observed_c41
)

maxT_global_c41 = empirical_maxT_p(
    null_global_c41,
    observed_c41
)

maxT_strat_c41 = empirical_maxT_p(
    null_strat_c41,
    observed_c41
)

assert np.all(
    maxT_global_c41 >= p_global_c41 - 1e-12
)

assert np.all(
    maxT_strat_c41 >= p_strat_c41 - 1e-12
)

# --------------------------------------------------
# 7. Build the results table
# --------------------------------------------------

C41_results = C4_panel[
    [
        "layer",
        "layer_num",
        "feature",
        "strong_support",
        "bh_significant"
    ]
].copy()

C41_results["observed_T"] = observed_c41
C41_results["global_p"] = p_global_c41
C41_results["strat_p"] = p_strat_c41
C41_results["global_maxT"] = maxT_global_c41
C41_results["strat_maxT"] = maxT_strat_c41

C41_results["passes_both_fixed_panel_maxT"] = (
    (maxT_global_c41 < 0.05)
    & (maxT_strat_c41 < 0.05)
)

print("\n=== EXPLORATORY PANEL SUMMARY ===")

print("Total features:", len(C41_results))

print(
    "Global feature-level p < 0.05:",
    int((p_global_c41 < 0.05).sum())
)

print(
    "Stratified feature-level p < 0.05:",
    int((p_strat_c41 < 0.05).sum())
)

print(
    "Global fixed-panel maxT p < 0.05:",
    int((maxT_global_c41 < 0.05).sum())
)

print(
    "Stratified fixed-panel maxT p < 0.05:",
    int((maxT_strat_c41 < 0.05).sum())
)

print(
    "Passing both fixed-panel maxT controls:",
    int(
        C41_results[
            "passes_both_fixed_panel_maxT"
        ].sum()
    )
)

# --------------------------------------------------
# 8. Summaries by prior BH result and gene support
# --------------------------------------------------

group_summary = C41_results.groupby(
    ["bh_significant", "strong_support"]
).agg(
    features=("feature", "count"),
    median_T=("observed_T", "median"),
    global_pass=(
        "global_maxT",
        lambda x: int((x < 0.05).sum())
    ),
    strat_pass=(
        "strat_maxT",
        lambda x: int((x < 0.05).sum())
    )
)

print("\n=== EXPLORATORY GROUP COMPARISON ===")

print(
    group_summary.round(4).to_string()
)

# --------------------------------------------------
# 9. Named diagnostic cases
# --------------------------------------------------

print("\n=== NAMED DECODER.0 FEATURES ===")

named_cases = C41_results.loc[
    C41_results["layer_num"].eq(0)
    & C41_results["feature"].isin(
        [0, 189, 2120]
    )
].sort_values("feature")

print(
    named_cases[
        [
            "feature",
            "observed_T",
            "global_p",
            "strat_p",
            "global_maxT",
            "strat_maxT"
        ]
    ].round(5).to_string(index=False)
)

print("\n=== STRONGEST OBSERVED SIGNALS ===")

print(
    C41_results.sort_values(
        "observed_T",
        ascending=False
    )[
        [
            "layer",
            "feature",
            "observed_T",
            "global_maxT",
            "strat_maxT"
        ]
    ].head(10).round(5).to_string(index=False)
)

print("\nRead-only permutation analysis completed.")

=== C.4.1. DEGREE-AWARE PERMUTATION TESTS ===
Panel features: 98
Functional terms: 4420
Observed-statistic verification: PASS
Permutations per null model: 1000

=== EXPLORATORY PANEL SUMMARY ===
Total features: 98
Global feature-level p < 0.05: 51
Stratified feature-level p < 0.05: 50
Global fixed-panel maxT p < 0.05: 22
Stratified fixed-panel maxT p < 0.05: 19
Passing both fixed-panel maxT controls: 19

=== EXPLORATORY GROUP COMPARISON ===
                               features  median_T  global_pass  strat_pass
bh_significant strong_support                                             
False          False                 24    3.2664            0           0
               True                  24    3.5143            0           0
True           False                 24    5.3460            8           7
               True                  26    6.6333           14          12

=== NAMED DECODER.0 FEATURES ===
 feature  observed_T  global_p  strat_p  global_maxT  strat_maxT
      

### C.4.2. Degree-Aware Permutation Results

The exploratory evaluation panel included 98 SAE
features from four decoder dictionaries.

Two null models were evaluated using 1,000
permutations each: global gene-label reassignment
and annotation-degree-stratified reassignment.

| Test | Features with p < 0.05 |
|---|---:|
| Global, feature-level | 51 |
| Degree-stratified, feature-level | 50 |
| Global fixed-panel maxT | 22 |
| Degree-stratified fixed-panel maxT | 19 |
| Passing both maxT controls | 19 |

Among the 48 previously BH-nonsignificant features,
none passed the fixed-panel maxT tests.

Among the previously BH-significant features,
adjusted enrichment was more frequent in the
stronger firing-support category.

Decoder.0 feature 0 passed both controls:

- Global maxT p = 0.02098.
- Degree-stratified maxT p = 0.02498.

### Interpretation

The results demonstrate that some functional
enrichment signals remain unusual under both
specified gene-reassignment null models.

However, the evaluation panel was constructed
using previously observed BH significance and
gene-support categories.

Therefore, these tests are exploratory sensitivity
analyses. Fixed-panel maxT does not fully correct
for the data-dependent selection of features.

The 19/98 result must not be extrapolated to the
complete population of 12,288 SAE features.

Degree stratification preserves coarse annotation
degree classes, not exact degrees or all relevant
gene-level covariates.

### Remaining statistical limitations

A full-dictionary, selection-aware degree-controlled
analysis has not yet been performed.

The current findings do not establish SAE-specific
functional mechanisms, perturbation sensitivity,
or computational or biological causality.

## D. Cross-Layer Statistical Validation

### D.1. Existing Cross-Layer Artifacts and Methods

The previous cross-layer analysis examined
relationships among SAE dictionaries using
feature similarity, matching, and co-activation.

Before constructing statistical null models,
the saved results and original matching
implementation are inspected.

The audit identifies:

- Available cross-layer result files.
- Feature-matching and co-activation methods.
- Existing permutation or random controls.
- The scope of previously evaluated layer pairs.

The subsequent statistical analysis will account
for feature selection and matching whenever
evaluating the significance of reported matches.

This stage is read-only and does not repeat
SAE training or activation extraction.

In [26]:
from pathlib import Path
import re

print("=== D.1. CROSS-LAYER ARTIFACT INVENTORY ===")

results_root = ROOT / "results"

keywords = (
    "cross_layer",
    "cross-layer",
    "coactivation",
    "co_activation",
    "matching",
    "mutual",
    "feature_pair",
    "mnn",
)

# --------------------------------------------------
# 1. Locate saved analysis artifacts
# --------------------------------------------------

result_files = [
    path
    for path in results_root.rglob("*")
    if path.is_file()
    and any(
        word in str(path.relative_to(results_root)).lower()
        for word in keywords
    )
]

result_files.sort(
    key=lambda path: str(path).lower()
)

print("\n=== SAVED CROSS-LAYER FILES ===")
print("Matching files:", len(result_files))

for path in result_files[:40]:

    print(
        str(path.relative_to(ROOT)),
        f"({path.stat().st_size:,} bytes)"
    )

if len(result_files) > 40:
    print(
        f"... {len(result_files) - 40} additional files"
    )

# --------------------------------------------------
# 2. Locate implementation files
# --------------------------------------------------

source_files = []

for directory in [ROOT / "src", ROOT / "scripts"]:

    if not directory.exists():
        continue

    for path in directory.rglob("*.py"):

        if any(
            word in path.name.lower()
            for word in keywords
        ):
            source_files.append(path)

source_files = sorted(set(source_files))

print("\n=== CROSS-LAYER SOURCE FILES ===")

for path in source_files:
    print(path.relative_to(ROOT))

# --------------------------------------------------
# 3. Identify methods without executing them
# --------------------------------------------------

pattern = re.compile(
    r"mutual|nearest|cosine|matching|"
    r"permutation|coactivation|co_activation|"
    r"random|overlap|spearman",
    re.IGNORECASE
)

print("\n=== SOURCE METHOD INDICATORS ===")

for path in source_files[:12]:

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if pattern.search(line)
    ]

    print(f"\n{path.relative_to(ROOT)}")
    print("Matching lines:", len(matches))

    for lineno, line in matches[:12]:
        print(f"  {lineno}: {line[:140]}")

# --------------------------------------------------
# 4. Verify previous notebook availability
# --------------------------------------------------

notebook_path = (
    ROOT / "notebooks"
    / "05_cross_layer_analysis.ipynb"
)

print("\n=== PREVIOUS NOTEBOOK ===")
print("Notebook 05 exists:", notebook_path.exists())

print("\nRead-only cross-layer inventory completed.")

=== D.1. CROSS-LAYER ARTIFACT INVENTORY ===

=== SAVED CROSS-LAYER FILES ===
Matching files: 25
results\cross_layer\encoder_0_1_matched_activations.npz (2,456,415 bytes)
results\cross_layer\encoder_0_to_encoder_1.npz (17,543 bytes)
results\cross_layer_independent\adjacent_dictionary_cosine.png (173,960 bytes)
results\cross_layer_independent\biological_correspondence\decoder_gene_correspondence.csv (957 bytes)
results\cross_layer_independent\biological_correspondence\decoder_gene_correspondence.json (2,517 bytes)
results\cross_layer_independent\coactivation\all_pairs_coactivation.csv (1,450 bytes)
results\cross_layer_independent\coactivation\all_pairs_coactivation.json (4,345 bytes)
results\cross_layer_independent\coactivation\coactivation_permutation_200.json (3,985 bytes)
results\cross_layer_independent\cross_layer_summary.csv (1,733 bytes)
results\cross_layer_independent\dictionary_correspondence.csv (1,131 bytes)
results\cross_layer_independent\dictionary_correspondence.json (3,708 

### D.2. Cross-Layer Matching and Null-Model Provenance

The saved cross-layer results include feature-dictionary
matches, co-activation summaries, permutation controls,
and biological correspondence reports.

Before introducing additional statistical tests,
the existing result schemas and matching algorithms
are inspected.

The audit focuses on:

- The similarity measure and matching direction.
- Whether matches are mutual or one-directional.
- The procedure used to select feature pairs.
- Existing permutation and random controls.
- The statistical families covered by previous tests.

A valid selection-aware null should reproduce the
relevant feature-matching procedure, rather than
compare selected high-similarity pairs against
unselected random pairs.

This is a read-only provenance audit.

In [27]:
import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

print("=== D.2. CROSS-LAYER METHOD AND RESULT AUDIT ===")

base = ROOT / "results" / "cross_layer_independent"

# --------------------------------------------------
# 1. Inspect saved dictionary matches
# --------------------------------------------------

match_dir = base / "dictionary_matches"

match_files = sorted(match_dir.glob("*.npz"))

print("\n=== DICTIONARY MATCH FILES ===")
print("Available files:", len(match_files))

sample_names = [
    "decoder_0_to_decoder_1.npz",
    "decoder_1_to_decoder_2.npz",
    "processor_1_to_processor_2.npz",
]

for name in sample_names:

    path = match_dir / name

    if not path.exists():
        print(f"\n{name}: NOT FOUND")
        continue

    print(f"\nFILE: {name}")

    with np.load(path, allow_pickle=False) as data:

        print("Stored arrays:", data.files)

        for key in data.files:

            try:
                array = data[key]

                print(
                    f"  {key}: "
                    f"shape={array.shape}, "
                    f"dtype={array.dtype}"
                )

                if (
                    array.size > 0
                    and np.issubdtype(
                        array.dtype, np.number
                    )
                ):
                    print(
                        "    Sample:",
                        array.ravel()[:5].tolist()
                    )

            except ValueError as exc:
                print(
                    f"  {key}: cannot inspect without "
                    f"pickle ({exc})"
                )

# --------------------------------------------------
# 2. Inspect compact CSV summaries
# --------------------------------------------------

csv_names = [
    "dictionary_correspondence.csv",
    "coactivation/coactivation_summary.csv",
    "biological_correspondence/decoder_gene_correspondence.csv",
]

print("\n=== SAVED CROSS-LAYER TABLES ===")

for name in csv_names:

    path = base / name

    if not path.exists():
        print(f"\n{name}: NOT FOUND")
        continue

    frame = pd.read_csv(path)

    print(f"\nFILE: {name}")
    print("Shape:", frame.shape)
    print("Columns:", frame.columns.tolist())

    print(
        frame.head(2).to_string(index=False)
    )

# --------------------------------------------------
# 3. Inspect existing co-activation permutation results
# --------------------------------------------------

permutation_path = (
    base
    / "coactivation"
    / "coactivation_permutation.json"
)

print("\n=== EXISTING PERMUTATION CONTROL ===")

if permutation_path.exists():

    payload = json.loads(
        permutation_path.read_text(
            encoding="utf-8"
        )
    )

    print("Top-level type:", type(payload).__name__)

    if isinstance(payload, dict):

        print("Keys:", list(payload.keys()))

        for key, value in payload.items():

            if isinstance(
                value, (str, int, float, bool)
            ):
                print(f"  {key}: {value}")

            elif isinstance(value, dict):
                print(
                    f"  {key}: nested keys = "
                    f"{list(value.keys())[:15]}"
                )

            elif isinstance(value, list):
                print(
                    f"  {key}: list of "
                    f"{len(value)} elements"
                )

                if value and isinstance(value[0], dict):
                    print(
                        "    First-record keys:",
                        list(value[0].keys())[:15]
                    )

    elif isinstance(payload, list):
        print("Number of records:", len(payload))

        if payload and isinstance(payload[0], dict):
            print(
                "First-record keys:",
                list(payload[0].keys())[:20]
            )

else:
    print("Permutation JSON not found.")

# --------------------------------------------------
# 4. Locate source implementations
# --------------------------------------------------

print("\n=== CROSS-LAYER SOURCE IMPLEMENTATIONS ===")

search_terms = [
    "mutual_nearest",
    "dictionary_matches",
    "coactivation",
    "co_activation",
    "cross_layer",
    "cosine_similarity",
]

source_candidates = []

for directory in [ROOT / "src", ROOT / "scripts"]:

    if not directory.exists():
        continue

    for path in directory.rglob("*.py"):

        text = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        score = sum(
            text.lower().count(term)
            for term in search_terms
        )

        if score > 0:
            source_candidates.append(
                (score, path, text)
            )

source_candidates.sort(
    key=lambda item: item[0],
    reverse=True
)

for score, path, source in source_candidates[:10]:

    print(
        f"\n{path.relative_to(ROOT)} "
        f"(keyword matches: {score})"
    )

    try:
        tree = ast.parse(source)
    except SyntaxError:
        print("  Source parsing failed.")
        continue

    functions = [
        node.name
        for node in ast.walk(tree)
        if isinstance(
            node, (ast.FunctionDef, ast.AsyncFunctionDef)
        )
    ]

    relevant = [
        name for name in functions
        if any(
            term in name.lower()
            for term in [
                "match",
                "cosine",
                "similar",
                "permut",
                "coactiv",
                "nearest",
                "null",
            ]
        )
    ]

    print("Relevant functions:", relevant[:20])

print("\nRead-only cross-layer provenance audit completed.")

=== D.2. CROSS-LAYER METHOD AND RESULT AUDIT ===

=== DICTIONARY MATCH FILES ===
Available files: 11

FILE: decoder_0_to_decoder_1.npz
Stored arrays: ['match_ab', 'best_cosine', 'mutual_mask']
  match_ab: shape=(3072,), dtype=int64
    Sample: [704, 2294, 1252, 1516, 921]
  best_cosine: shape=(3072,), dtype=float32
    Sample: [0.27530917525291443, 0.2154816836118698, 0.28342023491859436, 0.19598938524723053, 0.20255662500858307]
  mutual_mask: shape=(3072,), dtype=bool

FILE: decoder_1_to_decoder_2.npz
Stored arrays: ['match_ab', 'best_cosine', 'mutual_mask']
  match_ab: shape=(3072,), dtype=int64
    Sample: [1495, 2146, 2991, 210, 1573]
  best_cosine: shape=(3072,), dtype=float32
    Sample: [0.1954777091741562, 0.3175305128097534, 0.2190760374069214, 0.2759440243244171, 0.34862130880355835]
  mutual_mask: shape=(3072,), dtype=bool

FILE: processor_1_to_processor_2.npz
Stored arrays: ['match_ab', 'best_cosine', 'mutual_mask']
  match_ab: shape=(3072,), dtype=int64
    Sample: [2550,

### D.2.1. Matching Algorithm and Control Verification

The saved dictionary-matching results contain
best-match indices, cosine similarities, and
mutual-nearest-neighbor indicators.

Before evaluating statistical significance,
the original matching implementation and existing
co-activation permutation controls are inspected.

The audit verifies the precise matching definition,
the use of cosine similarity, and the scope of
previously computed null models.

A selection-aware statistical test must repeat
the matching procedure under the null model.

Permuting feature identifiers alone is insufficient
because it does not change the maximum similarity
between two fixed dictionaries.

This verification is read-only.

In [28]:
import json
import numpy as np
import pandas as pd

print("=== D.2.1. MATCHING AND NULL-MODEL AUDIT ===")

base = ROOT / "results" / "cross_layer_independent"
match_dir = base / "dictionary_matches"

# --------------------------------------------------
# 1. Summarize all saved dictionary matches
# --------------------------------------------------

rows = []

for path in sorted(match_dir.glob("*.npz")):

    with np.load(path, allow_pickle=False) as data:

        matches = np.asarray(data["match_ab"])
        cosine = np.asarray(data["best_cosine"])
        mutual = np.asarray(data["mutual_mask"])

    assert matches.shape == (3072,)
    assert cosine.shape == (3072,)
    assert mutual.shape == (3072,)
    assert np.isfinite(cosine).all()

    rows.append({
        "pair": path.stem,
        "mean_cosine": float(cosine.mean()),
        "median_cosine": float(np.median(cosine)),
        "mutual_pairs": int(mutual.sum()),
        "mutual_fraction": float(mutual.mean()),
    })

match_summary_d2 = pd.DataFrame(rows)

print("\n=== SAVED MATCHING SUMMARY ===")
print(match_summary_d2.round(4).to_string(index=False))

# --------------------------------------------------
# 2. Inspect existing co-activation permutation
# --------------------------------------------------

permutation_path = (
    base / "coactivation" / "coactivation_permutation.json"
)

print("\n=== EXISTING CO-ACTIVATION NULL ===")

if permutation_path.exists():

    payload = json.loads(
        permutation_path.read_text(encoding="utf-8")
    )

    print("Root type:", type(payload).__name__)

    if isinstance(payload, dict):

        for key, value in payload.items():

            if isinstance(value, (str, int, float, bool)):
                print(f"{key}: {value}")

            elif isinstance(value, dict):
                print(
                    f"{key}: nested keys = "
                    f"{list(value.keys())[:12]}"
                )

            elif isinstance(value, list):
                print(
                    f"{key}: list length = {len(value)}"
                )

else:
    print("No saved permutation JSON found.")

# --------------------------------------------------
# 3. Locate the actual matching implementation
# --------------------------------------------------

print("\n=== MATCHING SOURCE DEFINITIONS ===")

patterns = [
    "match_ab",
    "mutual_mask",
    "best_cosine",
]

source_hits = []

for directory in [ROOT / "src", ROOT / "scripts"]:

    if not directory.exists():
        continue

    for path in directory.rglob("*.py"):

        lines = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()

        matching = [
            i for i, line in enumerate(lines)
            if any(token in line for token in patterns)
        ]

        if matching:
            source_hits.append((path, lines, matching))

print("Source files found:", len(source_hits))

for path, lines, hits in source_hits[:5]:

    print(f"\nFILE: {path.relative_to(ROOT)}")

    selected = set()

    for i in hits[:5]:
        selected.update(
            range(
                max(0, i - 3),
                min(len(lines), i + 5)
            )
        )

    for i in sorted(selected):
        print(f"{i + 1:4d}: {lines[i]}")

print("\nRead-only matching audit completed.")

=== D.2.1. MATCHING AND NULL-MODEL AUDIT ===

=== SAVED MATCHING SUMMARY ===
                      pair  mean_cosine  median_cosine  mutual_pairs  mutual_fraction
    decoder_0_to_decoder_1       0.2749         0.2407          1255           0.4085
    decoder_1_to_decoder_2       0.2953         0.2552          1374           0.4473
    decoder_2_to_decoder_3       0.3227         0.2690          1442           0.4694
    encoder_0_to_encoder_1       0.4331         0.4103           810           0.2637
    encoder_1_to_encoder_2       0.5150         0.4927           904           0.2943
    encoder_2_to_encoder_3       0.5673         0.5538           928           0.3021
processor_0_to_processor_1       0.6158         0.6198          1059           0.3447
processor_1_to_processor_2       0.6200         0.6251          1100           0.3581
processor_2_to_processor_3       0.6212         0.6280          1132           0.3685
processor_3_to_processor_4       0.6247         0.6392         

### D.2.2. Recovery of Matching and Permutation Provenance

The previous inventory identified saved dictionary-matching
artifacts, but the targeted source-code search did not locate
their implementation.

The expected co-activation permutation JSON was also not
found at the constructed path.

This audit searches the existing results recursively and
inspects code cells in notebook 05 for the actual matching
procedure, similarity definitions, and null-model implementation.

The goal is to recover the exact method before introducing
new selection-aware statistical tests.

All operations are read-only.

In [29]:
import json
from pathlib import Path

print("=== D.2.2. PROVENANCE RECOVERY ===")

results_dir = ROOT / "results"
cross_dir = results_dir / "cross_layer_independent"

# --------------------------------------------------
# 1. Locate actual permutation files
# --------------------------------------------------

expected = (
    cross_dir
    / "coactivation"
    / "coactivation_permutation.json"
)

print("\n=== PERMUTATION FILE LOCATION ===")
print("Expected path:", expected)
print("Expected path exists:", expected.is_file())

candidates = sorted(
    p for p in results_dir.rglob("*.json")
    if "permut" in p.name.lower()
    and (
        "coactiv" in str(p).lower()
        or "cross_layer" in str(p).lower()
    )
)

print("Matching JSON files:", len(candidates))

for p in candidates[:15]:
    print(
        p.relative_to(ROOT),
        "| bytes:", p.stat().st_size
    )

if candidates:
    preferred = next(
        (
            p for p in candidates
            if p.name.lower() == "coactivation_permutation.json"
        ),
        candidates[0]
    )

    payload = json.loads(
        preferred.read_text(encoding="utf-8")
    )

    print("\n=== PERMUTATION JSON STRUCTURE ===")
    print("File:", preferred.relative_to(ROOT))
    print("Root type:", type(payload).__name__)

    if isinstance(payload, dict):
        print("Top-level keys:", list(payload.keys()))

        for key, value in payload.items():
            if isinstance(value, (str, int, float, bool)):
                print(f"{key}: {value}")
            elif isinstance(value, dict):
                print(
                    f"{key}: dictionary with keys "
                    f"{list(value.keys())[:12]}"
                )
            elif isinstance(value, list):
                print(f"{key}: list length {len(value)}")

# --------------------------------------------------
# 2. Locate matching code inside notebook 05
# --------------------------------------------------

notebook_path = (
    ROOT / "notebooks" / "05_cross_layer_analysis.ipynb"
)

print("\n=== NOTEBOOK 05 MATCHING IMPLEMENTATION ===")
print("Notebook exists:", notebook_path.is_file())

if notebook_path.is_file():

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    tokens = [
        "match_ab",
        "mutual_mask",
        "best_cosine",
        "dictionary_matches",
        "coactivation_permutation",
        "argmax",
    ]

    matches = []

    for cell_index, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        hits = [
            i for i, line in enumerate(lines)
            if any(
                token in line.lower()
                for token in tokens
            )
        ]

        if hits:
            matches.append((cell_index, lines, hits))

    print("Matching code cells:", len(matches))

    for cell_index, lines, hits in matches[:5]:

        print(f"\nNotebook cell index: {cell_index}")
        print("Total lines:", len(lines))

        selected = set()

        for i in hits[:4]:
            selected.update(
                range(
                    max(0, i - 3),
                    min(len(lines), i + 5)
                )
            )

        for i in sorted(selected)[:35]:
            print(f"{i + 1:4d}: {lines[i][:180]}")

# --------------------------------------------------
# 3. Broader implementation search
# --------------------------------------------------

print("\n=== SOURCE CODE SEARCH ===")

source_tokens = [
    "cosine",
    "nearest",
    "mutual",
    "dictionary_correspondence",
    "cross_layer",
    "coactivation",
]

found = []

for directory in [ROOT / "src", ROOT / "scripts"]:

    if not directory.exists():
        continue

    for path in directory.rglob("*.py"):

        source = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).lower()

        count = sum(
            source.count(token)
            for token in source_tokens
        )

        if count:
            found.append((count, path))

found.sort(key=lambda x: x[0], reverse=True)

print("Candidate Python files:", len(found))

for count, path in found[:12]:
    print(
        path.relative_to(ROOT),
        "| keyword occurrences:", count
    )

print("\nRead-only provenance recovery completed.")

=== D.2.2. PROVENANCE RECOVERY ===

=== PERMUTATION FILE LOCATION ===
Expected path: D:\bio_paper\pie-sae\results\cross_layer_independent\coactivation\coactivation_permutation.json
Expected path exists: False
Matching JSON files: 1
results\cross_layer_independent\coactivation\coactivation_permutation_200.json | bytes: 3985

=== PERMUTATION JSON STRUCTURE ===
File: results\cross_layer_independent\coactivation\coactivation_permutation_200.json
Root type: dict
Top-level keys: ['n_permutations', 'positions_per_condition', 'split', 'seed', 'results']
n_permutations: 200
positions_per_condition: 128
split: validation
seed: 42
results: list length 11

=== NOTEBOOK 05 MATCHING IMPLEMENTATION ===
Notebook exists: True
Matching code cells: 7

Notebook cell index: 4
Total lines: 95
  15: def compare_dictionary_pair(layer_a: str, layer_b: str) -> dict:
  16:     A, B = dictionary_vectors(layer_a), dictionary_vectors(layer_b)
  17:     sim = A @ B.T  # signed directional cosine; both SAEs use nonne

### D.2.3. Dictionary Matching and Co-activation Controls

The dictionary-matching implementation was located in
notebook 05, while the existing co-activation permutation
results were recovered from the saved analysis artifacts.

The matching procedure is examined to determine the
representation vectors, similarity normalization,
nearest-neighbor selection, and mutual matching criterion.

The co-activation controls are inspected separately
because their null hypotheses differ from those of
dictionary matching.

These details establish the statistical design for
subsequent cross-layer validation.

In [30]:
import ast
import json
import re

print("=== D.2.3. MATCHING IMPLEMENTATION ===")

# --------------------------------------------------
# 1. Read the original matching function
# --------------------------------------------------

notebook_path = (
    ROOT / "notebooks/05_cross_layer_analysis.ipynb"
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

cell = notebook["cells"][4]

assert cell["cell_type"] == "code"

source = "".join(cell["source"])
lines = source.splitlines()

tree = ast.parse(source)

matching_functions = [
    node for node in ast.walk(tree)
    if isinstance(node, ast.FunctionDef)
    and node.name == "compare_dictionary_pair"
]

assert len(matching_functions) == 1

function = matching_functions[0]

print("Function:", function.name)
print(
    "Source lines:",
    function.lineno,
    "to",
    function.end_lineno
)

# --------------------------------------------------
# 2. Print essential matching operations
# --------------------------------------------------

pattern = re.compile(
    r"def compare_dictionary_pair|"
    r"weight|dictionary|cosine|"
    r"normaliz|norm|argmax|"
    r"mutual|match|similar|"
    r"einsum|savez|transpose|"
    r"\.T\b|@",
    re.IGNORECASE
)

selected = set()

for i in range(
    function.lineno - 1,
    function.end_lineno
):
    if pattern.search(lines[i]):
        selected.update(
            range(
                max(function.lineno - 1, i - 1),
                min(function.end_lineno, i + 2)
            )
        )

print("\n=== MATCHING OPERATIONS ===")

previous = -2

for i in sorted(selected):
    if i > previous + 1:
        print("    ...")

    print(f"{i + 1:3d}: {lines[i]}")
    previous = i

# --------------------------------------------------
# 3. Inspect actual co-activation null structure
# --------------------------------------------------

permutation_path = (
    ROOT
    / "results/cross_layer_independent"
    / "coactivation"
    / "coactivation_permutation_200.json"
)

payload = json.loads(
    permutation_path.read_text(encoding="utf-8")
)

print("\n=== CO-ACTIVATION CONTROL ===")

for key in [
    "n_permutations",
    "positions_per_condition",
    "split",
    "seed",
]:
    print(f"{key}: {payload.get(key)}")

results = payload["results"]

print("Result records:", len(results))

print("\nFirst two result records:")

for i, record in enumerate(results[:2]):
    print(f"\nRecord {i}:")
    print(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        )[:1800]
    )

print("\nRead-only method inspection completed.")

=== D.2.3. MATCHING IMPLEMENTATION ===
Function: compare_dictionary_pair
Source lines: 15 to 37

=== MATCHING OPERATIONS ===
    ...
 15: def compare_dictionary_pair(layer_a: str, layer_b: str) -> dict:
 16:     A, B = dictionary_vectors(layer_a), dictionary_vectors(layer_b)
 17:     sim = A @ B.T  # signed directional cosine; both SAEs use nonnegative feature activations
 18:     best_cos, match_ab = sim.max(dim=1)
 19:     match_ba = sim.argmax(dim=0)
 20:     ids = torch.arange(A.shape[0], device=DEVICE)
 21:     mutual = match_ba[match_ab] == ids
 22:     same_index = (match_ab == ids)
 23:     result = {
    ...
 25:         "layer_b": layer_b,
 26:         "mean_best_cosine": float(best_cos.mean().item()),
 27:         "median_best_cosine": float(best_cos.median().item()),
 28:         "p95_best_cosine": float(torch.quantile(best_cos, 0.95).item()),
 29:         "mutual_nn_pairs": int(mutual.sum().item()),
 30:         "same_index_best_matches": int(same_index.sum().item()),
 31:

### D.3. Geometry-Preserving Dictionary Alignment Control

The matching procedure selects the highest signed cosine
similarity for each SAE feature and identifies mutual
nearest-neighbor pairs.

The analysis first reconstructs decoder.0 and decoder.1
dictionary vectors from their existing SAE checkpoints.

The reconstructed nearest-neighbor indices, cosine
similarities, and mutual matches must agree with the
saved results from notebook 05.

A geometry-preserving control is then performed using
random signed coordinate permutations of the second
dictionary.

Each transformation is orthogonal and preserves all
within-dictionary pairwise cosine similarities, while
altering the relative orientation of the two dictionaries.

For each transformed dictionary, the full nearest-neighbor
matching procedure is repeated.

Two summary statistics are evaluated:

- Mean best-match cosine similarity.
- Fraction of mutual nearest-neighbor matches.

The test evaluates dictionary-level geometric alignment,
including the selection of nearest neighbors.

The null is a restricted family of orthogonal transformations,
not a complete generative model of independently trained
SAE dictionaries.

The resulting empirical probabilities are exploratory
geometric-control diagnostics, not causal evidence
or confirmatory feature-level p-values.

In [31]:
import numpy as np
import torch
import torch.nn.functional as F

print("=== D.3. GEOMETRIC DICTIONARY ALIGNMENT CONTROL ===")

D3_LAYER_A = "decoder.0"
D3_LAYER_B = "decoder.1"
D3_N_NULL = 99
D3_SEED = 20261113

device_d3 = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

# --------------------------------------------------
# 1. Reconstruct SAE dictionary vectors
# --------------------------------------------------

def load_dictionary_vectors(layer_name):

    entry = registry[layer_name]
    path = ROOT / entry["checkpoint"]

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    state = checkpoint["model_state_dict"]

    candidates = []

    for name, tensor in state.items():

        if not isinstance(tensor, torch.Tensor):
            continue

        if tensor.ndim != 2:
            continue

        if tuple(tensor.shape) not in [
            (3072, 768),
            (768, 3072),
        ]:
            continue

        name_lower = name.lower()

        if any(token in name_lower for token in [
            "decoder", "w_dec", "wdec"
        ]):
            candidates.append((name, tensor))

    if len(candidates) != 1:
        raise ValueError(
            f"{layer_name}: expected one decoder weight "
            f"matrix, found {[name for name, _ in candidates]}"
        )

    weight_name, weight = candidates[0]

    vectors = weight.detach().float()

    if vectors.shape == (768, 3072):
        vectors = vectors.T

    assert vectors.shape == (3072, 768)

    vectors = F.normalize(vectors, dim=1)

    print(
        f"{layer_name}: {weight_name}, "
        f"shape={tuple(vectors.shape)}"
    )

    return vectors.to(device_d3)


A_d3 = load_dictionary_vectors(D3_LAYER_A)
B_d3 = load_dictionary_vectors(D3_LAYER_B)

# --------------------------------------------------
# 2. Reproduce the original matching procedure
# --------------------------------------------------

@torch.inference_mode()
def matching_statistics(A, B):

    similarity = A @ B.T

    best_cos, match_ab = similarity.max(dim=1)
    match_ba = similarity.argmax(dim=0)

    indices = torch.arange(
        A.shape[0],
        device=A.device
    )

    mutual = match_ba[match_ab] == indices

    return {
        "mean_cosine": float(best_cos.mean().item()),
        "mutual_fraction": float(mutual.float().mean().item()),
        "mutual_count": int(mutual.sum().item()),
        "match_ab": match_ab.cpu().numpy(),
        "best_cosine": best_cos.cpu().numpy(),
        "mutual_mask": mutual.cpu().numpy(),
    }


observed_d3 = matching_statistics(A_d3, B_d3)

saved_path = (
    ROOT
    / "results/cross_layer_independent"
    / "dictionary_matches"
    / "decoder_0_to_decoder_1.npz"
)

with np.load(saved_path, allow_pickle=False) as saved:

    indices_match = np.array_equal(
        observed_d3["match_ab"],
        saved["match_ab"]
    )

    cosine_match = np.allclose(
        observed_d3["best_cosine"],
        saved["best_cosine"],
        rtol=1e-5,
        atol=1e-5
    )

    mutual_match = np.array_equal(
        observed_d3["mutual_mask"],
        saved["mutual_mask"]
    )

print("\n=== ORIGINAL MATCHING REPRODUCTION ===")

print("Best-match indices:", indices_match)
print("Cosine similarities:", cosine_match)
print("Mutual matches:", mutual_match)

print(
    "Observed mean cosine:",
    f"{observed_d3['mean_cosine']:.6f}"
)

print(
    "Observed mutual pairs:",
    observed_d3["mutual_count"]
)

# Do not run a null model on unverified dictionary vectors.
if not (indices_match and cosine_match and mutual_match):
    raise RuntimeError(
        "Dictionary extraction does not reproduce "
        "notebook 05. Check the original dictionary_vectors "
        "implementation before proceeding."
    )

# --------------------------------------------------
# 3. Signed-coordinate-permutation controls
# --------------------------------------------------

generator_d3 = torch.Generator(device="cpu")
generator_d3.manual_seed(D3_SEED)

null_cosine_d3 = np.empty(D3_N_NULL, dtype=np.float64)
null_mutual_d3 = np.empty(D3_N_NULL, dtype=np.float64)

dimension = B_d3.shape[1]

for b in range(D3_N_NULL):

    permutation = torch.randperm(
        dimension,
        generator=generator_d3
    ).to(device_d3)

    signs = (
        2 * torch.randint(
            0, 2,
            (dimension,),
            generator=generator_d3
        ) - 1
    ).to(
        device=device_d3,
        dtype=B_d3.dtype
    )

    transformed_B = B_d3[:, permutation] * signs

    result = matching_statistics(
        A_d3,
        transformed_B
    )

    null_cosine_d3[b] = result["mean_cosine"]
    null_mutual_d3[b] = result["mutual_fraction"]

# --------------------------------------------------
# 4. Empirical comparison
# --------------------------------------------------

def empirical_p_d3(observed, null_values):

    return (
        1 + np.count_nonzero(null_values >= observed)
    ) / (len(null_values) + 1)


print("\n=== GEOMETRIC CONTROL RESULTS ===")

print("Null repetitions:", D3_N_NULL)

for label, observed, null_values in [
    (
        "Mean best-match cosine",
        observed_d3["mean_cosine"],
        null_cosine_d3
    ),
    (
        "Mutual-match fraction",
        observed_d3["mutual_fraction"],
        null_mutual_d3
    ),
]:

    print(f"\n{label}")
    print(f"Observed: {observed:.6f}")
    print(f"Null mean: {null_values.mean():.6f}")
    print(
        "Null 95th percentile:",
        f"{np.percentile(null_values, 95):.6f}"
    )
    print(
        "Empirical upper-tail p:",
        f"{empirical_p_d3(observed, null_values):.6f}"
    )

print("\nRead-only geometric control completed.")

=== D.3. GEOMETRIC DICTIONARY ALIGNMENT CONTROL ===
decoder.0: W_dec.weight, shape=(3072, 768)
decoder.1: W_dec.weight, shape=(3072, 768)

=== ORIGINAL MATCHING REPRODUCTION ===
Best-match indices: True
Cosine similarities: True
Mutual matches: True
Observed mean cosine: 0.274934
Observed mutual pairs: 1255

=== GEOMETRIC CONTROL RESULTS ===
Null repetitions: 99

Mean best-match cosine
Observed: 0.274934
Null mean: 0.127038
Null 95th percentile: 0.127521
Empirical upper-tail p: 0.010000

Mutual-match fraction
Observed: 0.408529
Null mean: 0.465113
Null 95th percentile: 0.475684
Empirical upper-tail p: 1.000000

Read-only geometric control completed.


### D.3.1. Geometry-Preserving Controls Across Layer Transitions

The initial decoder.0-to-decoder.1 experiment showed
higher best-match cosine similarity than the signed
coordinate-permutation null.

However, the observed mutual-nearest-neighbor fraction
was lower than the corresponding null mean.

These results indicate that dictionary similarity
and mutual matching capture different properties
of cross-layer geometry.

The present analysis extends the same control to
all eleven saved adjacent within-component transitions.

For each transition, the analysis measures:

1. Mean best-match signed cosine similarity.
2. Fraction of mutual nearest-neighbor pairs.
3. Fraction of unique target features selected
   as nearest neighbors.

The original matching procedure is reproduced
and checked against the saved notebook 05 results.

Each null replicate independently applies a random
signed coordinate permutation to the second dictionary,
then repeats the complete matching procedure.

Both upper- and lower-tail empirical probabilities
are reported as exploratory diagnostics.

The null preserves within-dictionary cosine geometry,
but does not represent independently retrained SAEs
or a complete biological null model.

Comparisons across transitions are descriptive.
No confirmatory multiple-testing claims are made.

In [32]:
import re
import numpy as np
import pandas as pd
import torch

print("=== D.3.1. ALL-TRANSITION GEOMETRIC CONTROLS ===")

D31_N_NULL = 199
D31_SEED = 20261113

match_dir_d31 = (
    ROOT
    / "results/cross_layer_independent"
    / "dictionary_matches"
)

match_files_d31 = sorted(match_dir_d31.glob("*.npz"))

assert len(match_files_d31) == 11

generator_d31 = torch.Generator(device="cpu")
generator_d31.manual_seed(D31_SEED)

# --------------------------------------------------
# 1. Recover the eleven layer pairs
# --------------------------------------------------

pattern_d31 = re.compile(
    r"^(encoder|processor|decoder)_(\d+)"
    r"_to_(encoder|processor|decoder)_(\d+)$"
)

pairs_d31 = []

for path in match_files_d31:

    match = pattern_d31.fullmatch(path.stem)

    if match is None:
        raise ValueError(
            f"Unexpected matching filename: {path.name}"
        )

    kind_a, index_a, kind_b, index_b = match.groups()

    layer_a = f"{kind_a}.{index_a}"
    layer_b = f"{kind_b}.{index_b}"

    pairs_d31.append((path, layer_a, layer_b))

# --------------------------------------------------
# 2. Load each pretrained dictionary only once
# --------------------------------------------------

layer_names_d31 = sorted({
    name
    for _, layer_a, layer_b in pairs_d31
    for name in (layer_a, layer_b)
})

vectors_d31 = {
    name: load_dictionary_vectors(name)
    for name in layer_names_d31
}

device_d31 = next(iter(vectors_d31.values())).device

# --------------------------------------------------
# 3. Matching statistics
# --------------------------------------------------

@torch.inference_mode()
def calculate_d31(A, B):

    similarity = A @ B.T

    best_cos, match_ab = similarity.max(dim=1)
    match_ba = similarity.argmax(dim=0)

    mutual = (
        match_ba[match_ab]
        == torch.arange(A.shape[0], device=A.device)
    )

    return (
        float(best_cos.mean().item()),
        float(mutual.float().mean().item()),
        float(torch.unique(match_ab).numel() / A.shape[0]),
        match_ab.cpu().numpy(),
        best_cos.cpu().numpy(),
        mutual.cpu().numpy(),
    )


def empirical_tails_d31(observed, null_values):

    upper = (
        1 + np.count_nonzero(null_values >= observed)
    ) / (len(null_values) + 1)

    lower = (
        1 + np.count_nonzero(null_values <= observed)
    ) / (len(null_values) + 1)

    return float(upper), float(lower)


# --------------------------------------------------
# 4. Reproduce and evaluate every transition
# --------------------------------------------------

D31_records = []
D31_nulls = {}

for path, layer_a, layer_b in pairs_d31:

    A = vectors_d31[layer_a]
    B = vectors_d31[layer_b]

    observed = calculate_d31(A, B)

    with np.load(path, allow_pickle=False) as saved:

        assert np.array_equal(
            observed[3], saved["match_ab"]
        ), f"Match-index mismatch: {path.name}"

        assert np.allclose(
            observed[4],
            saved["best_cosine"],
            atol=1e-5,
            rtol=1e-5
        ), f"Cosine mismatch: {path.name}"

        assert np.array_equal(
            observed[5],
            saved["mutual_mask"]
        ), f"MNN mismatch: {path.name}"

    null_statistics = np.empty(
        (D31_N_NULL, 3),
        dtype=np.float64
    )

    dimension = B.shape[1]

    for b in range(D31_N_NULL):

        permutation = torch.randperm(
            dimension,
            generator=generator_d31
        ).to(device_d31)

        signs = (
            2 * torch.randint(
                0, 2,
                (dimension,),
                generator=generator_d31
            ) - 1
        ).to(
            device=device_d31,
            dtype=B.dtype
        )

        transformed_B = B[:, permutation] * signs

        result = calculate_d31(A, transformed_B)

        null_statistics[b] = result[:3]

    D31_nulls[path.stem] = null_statistics

    row = {"pair": path.stem}

    labels = ["cosine", "mnn", "unique_target"]

    for j, label in enumerate(labels):

        upper, lower = empirical_tails_d31(
            observed[j],
            null_statistics[:, j]
        )

        row[f"observed_{label}"] = observed[j]
        row[f"null_mean_{label}"] = (
            null_statistics[:, j].mean()
        )
        row[f"upper_p_{label}"] = upper
        row[f"lower_p_{label}"] = lower

    D31_records.append(row)

    print(
        f"{path.stem}: "
        f"cosine={observed[0]:.4f}, "
        f"MNN={observed[1]:.4f}, "
        f"unique_target={observed[2]:.4f}"
    )

# --------------------------------------------------
# 5. Summary
# --------------------------------------------------

D31_results = pd.DataFrame(D31_records)

assert len(D31_results) == 11
assert np.isfinite(
    D31_results.select_dtypes(include="number")
    .to_numpy()
).all()

print("\n=== COSINE CONTROL SUMMARY ===")

print(
    D31_results[
        [
            "pair",
            "observed_cosine",
            "null_mean_cosine",
            "upper_p_cosine",
        ]
    ].round(5).to_string(index=False)
)

print("\n=== MNN CONTROL SUMMARY ===")

print(
    D31_results[
        [
            "pair",
            "observed_mnn",
            "null_mean_mnn",
            "upper_p_mnn",
            "lower_p_mnn",
        ]
    ].round(5).to_string(index=False)
)

print("\n=== UNIQUE-TARGET SUMMARY ===")

print(
    D31_results[
        [
            "pair",
            "observed_unique_target",
            "null_mean_unique_target",
        ]
    ].round(5).to_string(index=False)
)

print("\nRead-only all-transition analysis completed.")

=== D.3.1. ALL-TRANSITION GEOMETRIC CONTROLS ===
decoder.0: W_dec.weight, shape=(3072, 768)
decoder.1: W_dec.weight, shape=(3072, 768)
decoder.2: W_dec.weight, shape=(3072, 768)
decoder.3: W_dec.weight, shape=(3072, 768)
encoder.0: W_dec.weight, shape=(3072, 768)
encoder.1: W_dec.weight, shape=(3072, 768)
encoder.2: W_dec.weight, shape=(3072, 768)
encoder.3: W_dec.weight, shape=(3072, 768)
processor.0: W_dec.weight, shape=(3072, 768)
processor.1: W_dec.weight, shape=(3072, 768)
processor.2: W_dec.weight, shape=(3072, 768)
processor.3: W_dec.weight, shape=(3072, 768)
processor.4: W_dec.weight, shape=(3072, 768)
processor.5: W_dec.weight, shape=(3072, 768)
decoder_0_to_decoder_1: cosine=0.2749, MNN=0.4085, unique_target=0.5876
decoder_1_to_decoder_2: cosine=0.2953, MNN=0.4473, unique_target=0.6149
decoder_2_to_decoder_3: cosine=0.3227, MNN=0.4694, unique_target=0.6156
encoder_0_to_encoder_1: cosine=0.4331, MNN=0.2637, unique_target=0.3991
encoder_1_to_encoder_2: cosine=0.5150, MNN=0.2943

### D.3.2. Cross-Layer Geometric Control Summary

All eleven saved adjacent-layer dictionary matches were
successfully reproduced from the pretrained SAE weights.

For each transition, 199 random signed-coordinate
permutations were used to evaluate the relative
orientation of the two dictionaries.

The analysis distinguishes best-match cosine similarity,
mutual-nearest-neighbor fraction, and target-feature
coverage.

Upper-tail and lower-tail empirical probabilities are
reported separately where relevant.

These comparisons are exploratory. The geometric null
preserves within-dictionary cosine relationships but
does not reproduce SAE training or the statistical
dependencies of the original representations.

The finite permutation count limits p-value resolution.
No family-wise significance claims across the eleven
transitions are made at this stage.

In [33]:
import pandas as pd

print("=== D.3.2. CROSS-LAYER GEOMETRIC SUMMARY ===")

summary_d32 = D31_results[
    [
        "pair",
        "observed_cosine",
        "null_mean_cosine",
        "upper_p_cosine",
        "observed_mnn",
        "null_mean_mnn",
        "upper_p_mnn",
        "lower_p_mnn",
    ]
].copy()

summary_d32.columns = [
    "pair",
    "cos_obs",
    "cos_null",
    "cos_p_upper",
    "mnn_obs",
    "mnn_null",
    "mnn_p_upper",
    "mnn_p_lower",
]

print(summary_d32.round(4).to_string(index=False))

print("\n=== DESCRIPTIVE COUNTS ===")

print(
    "Transitions with cosine above null mean:",
    int(
        (summary_d32["cos_obs"] >
         summary_d32["cos_null"]).sum()
    ),
    "/11"
)

print(
    "Transitions with MNN below null mean:",
    int(
        (summary_d32["mnn_obs"] <
         summary_d32["mnn_null"]).sum()
    ),
    "/11"
)

print(
    "Cosine upper-tail p <= 0.05:",
    int((summary_d32["cos_p_upper"] <= 0.05).sum())
)

print(
    "MNN lower-tail p <= 0.05:",
    int((summary_d32["mnn_p_lower"] <= 0.05).sum())
)

print("\nExploratory summary completed.")

=== D.3.2. CROSS-LAYER GEOMETRIC SUMMARY ===
                      pair  cos_obs  cos_null  cos_p_upper  mnn_obs  mnn_null  mnn_p_upper  mnn_p_lower
    decoder_0_to_decoder_1   0.2749    0.1270        0.005   0.4085    0.4642        1.000        0.005
    decoder_1_to_decoder_2   0.2953    0.1270        0.005   0.4473    0.4638        1.000        0.005
    decoder_2_to_decoder_3   0.3227    0.1270        0.005   0.4694    0.4649        0.275        0.745
    encoder_0_to_encoder_1   0.4331    0.1271        0.005   0.2637    0.4707        1.000        0.005
    encoder_1_to_encoder_2   0.5150    0.1270        0.005   0.2943    0.4651        1.000        0.005
    encoder_2_to_encoder_3   0.5673    0.1267        0.005   0.3021    0.4557        1.000        0.005
processor_0_to_processor_1   0.6158    0.1266        0.005   0.3447    0.4468        1.000        0.005
processor_1_to_processor_2   0.6200    0.1267        0.005   0.3581    0.4468        1.000        0.005
processor_2_to_proc

### D.3.3. Geometric Alignment Results

All eleven adjacent-layer dictionary matches were
reproduced exactly from the pretrained SAE weights.

For each transition, 199 random signed-coordinate
permutations were evaluated, with nearest-neighbor
matching recomputed in every replicate.

The observed mean best-match cosine similarity exceeded
the corresponding null mean for all eleven transitions.
All eleven upper-tail empirical p-values reached the
minimum available value of 0.005.

In contrast, the mutual-nearest-neighbor fraction was
below the null mean for ten of eleven transitions.
These ten transitions had lower-tail empirical
p-values of 0.005.

The decoder.2-to-decoder.3 transition did not show
the same reduction in mutual matching.

The contrasting behavior of cosine similarity and
mutual matching suggests that strong geometric
alignment does not necessarily correspond to
one-to-one feature alignment.

The results are conditional on the signed-coordinate
permutation null and are not evidence of computational
causality or equivalent biological function.

With 199 permutations, the p-value resolution is
insufficient for individual Holm-adjusted significance
claims across all eleven transitions at FWER 0.05.

Independent replication and alternative geometric
null models remain outside the scope of this analysis.

### D.4. Existing Cross-Layer Co-activation Controls

Dictionary geometry and feature co-activation address
different questions.

Dictionary matching measures similarity between SAE
decoder directions. Co-activation measures relationships
between feature firing events on observed representations.

The previous cross-layer analysis includes a saved
200-permutation validation experiment for eleven
adjacent-layer transitions.

Before interpreting its empirical p-values, the
original permutation procedure is inspected to identify:

- The observed co-activation statistic.
- The unit and scope of permutation.
- Whether feature matching was fixed or repeated.
- The exchangeability assumptions of the null model.
- The statistical comparisons covered by the results.

The saved results are summarized without rerunning
the original experiment.

These controls do not constitute causal interventions.

In [34]:
import json
import pandas as pd
import numpy as np

print("=== D.4. SAVED CO-ACTIVATION VALIDATION ===")

# --------------------------------------------------
# 1. Load the actual saved permutation results
# --------------------------------------------------

path_d4 = (
    ROOT
    / "results/cross_layer_independent"
    / "coactivation/coactivation_permutation_200.json"
)

payload_d4 = json.loads(
    path_d4.read_text(encoding="utf-8")
)

assert isinstance(payload_d4, dict)

records_d4 = payload_d4["results"]

assert isinstance(records_d4, list)
assert len(records_d4) == 11

print("\n=== EXPERIMENT METADATA ===")

for key in [
    "n_permutations",
    "positions_per_condition",
    "split",
    "seed",
]:
    print(f"{key}: {payload_d4.get(key)}")

# --------------------------------------------------
# 2. Inspect the saved statistics compactly
# --------------------------------------------------

print("\n=== RESULT SCHEMA ===")

first_d4 = records_d4[0]

assert isinstance(first_d4, dict)

print("First record keys:", list(first_d4.keys()))

print("\nFirst result record:")
print(
    json.dumps(
        first_d4,
        indent=2,
        ensure_ascii=False
    )[:2500]
)

# Flatten nested result dictionaries, if present.
flat_d4 = pd.json_normalize(
    records_d4,
    sep="."
)

print("\n=== AVAILABLE RESULT FIELDS ===")
print(flat_d4.columns.tolist())

numeric_d4 = flat_d4.select_dtypes(
    include=[np.number]
)

print("\n=== NUMERIC RESULTS ===")

if not numeric_d4.empty:

    label_candidates = [
        c for c in flat_d4.columns
        if any(
            token in c.lower()
            for token in ["pair", "layer", "transition"]
        )
        and c not in numeric_d4.columns
    ]

    result_columns = (
        label_candidates[:2]
        + numeric_d4.columns.tolist()[:8]
    )

    print(
        flat_d4[result_columns]
        .round(5)
        .to_string(index=False)
    )

else:
    print("No directly accessible scalar numeric columns.")

# --------------------------------------------------
# 3. Inspect the original notebook's null procedure
# --------------------------------------------------

notebook_path_d4 = (
    ROOT / "notebooks/05_cross_layer_analysis.ipynb"
)

notebook_d4 = json.loads(
    notebook_path_d4.read_text(encoding="utf-8")
)

print("\n=== ORIGINAL PERMUTATION IMPLEMENTATION ===")

search_tokens_d4 = [
    "permutation",
    "permutations",
    "rng.permutation",
    "shuffle",
    "shuffled",
    "coactivation_permutation",
    "empirical_p",
]

matched_cells_d4 = []

for cell_index, cell in enumerate(
    notebook_d4["cells"]
):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(
        cell.get("source", [])
    ).splitlines()

    hits = [
        i for i, line in enumerate(lines)
        if any(
            token in line.lower()
            for token in search_tokens_d4
        )
    ]

    if hits:
        matched_cells_d4.append(
            (cell_index, lines, hits)
        )

print(
    "Code cells with permutation references:",
    len(matched_cells_d4)
)

# Print only targeted excerpts to avoid
# excessively large notebook outputs.

for cell_index, lines, hits in matched_cells_d4[:4]:

    print(f"\nNotebook 05, cell index {cell_index}")

    selected_lines = set()

    for hit in hits[:5]:
        selected_lines.update(
            range(
                max(0, hit - 2),
                min(len(lines), hit + 3)
            )
        )

    for i in sorted(selected_lines)[:25]:
        print(f"{i + 1:4d}: {lines[i][:160]}")

print("\nRead-only co-activation audit completed.")

=== D.4. SAVED CO-ACTIVATION VALIDATION ===

=== EXPERIMENT METADATA ===
n_permutations: 200
positions_per_condition: 128
split: validation
seed: 42

=== RESULT SCHEMA ===
First record keys: ['layer_a', 'layer_b', 'mutual_pairs', 'sampled_observations', 'observed', 'null_mean', 'null_std', 'null_p95', 'empirical_p']

First result record:
{
  "layer_a": "encoder.0",
  "layer_b": "encoder.1",
  "mutual_pairs": 810,
  "sampled_observations": 4096,
  "observed": 0.4304211530598337,
  "null_mean": 0.009432329289900503,
  "null_std": 0.0016807603275200364,
  "null_p95": 0.01196946981054927,
  "empirical_p": 0.004975124378109453
}

=== AVAILABLE RESULT FIELDS ===
['layer_a', 'layer_b', 'mutual_pairs', 'sampled_observations', 'observed', 'null_mean', 'null_std', 'null_p95', 'empirical_p']

=== NUMERIC RESULTS ===
    layer_a     layer_b  mutual_pairs  sampled_observations  observed  null_mean  null_std  null_p95  empirical_p
  encoder.0   encoder.1           810                  4096   0.43042

### D.4.1. Co-activation Null-Model Verification

The saved validation experiment contains 200 permutation
replicates for each of eleven adjacent-layer transitions.

The observed co-activation statistic and null distribution
are summarized across all transitions.

The original implementation is inspected at the point
where null samples are generated and evaluated.

Particular attention is paid to the resampling unit,
the preservation of experimental-condition structure,
and whether dictionary matching remains fixed.

Empirical p-values are interpreted conditionally on
the implemented null model. The 4,096 sampled positions
must not be treated as independent perturbation experiments.

No original computations are repeated or modified.

In [35]:
import json
import numpy as np
import pandas as pd

print("=== D.4.1. CO-ACTIVATION NULL VERIFICATION ===")

# --------------------------------------------------
# 1. Summarize all eleven saved transitions
# --------------------------------------------------

path_d41 = (
    ROOT
    / "results/cross_layer_independent"
    / "coactivation/coactivation_permutation_200.json"
)

data_d41 = json.loads(
    path_d41.read_text(encoding="utf-8")
)

assert data_d41["n_permutations"] == 200
assert data_d41["split"] == "validation"
assert len(data_d41["results"]) == 11

rows_d41 = []

for record in data_d41["results"]:

    observed = float(record["observed"])
    null_mean = float(record["null_mean"])

    rows_d41.append({
        "pair": (
            record["layer_a"]
            + " -> "
            + record["layer_b"]
        ),
        "mutual_pairs": int(record["mutual_pairs"]),
        "observations": int(record["sampled_observations"]),
        "observed": observed,
        "null_mean": null_mean,
        "null_p95": float(record["null_p95"]),
        "ratio": (
            observed / null_mean
            if null_mean > 0 else np.nan
        ),
        "empirical_p": float(record["empirical_p"]),
    })

D41_summary = pd.DataFrame(rows_d41)

print("\n=== ALL TRANSITION RESULTS ===")

print(
    D41_summary.round(5).to_string(index=False)
)

print("\n=== DESCRIPTIVE CHECKS ===")

print(
    "Observed > null mean:",
    int(
        (
            D41_summary["observed"]
            > D41_summary["null_mean"]
        ).sum()
    ),
    "/11"
)

print(
    "Observed > null 95th percentile:",
    int(
        (
            D41_summary["observed"]
            > D41_summary["null_p95"]
        ).sum()
    ),
    "/11"
)

print(
    "Minimum reported empirical p:",
    D41_summary["empirical_p"].min()
)

# --------------------------------------------------
# 2. Inspect exact permutation implementation
# --------------------------------------------------

notebook_path_d41 = (
    ROOT / "notebooks/05_cross_layer_analysis.ipynb"
)

notebook_d41 = json.loads(
    notebook_path_d41.read_text(encoding="utf-8")
)

print("\n=== ORIGINAL NULL-MODEL SOURCE ===")

found_d41 = 0

for cell_index, cell in enumerate(
    notebook_d41["cells"]
):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(
        cell.get("source", [])
    ).splitlines()

    anchors = [
        i for i, line in enumerate(lines)
        if "null_values[p]" in line
    ]

    for i in anchors:

        print(
            f"\nNotebook 05, code cell {cell_index}"
        )

        start = max(0, i - 35)
        end = min(len(lines), i + 18)

        print(
            f"Source lines {start + 1}–{end}:"
        )

        for j in range(start, end):
            print(
                f"{j + 1:4d}: {lines[j][:180]}"
            )

        found_d41 += 1

print("\nNull-statistic assignments found:", found_d41)

if found_d41 == 0:
    print(
        "No exact assignment found; "
        "inspect the original permutation cell."
    )

print("\nRead-only co-activation verification completed.")

=== D.4.1. CO-ACTIVATION NULL VERIFICATION ===

=== ALL TRANSITION RESULTS ===
                      pair  mutual_pairs  observations  observed  null_mean  null_p95    ratio  empirical_p
    encoder.0 -> encoder.1           810          4096   0.43042    0.00943   0.01197 45.63254      0.00498
    encoder.1 -> encoder.2           904          4096   0.49613    0.01321   0.01707 37.54724      0.00498
    encoder.2 -> encoder.3           928          4096   0.43837    0.01260   0.01785 34.79009      0.00498
processor.0 -> processor.1          1059          4096   0.51817    0.01466   0.02082 35.33525      0.00498
processor.1 -> processor.2          1100          4096   0.51465    0.01416   0.01793 36.33618      0.00498
processor.2 -> processor.3          1132          4096   0.49471    0.01393   0.01938 35.50675      0.00498
processor.3 -> processor.4          1180          4096   0.49282    0.01365   0.01844 36.11410      0.00498
processor.4 -> processor.5          1227          4096   

### D.4.2. Co-activation Permutation Mechanism

The saved co-activation results show consistently higher
observed statistics than the permutation-null distributions
across all eleven adjacent-layer transitions.

The interpretation of these results depends on the exact
randomization mechanism.

This audit examines the source code responsible for
constructing null samples and calculating empirical
probabilities.

The analysis distinguishes fixed-match randomization,
activation shuffling, and condition-level permutations,
which test different null hypotheses.

No additional permutation experiments are performed.

In [36]:
import json

path = ROOT / "notebooks/05_cross_layer_analysis.ipynb"
notebook = json.loads(path.read_text(encoding="utf-8"))

lines = "".join(notebook["cells"][12]["source"]).splitlines()

print("=== CO-ACTIVATION NULL CONSTRUCTION ===")

for i in range(94, 123):
    print(f"{i + 1:3d}: {lines[i]}")

=== CO-ACTIVATION NULL CONSTRUCTION ===
 95:     denominator = int(A.sum())
 96:     assert denominator > 0
 97: 
 98:     observed = float(
 99:         np.count_nonzero(A & B) / denominator
100:     )
101: 
102:     null_values = np.empty(
103:         N_PERMUTATIONS,
104:         dtype=np.float64,
105:     )
106: 
107:     n_pairs = A.shape[1]
108: 
109:     for p in range(N_PERMUTATIONS):
110: 
111:         shuffled = rng.permutation(n_pairs)
112: 
113:         null_values[p] = (
114:             np.count_nonzero(A & B[:, shuffled])
115:             / denominator
116:         )
117: 
118:     empirical_p = float(
119:         (
120:             1 + np.count_nonzero(null_values >= observed)
121:         )
122:         / (N_PERMUTATIONS + 1)
123:     )


### D.4.3. Co-activation Control Results and Interpretation

The saved co-activation analysis evaluated eleven
adjacent-layer SAE dictionary transitions using
32 validation conditions and 128 sampled positions
per condition.

Each transition was assessed with 200 permutation
replicates.

The observed statistic is the fraction of first-layer
feature firing events accompanied by firing of their
geometrically matched second-layer partners.

The null model randomly reassigns second-layer feature
columns among the already selected pairs.

This procedure preserves the recorded feature
activations and their marginal firing frequencies,
while disrupting the original cross-layer pairing.

For all eleven transitions, observed co-activation
exceeded the 95th percentile of the permutation
distribution.

All eleven empirical upper-tail p-values were
0.004975, the minimum attainable with 200 permutations.

These results support elevated co-activation of
geometrically matched SAE features relative to the
specified random-pairing null.

### Statistical limitations

The matching procedure was not repeated during
permutation. Therefore, the test does not provide
full selection-aware inference for dictionary matching.

Experimental positions within the same condition
are dependent observations and must not be treated
as independent biological replicates.

The permutation null does not exclude shared
cell-context effects or other common drivers
of feature activation.

The available permutation resolution is insufficient
for individual Holm-adjusted rejections across all
eleven transitions at family-wise alpha = 0.05.

The results establish neither computational causality
nor equivalent biological function across layers.

Additional condition-aware controls and causal
interventions remain subjects for subsequent work.

### D.5. Biological Correspondence of Matched SAE Features

The preceding analyses evaluated dictionary geometry
and feature co-activation across eleven adjacent-layer
transitions.

This section examines whether geometrically matched
SAE features also share biological associations.

The existing cross-layer results include decoder
gene-correspondence summaries and dictionary-matching
records.

The analysis distinguishes:

1. Direct overlap between feature-associated gene sets.
2. Functional agreement based on ontology annotations.
3. Geometric similarity and mutual nearest-neighbor status.

Before statistical comparisons, the saved biological
correspondence outputs and their definitions are audited.

Matched features were selected using dictionary geometry.
Consequently, biological correspondence tests must
account for the matching and selection procedure.

Shared gene associations do not establish shared
computational mechanisms or biological causality.

All operations are read-only.

In [37]:
import json
import pandas as pd
from pathlib import Path

print("=== D.5. BIOLOGICAL CORRESPONDENCE AUDIT ===")

base_d5 = (
    ROOT / "results" / "cross_layer_independent"
)

# --------------------------------------------------
# 1. Inspect existing biological correspondence
# --------------------------------------------------

csv_files_d5 = {
    "Dictionary correspondence":
        base_d5 / "dictionary_correspondence.csv",

    "Decoder gene correspondence":
        base_d5 / "biological_correspondence"
        / "decoder_gene_correspondence.csv",
}

tables_d5 = {}

print("\n=== SAVED CORRESPONDENCE TABLES ===")

for label, path in csv_files_d5.items():

    print(f"\n{label}")
    print("Path:", path.relative_to(ROOT))

    if not path.is_file():
        print("Status: NOT FOUND")
        continue

    df = pd.read_csv(path)

    tables_d5[label] = df

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst records:")

    print(
        df.head(5).to_string(index=False)
    )

# --------------------------------------------------
# 2. Inspect biological correspondence metadata
# --------------------------------------------------

json_path_d5 = (
    base_d5
    / "biological_correspondence"
    / "decoder_gene_correspondence.json"
)

print("\n=== BIOLOGICAL CORRESPONDENCE JSON ===")
print("Exists:", json_path_d5.is_file())

if json_path_d5.is_file():

    data_d5 = json.loads(
        json_path_d5.read_text(
            encoding="utf-8"
        )
    )

    print("Root type:", type(data_d5).__name__)

    if isinstance(data_d5, dict):

        print("Top-level keys:", list(data_d5.keys()))

        for key, value in data_d5.items():

            if isinstance(
                value, (str, int, float, bool)
            ):
                print(f"{key}: {value}")

            elif isinstance(value, list):

                print(
                    f"{key}: list with "
                    f"{len(value)} elements"
                )

                if value and isinstance(value[0], dict):
                    print(
                        "First-record keys:",
                        list(value[0].keys())
                    )

            elif isinstance(value, dict):

                print(
                    f"{key}: dictionary keys = "
                    f"{list(value.keys())[:15]}"
                )

    elif isinstance(data_d5, list):

        print("Number of records:", len(data_d5))

        if data_d5 and isinstance(data_d5[0], dict):
            print(
                "First-record keys:",
                list(data_d5[0].keys())
            )

# --------------------------------------------------
# 3. Identify original biological analysis cells
# --------------------------------------------------

notebook_path_d5 = (
    ROOT
    / "notebooks"
    / "05_cross_layer_analysis.ipynb"
)

notebook_d5 = json.loads(
    notebook_path_d5.read_text(
        encoding="utf-8"
    )
)

search_terms_d5 = [
    "decoder_gene_correspondence",
    "biological_correspondence",
    "gene_overlap",
    "jaccard",
    "annotation_overlap",
]

print("\n=== NOTEBOOK 05 BIOLOGICAL ANALYSIS ===")

for cell_index, cell in enumerate(
    notebook_d5["cells"]
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    hits = [
        term
        for term in search_terms_d5
        if term in source.lower()
    ]

    if hits:

        print(
            f"Code cell {cell_index}: "
            f"{', '.join(hits)}"
        )

        lines = source.splitlines()

        matching_lines = [
            (i + 1, line.strip())
            for i, line in enumerate(lines)
            if any(
                term in line.lower()
                for term in hits
            )
        ]

        for line_number, line in matching_lines[:5]:
            print(
                f"  {line_number}: {line[:150]}"
            )

print("\nRead-only biological correspondence audit completed.")

=== D.5. BIOLOGICAL CORRESPONDENCE AUDIT ===

=== SAVED CORRESPONDENCE TABLES ===

Dictionary correspondence
Path: results\cross_layer_independent\dictionary_correspondence.csv
Shape: (11, 8)
Columns: ['layer_a', 'layer_b', 'mean_best_cosine', 'median_best_cosine', 'p95_best_cosine', 'mutual_nn_pairs', 'same_index_best_matches', 'n_features']

First records:
    layer_a     layer_b  mean_best_cosine  median_best_cosine  p95_best_cosine  mutual_nn_pairs  same_index_best_matches  n_features
  encoder.0   encoder.1          0.433135            0.410189         0.622159              810                        1        3072
  encoder.1   encoder.2          0.514959            0.492676         0.731057              904                        0        3072
  encoder.2   encoder.3          0.567330            0.553802         0.768744              928                        3        3072
processor.0 processor.1          0.615828            0.619814         0.835550             1059            

### D.5.1. Decoder Gene Correspondence: Results and Null Model

The saved decoder correspondence analysis evaluates
the overlap of Top-20 gene associations between
geometrically matched SAE features.

Three adjacent decoder transitions were examined
using both training and validation catalogs.

The analysis reports mean and median gene overlap,
Jaccard similarity, the fraction of pairs sharing
at least one gene, and empirical permutation results.

This section compares the six saved results and
inspects the original null-model implementation.

Because feature pairs were selected through dictionary
similarity, gene-overlap significance must be interpreted
conditional on that selection procedure.

Gene correspondence alone does not establish shared
biological function or computational causality.

In [38]:
import json
import numpy as np
import pandas as pd

print("=== D.5.1. DECODER GENE CORRESPONDENCE ===")

# --------------------------------------------------
# 1. Load all saved train and validation results
# --------------------------------------------------

path_d51 = (
    ROOT
    / "results/cross_layer_independent"
    / "biological_correspondence"
    / "decoder_gene_correspondence.csv"
)

D51_results = pd.read_csv(path_d51)

assert len(D51_results) == 6

required_columns = {
    "layer_a", "layer_b", "split",
    "mutual_pairs", "mean_gene_overlap",
    "median_gene_overlap", "mean_jaccard",
    "fraction_with_overlap",
    "null_mean_overlap", "empirical_p",
}

assert required_columns.issubset(D51_results.columns)

D51_results["pair"] = (
    D51_results["layer_a"]
    + " -> "
    + D51_results["layer_b"]
)

D51_results["observed_null_ratio"] = (
    D51_results["mean_gene_overlap"]
    / D51_results["null_mean_overlap"]
)

print("\n=== ALL SIX SAVED RESULTS ===")

columns = [
    "pair",
    "split",
    "mutual_pairs",
    "mean_gene_overlap",
    "median_gene_overlap",
    "mean_jaccard",
    "fraction_with_overlap",
    "null_mean_overlap",
    "observed_null_ratio",
    "empirical_p",
]

print(
    D51_results[columns]
    .round(5)
    .to_string(index=False)
)

# --------------------------------------------------
# 2. Compare train and validation descriptively
# --------------------------------------------------

print("\n=== TRAIN / VALIDATION COMPARISON ===")

comparison = D51_results.pivot(
    index="pair",
    columns="split",
    values="mean_gene_overlap"
)

print(comparison.round(4).to_string())

# --------------------------------------------------
# 3. Inspect the original null-model calculation
# --------------------------------------------------

print("\n=== ORIGINAL GENE-OVERLAP NULL SOURCE ===")

notebook_path = (
    ROOT / "notebooks/05_cross_layer_analysis.ipynb"
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

tokens = (
    "null_mean_overlap",
    "decoder_gene_correspondence.csv",
    "mean_gene_overlap",
)

found = 0

for cell_index, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(
        cell.get("source", [])
    ).splitlines()

    hits = [
        i for i, line in enumerate(lines)
        if any(token in line for token in tokens)
    ]

    if not hits:
        continue

    print(f"\nNotebook 05, code cell {cell_index}")

    selected_lines = set()

    for i in hits[:3]:
        selected_lines.update(
            range(
                max(0, i - 10),
                min(len(lines), i + 10)
            )
        )

    for i in sorted(selected_lines)[:55]:
        print(f"{i + 1:4d}: {lines[i][:170]}")

    found += 1

print("\nMatching source cells:", found)
print("Read-only correspondence audit completed.")

=== D.5.1. DECODER GENE CORRESPONDENCE ===

=== ALL SIX SAVED RESULTS ===
                  pair      split  mutual_pairs  mean_gene_overlap  median_gene_overlap  mean_jaccard  fraction_with_overlap  null_mean_overlap  observed_null_ratio  empirical_p
decoder.0 -> decoder.1      train          1255            4.44622                  3.0       0.13986                0.92112            0.06246             71.18618        0.002
decoder.0 -> decoder.1 validation          1255            4.43745                  3.0       0.13940                0.92191            0.06251             70.98970        0.002
decoder.1 -> decoder.2      train          1374            4.62154                  4.0       0.14570                0.93086            0.06416             72.03140        0.002
decoder.1 -> decoder.2 validation          1374            4.57715                  4.0       0.14408                0.92285            0.06351             72.06536        0.002
decoder.2 -> decoder.3      train   

### D.5.2. Decoder Gene-Overlap Null-Model Verification

The saved analysis reports consistent gene-set overlap
between geometrically matched decoder SAE features
across training and validation conditions.

Before interpreting the empirical probabilities,
the original random-pairing procedure is inspected.

The audit determines the candidate pool used for
random partners, whether feature matching is fixed,
and whether the null reproduces the geometric
selection procedure.

The saved results are retained unchanged.

In [39]:
import json

print("=== D.5.2. GENE-OVERLAP NULL CONSTRUCTION ===")

path = ROOT / "notebooks/05_cross_layer_analysis.ipynb"

notebook = json.loads(
    path.read_text(encoding="utf-8")
)

lines = "".join(
    notebook["cells"][8]["source"]
).splitlines()

print("=== RANDOMIZATION SOURCE LINES ===")

# The null-model loop and empirical-p calculation.
for i in range(85, min(118, len(lines))):
    print(f"{i + 1:3d}: {lines[i]}")

print("\n=== ADDITIONAL RANDOMIZATION REFERENCES ===")

for i, line in enumerate(lines[:85]):
    if any(
        token in line.lower()
        for token in [
            "rng",
            "random",
            "permutation",
            "null_means",
            "n_permutations",
        ]
    ):
        print(f"{i + 1:3d}: {line.strip()}")

print("\nRead-only null-model verification completed.")

=== D.5.2. GENE-OVERLAP NULL CONSTRUCTION ===
=== RANDOMIZATION SOURCE LINES ===
 86:         ])
 87: 
 88:         jaccards = np.array([
 89:             len(genes_a[int(i)] & genes_b[int(j)])
 90:             / max(
 91:                 len(genes_a[int(i)] | genes_b[int(j)]),
 92:                 1,
 93:             )
 94:             for i, j in zip(ids_a, ids_b)
 95:         ])
 96: 
 97:         null_means = np.empty(
 98:             N_PERMUTATIONS,
 99:             dtype=np.float64,
100:         )
101: 
102:         for p in range(N_PERMUTATIONS):
103: 
104:             shuffled_b = rng.permutation(ids_b)
105: 
106:             null_means[p] = np.mean([
107:                 len(genes_a[int(i)] & genes_b[int(j)])
108:                 for i, j in zip(ids_a, shuffled_b)
109:             ])
110: 
111:         observed = float(overlaps.mean())
112: 
113:         empirical_p = float(
114:             (1 + np.count_nonzero(null_means >= observed))
115:             / (N_PERMUTATIONS + 1

### D.6. Cross-Layer Statistical Validation — Conclusions

Cross-layer relationships were examined through
dictionary geometry, feature co-activation, and
gene-association correspondence.

All eleven adjacent-layer dictionary-matching
results were reproduced from the existing SAE
checkpoints.

The signed-coordinate-permutation controls showed
higher observed best-match cosine similarity than
the null distributions across all eleven transitions.

Mutual-nearest-neighbor fractions were lower than
their geometric null means in ten transitions,
demonstrating that high similarity does not
necessarily imply one-to-one feature alignment.

The saved co-activation analysis showed elevated
joint firing of geometrically matched features
relative to random reassignment of their partners.
All eleven comparisons reached the minimum empirical
p-value of 1/201 under 200 permutations.

Decoder gene correspondence was examined for three
adjacent transitions in both training and validation
conditions.

Mean Top-20 gene overlap ranged from approximately
4.44 to 5.02, substantially exceeding the saved
random-partner null means of approximately 0.06.

All six correspondence comparisons reached the
minimum empirical p-value of 1/501.

The gene-overlap and co-activation null models
randomly reassign partners among previously matched
features. Neither repeats the full geometric
matching procedure during permutation.

The geometric control instead changes relative
dictionary orientation and recomputes matching,
but relies on a restricted transformation family.

Taken together, these analyses support reproducible
geometric, activation-level, and gene-association
correspondence under their respective null models.

They do not establish independent biological
replication, computational causality, or shared
mechanistic function.

The remaining inferential limitations are retained
in the final statistical synthesis.

## E. Statistical Synthesis

### E.1. Consolidated Evidence and Candidate Assessment

This section consolidates the statistical analyses
performed in notebook 06b.

The evidence is organized into four categories:

- Reproducibility of feature and annotation results.
- Stability of activation and gene-association statistics.
- Functional enrichment under multiple null models.
- Cross-layer geometric, activation, and gene correspondence.

Three previously investigated decoder.0 features
(0, 189, and 2120) are compared as diagnostic cases.

The assessment retains the distinction between
global BH-adjusted enrichment, exploratory fixed-panel
permutation controls, and bootstrap ranking stability.

The selected features were not chosen through an
independent confirmatory procedure.

Consequently, the assessment supports prioritization
for further experiments, not claims of validated
biological mechanisms.

The next notebook will evaluate computational
interventions separately from statistical association.

In [40]:
import numpy as np
import pandas as pd

print("=== E.1. CONSOLIDATED STATISTICAL EVIDENCE ===")

# --------------------------------------------------
# 1. Check key results from previous sections
# --------------------------------------------------

assert len(C3_significant) == 73646
assert len(C41_results) == 98
assert len(D31_results) == 11
assert len(D41_summary) == 11
assert len(D51_results) == 6

n_annotated = (
    C3_significant[["layer", "feature"]]
    .drop_duplicates()
    .shape[0]
)

assert n_annotated == 6491

print("\n=== GLOBAL RESULTS ===")

global_summary_e1 = pd.DataFrame([
    {
        "Analysis": "Global enrichment",
        "Statistic": "BH-significant associations",
        "Result": len(C3_significant),
    },
    {
        "Analysis": "Global enrichment",
        "Statistic": "Features with significant terms",
        "Result": n_annotated,
    },
    {
        "Analysis": "Degree-aware panel",
        "Statistic": "Features passing both maxT controls",
        "Result": int(
            C41_results[
                "passes_both_fixed_panel_maxT"
            ].sum()
        ),
    },
    {
        "Analysis": "Cross-layer geometry",
        "Statistic": "Evaluated transitions",
        "Result": len(D31_results),
    },
    {
        "Analysis": "Co-activation",
        "Statistic": "Saved permutation comparisons",
        "Result": len(D41_summary),
    },
    {
        "Analysis": "Gene correspondence",
        "Statistic": "Saved train/validation comparisons",
        "Result": len(D51_results),
    },
])

print(global_summary_e1.to_string(index=False))

# --------------------------------------------------
# 2. Compare the three diagnostic SAE features
# --------------------------------------------------

candidate_rows_e1 = []

for fid in [0, 189, 2120]:

    bootstrap = summary_b8.loc[
        summary_b8["feature"].eq(fid)
    ]

    support = support_df_c1.loc[
        support_df_c1["layer"].eq("decoder.0")
        & support_df_c1["feature"].eq(fid)
    ]

    cross_split = stability_df.loc[
        stability_df["layer"].eq("decoder.0")
        & stability_df["feature"].eq(fid)
    ]

    permutation = C41_results.loc[
        C41_results["layer_num"].eq(0)
        & C41_results["feature"].eq(fid)
    ]

    enrichment = C3_significant.loc[
        C3_significant["layer"].eq(0)
        & C3_significant["feature"].eq(fid)
    ]

    assert all(
        len(frame) == 1
        for frame in [
            bootstrap,
            support,
            cross_split,
            permutation,
        ]
    )

    b = bootstrap.iloc[0]
    s = support.iloc[0]
    c = cross_split.iloc[0]
    p = permutation.iloc[0]

    candidate_rows_e1.append({
        "feature": fid,
        "train_val_shared_top20": int(
            c["shared_genes"]
        ),
        "bootstrap_mean_shared_top20": float(
            b["mean_shared_top20"]
        ),
        "train_median_support": float(
            s["median_support"]
        ),
        "significant_BH_terms": len(enrichment),
        "global_panel_maxT": float(
            p["global_maxT"]
        ),
        "degree_panel_maxT": float(
            p["strat_maxT"]
        ),
    })

E1_candidates = pd.DataFrame(candidate_rows_e1)

print("\n=== DIAGNOSTIC FEATURE COMPARISON ===")

print(
    E1_candidates.round(4).to_string(index=False)
)

# --------------------------------------------------
# 3. Summarize cross-layer evidence
# --------------------------------------------------

print("\n=== CROSS-LAYER EVIDENCE ===")

print(
    "Transitions with cosine above null mean:",
    int(
        (
            D31_results["observed_cosine"]
            > D31_results["null_mean_cosine"]
        ).sum()
    ),
    "/11"
)

print(
    "Transitions with MNN below null mean:",
    int(
        (
            D31_results["observed_mnn"]
            < D31_results["null_mean_mnn"]
        ).sum()
    ),
    "/11"
)

print(
    "Co-activation observed above null mean:",
    int(
        (
            D41_summary["observed"]
            > D41_summary["null_mean"]
        ).sum()
    ),
    "/11"
)

print(
    "Gene correspondence observed above null mean:",
    int(
        (
            D51_results["mean_gene_overlap"]
            > D51_results["null_mean_overlap"]
        ).sum()
    ),
    "/6"
)

print("\nRead-only evidence synthesis completed.")

=== E.1. CONSOLIDATED STATISTICAL EVIDENCE ===

=== GLOBAL RESULTS ===
            Analysis                           Statistic  Result
   Global enrichment         BH-significant associations   73646
   Global enrichment     Features with significant terms    6491
  Degree-aware panel Features passing both maxT controls      19
Cross-layer geometry               Evaluated transitions      11
       Co-activation       Saved permutation comparisons      11
 Gene correspondence  Saved train/validation comparisons       6

=== DIAGNOSTIC FEATURE COMPARISON ===
 feature  train_val_shared_top20  bootstrap_mean_shared_top20  train_median_support  significant_BH_terms  global_panel_maxT  degree_panel_maxT
       0                      20                      19.8680                 128.0                     2             0.0210             0.0250
     189                      19                      19.5440                 128.0                     3             0.0010             0.0020
   

### E.2. Statistical Validation — Final Findings

This notebook extended the exploratory statistical analysis
of PIE-SAE features through reproducibility checks,
bootstrap resampling, full-dictionary functional enrichment,
degree-aware permutation controls, and cross-layer validation.

#### Reproducibility and statistical protocol

The current reference implementation reconstructs 4,420
GO BP, KEGG, and Reactome terms, together with one
additional curated TRRUST transcription-factor term.

The historical annotation summaries report 4,422 terms.
The remaining discrepancy was not resolved.
After accounting for the TRRUST term, 337 historical explicit
overlap-based tests also remain unexplained across the four layers.

The reconstructed GO BP, KEGG, and Reactome terms agree
with the original term-construction implementation in
their identifiers and gene memberships.

Subsequent enrichment analyses used an explicitly defined
4,420-term reference family.

#### Activation and gene-ranking stability

Context-stratified bootstrap analyses reproduced and
characterized the activation statistics of decoder.0
feature 0.

Cross-split comparisons covered all 12,288 decoder
SAE features, of which 12,285 had complete Top-20
gene-association catalogs in both splits.

Most features showed high agreement between training
and validation gene rankings.

Detailed condition-level bootstrap analyses revealed
important differences between individual features.

Features 0 and 189 exhibited high Top-20 stability
and strong firing support.

Feature 2120 exhibited low gene-ranking stability,
sparse firing support among its highest-ranked genes,
and substantial sensitivity to the association statistic.

#### Full-dictionary functional enrichment

The primary analysis evaluated 54,312,960 feature–term
hypotheses using one-sided hypergeometric enrichment
and global Benjamini–Hochberg correction.

At q <= 0.05, 73,646 feature–term associations were
identified across 6,491 decoder SAE features.

These associations are significant relative to the
specified hypergeometric model, subject to its
gene-selection and dependency assumptions.
The formal FDR interpretation of BH additionally depends
on the relevant assumptions about the joint null-p-value
distribution; these were not established for the present
correlated feature–ontology tests.

An exploratory panel of 98 features was subsequently
evaluated using global and annotation-degree-stratified
gene-reassignment controls.

Nineteen panel features passed both fixed-panel
maxT procedures at p < 0.05.

Because the panel was partly selected using the prior
enrichment results, this does not establish full
selection-adjusted significance across the complete
SAE feature population.

#### Cross-layer correspondence

All eleven adjacent-layer dictionary-matching results
were reproduced from the pretrained SAE checkpoints.

Geometry-preserving controls demonstrated elevated
best-match cosine similarity under the selected
signed-coordinate-permutation null.

Saved co-activation and decoder gene-correspondence
analyses showed substantial agreement among geometrically
matched features relative to random partner reassignment.

The available null models test distinct properties of
cross-layer correspondence and do not establish causal
relationships between SAE features.

#### Diagnostic feature prioritization

Decoder.0 feature 189 is prioritized for subsequent
computational intervention experiments because of its
strong gene-ranking stability, firing support, and
exploratory enrichment-control results.

Feature 0 provides an additional candidate with high
cross-split stability and consistent statistical evidence.

Feature 2120 is retained as a contrasting case of
BH-significant enrichment with low ranking stability
and weak support under the degree-aware maxT control.

These priorities are exploratory and are not independent
confirmatory discoveries.

#### Outstanding limitations

The study does not establish statistical significance
under a fully selection-aware, degree-controlled null
across all 12,288 decoder SAE features.

The analyzed SAE dictionaries were not independently
retrained across multiple random seeds.

High feature–gene stability may reflect persistent
gene-identity effects rather than perturbation-specific
representations.

Permutation and bootstrap results depend on their
respective null models and exchangeability assumptions.

No SAE feature ablation, activation patching, or other
computational causal intervention was performed in this
notebook.

Independent biological validation remains outside the
scope of the present work.

#### Conclusion

The analysis provides reproducible evidence of stable
gene-associated SAE representations, extensive functional
enrichment under the specified statistical models, and
cross-layer correspondence in the pretrained PIE system.

These results support the selection of computational
intervention targets but do not establish that the
identified SAE features are biologically causal,
mechanistically unique, or independently validated.

Computational causal validation is the next research stage.

### E.3. Final Computational Consistency Checks

This final read-only audit verifies that the principal
results used in the statistical synthesis agree with
the intermediate analysis outputs.

The checks cover the predefined hypothesis family,
BH results, exploratory permutation panel, bootstrap
feature comparisons, and saved cross-layer evidence.

Passing these checks establishes internal numerical
consistency of the evaluated results.

It does not replace source-code review, independent
reproduction, or validation of statistical assumptions.

In [41]:
import numpy as np

print("=== E.3. FINAL COMPUTATIONAL CONSISTENCY AUDIT ===")

checks = {}

# --------------------------------------------------
# 1. Hypothesis family and enrichment
# --------------------------------------------------

checks["Gene universe"] = len(C2_genes) == 6642

checks["Ontology family"] = (
    len(C2_terms) == 4420
)

checks["Decoder features"] = (
    C2_feature_matrix.shape == (12288, 6642)
)

checks["Feature-term hypotheses"] = (
    C2_overlap.shape == (12288, 4420)
)

checks["Explicit enrichment tests"] = (
    N_EXPLICIT == 609033
)

checks["BH-significant associations"] = (
    len(C3_significant) == 73646
)

checks["BH-significant features"] = (
    C3_significant[
        ["layer", "feature"]
    ].drop_duplicates().shape[0] == 6491
)

# --------------------------------------------------
# 2. Bootstrap and candidate evidence
# --------------------------------------------------

checks["Bootstrap feature coverage"] = (
    set(summary_b8["feature"]) == {0, 189, 2120}
)

checks["Candidate synthesis coverage"] = (
    set(E1_candidates["feature"]) == {0, 189, 2120}
)

checks["Cross-split feature coverage"] = (
    len(stability_df) == 12288
)

checks["Complete cross-split catalogs"] = (
    int(stability_df["complete_pair"].sum()) == 12285
)

# --------------------------------------------------
# 3. Exploratory degree-aware controls
# --------------------------------------------------

checks["Exploratory panel size"] = (
    len(C41_results) == 98
)

checks["Both panel maxT controls"] = (
    int(
        C41_results[
            "passes_both_fixed_panel_maxT"
        ].sum()
    ) == 19
)

# --------------------------------------------------
# 4. Cross-layer results
# --------------------------------------------------

checks["Dictionary transition coverage"] = (
    len(D31_results) == 11
)

checks["Coactivation transition coverage"] = (
    len(D41_summary) == 11
)

checks["Decoder gene correspondence coverage"] = (
    len(D51_results) == 6
)

checks["Cosine above geometric null mean"] = (
    int((
        D31_results["observed_cosine"]
        > D31_results["null_mean_cosine"]
    ).sum()) == 11
)

checks["Coactivation above null mean"] = (
    int((
        D41_summary["observed"]
        > D41_summary["null_mean"]
    ).sum()) == 11
)

checks["Gene overlap above null mean"] = (
    int((
        D51_results["mean_gene_overlap"]
        > D51_results["null_mean_overlap"]
    ).sum()) == 6
)

# --------------------------------------------------
# 5. Report results
# --------------------------------------------------

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

failed = [
    name
    for name, passed in checks.items()
    if not passed
]

print("\n=== FINAL STATUS ===")

print("Checks performed:", len(checks))
print("Checks passed:", len(checks) - len(failed))
print("Checks failed:", len(failed))

if failed:
    raise AssertionError(
        "Final consistency audit failed: "
        + ", ".join(failed)
    )

print("\nInternal computational consistency: PASS")
print("Independent methodological audit: NOT YET PERFORMED")
print("Computational causal validation: NOT YET PERFORMED")
print("Original research artifacts remain unchanged.")

=== E.3. FINAL COMPUTATIONAL CONSISTENCY AUDIT ===
Gene universe: PASS
Ontology family: PASS
Decoder features: PASS
Feature-term hypotheses: PASS
Explicit enrichment tests: PASS
BH-significant associations: PASS
BH-significant features: PASS
Bootstrap feature coverage: PASS
Candidate synthesis coverage: PASS
Cross-split feature coverage: PASS
Complete cross-split catalogs: PASS
Exploratory panel size: PASS
Both panel maxT controls: PASS
Dictionary transition coverage: PASS
Coactivation transition coverage: PASS
Decoder gene correspondence coverage: PASS
Cosine above geometric null mean: PASS
Coactivation above null mean: PASS
Gene overlap above null mean: PASS

=== FINAL STATUS ===
Checks performed: 19
Checks passed: 19
Checks failed: 0

Internal computational consistency: PASS
Independent methodological audit: NOT YET PERFORMED
Computational causal validation: NOT YET PERFORMED
Original research artifacts remain unchanged.
